# Fabric Database Hub hybrid SQL demo: live activity and analysis

Use this notebook to **analyze a demo session's locks, blocking, and overall consumption** with the same telemetry that powers Database Hub **Performance**, and to **save its history** for the Power BI report and the Insights app.

**Where the activity comes from.** A demo session runs from **`NB_DatabaseHub_Control`**, in the control workspace: its one Run all starts the targets, runs the showcase, a rotation of bounded incidents on T1 to T5, follows the session to its end, and then runs this notebook in its own Spark session (a reference run; as a job in this workspace when it can't) with the session's ID, its beats, and T4's and T5's metrics (ADR 0026). The activity itself runs on the servers: the Contoso Care runner on T1 (driving T1, T3, T4, and T5) and in the T2 guest, and every profile has a maximum duration and stops before auto-shutdown. This notebook changes no activity and uses no GitHub Actions; `./scripts/demo-control.ps1` sets one activity from a PC.

**Safety.** The analysis is read-only and writes only to the lakehouse tables `scenario_runs`, `telemetry_snapshots`, `analysis_findings`, and `incident_windows`, and the Incident RCA's `rca_*` tables (its evidence's eight and its sidecar's six, with the sidecar's acknowledgements in `Files/rca/acks`), which the Power BI history report reads. A session already saved isn't saved twice (section 3), and a save that a cancel cut off is completed by the session's next run, not repeated (section 11). Tokens are never printed. **Run all** analyzes the last `WINDOW_MINUTES` without a session.

**T4, the PostgreSQL flexible server, and its read replica** aren't in that telemetry: Database Hub charts their Azure Monitor metrics, which a notebook can't read. The control's runbook (at a session's end), `./scripts/stop-demo.ps1 -SaveHistory`, and `./scripts/deploy.ps1` read them and pass them in the `POSTGRES_METRICS` parameter, and section 7 adds them to the history (ADR 0025).

**The Incident RCA's evidence** (ADR 0031): during a session, an observer beside the workload on the VMs captures what a DBA would look at: blocking chains with each session's statement, login, and program; waits; errors and deadlocks; Query Store's plans; resource samples; and T4's read replica's replay. The control's runbook (at a session's end), `./scripts/stop-demo.ps1 -SaveHistory`, and `./scripts/deploy.ps1` export it from the VMs and pass it in `RCA_EVIDENCE_FILE` (or `RCA_EVIDENCE`), and section 11 checks it against the contract (`rca/`) and writes the seven detection tables and, apart from them, the answer key's table, `rca_answer_key`, which only the tests and the scoreboard read.

**T5, the Azure Cosmos DB account,** isn't in that telemetry either: Database Hub's Cosmos DB views chart its Azure Monitor platform metrics. The same runbook and scripts read them and pass them in the `COSMOS_METRICS` parameter, and section 7 adds them to the history (ADR 0027).

> **Preview disclosure:** Database Hub is generally available (2026-11-17); its Copilot and Database Agents, and the SQL telemetry endpoint, are previews, whose tables, columns, and behavior can change. Using a notebook `kusto` token against the telemetry endpoint isn't documented, so section 4 checks it and every query falls back to printing its KQL with the documented Azure Data Explorer web UI steps.

**Contents:** 1 Parameters · 2 Helpers · 3 This session · 4 Telemetry access · 5 Estate pulse · 6 Locks and blocking · 7 Overall consumption, T4's PostgreSQL metrics, and T5's Azure Cosmos DB metrics · 8 Correlation · 9 See it in the Hub · 10 Optional Real-Time Dashboard · 11 Persist · 12 Next steps

The companion notebook **`NB_DatabaseHub_Deployment`** validates the deployment and scripts the rest of the demo.

## 1. Parameters

The next cell is the **parameters cell** (in Fabric: **...** > **Toggle parameter cell**), so a job, a pipeline notebook activity, or `notebookutils.notebook.run` can override any value. It holds environment names, IDs, the analysis window, `POSTGRES_METRICS` and `COSMOS_METRICS` (T4's and T5's Azure Monitor metrics when the scripts or the control pass them), and a session's `SESSION_ID` and `SESSION_PLAN`. **Never put a credential in it.**

In [ ]:
# Parameters cell (tagged "parameters"). Environment values only: never put a credential here.

# Fabric workspace and default lakehouse (prerequisite P5).
# The repo stores no Fabric IDs (ADR 0021): the scripts' import fills these two in. Empty means this session's
# workspace and default lakehouse, so a notebook you import by hand works too.
WORKSPACE_ID = ""
LAKEHOUSE_ID = ""
LAKEHOUSE_NAME = "LH_DatabaseHub"

# Demo environment names (docs/architecture.md).
RESOURCE_GROUP = "rg-dbhub-demo"
T1_VM_NAME = "sqlvm01"
T2_MACHINE_NAME = "arcsql01"
T3_SERVER_PREFIX = "azuresql01-"
DEMO_DATABASE = "ContosoCare"
TARGETS = "T1,T2,T3"

# Analysis window: the last WINDOW_MINUTES, or an explicit UTC window inside the 7-day retention, for example
# WINDOW_START_UTC = "2026-10-20T14:00:00Z". BIN_MINUTES = 0 picks a bin size from the window length.
WINDOW_MINUTES = 60
WINDOW_START_UTC = ""
WINDOW_END_UTC = ""
BIN_MINUTES = 0
# Telemetry arrives 3-15 minutes after the activity; correlation looks this far past each scenario window.
TELEMETRY_LAG_MINUTES = 15

# T4, the PostgreSQL flexible server, and its read replica (2026-11-13, ADR 0025): their Azure Monitor metrics as compact
# JSON, which the control's runbook (a session's end), ./scripts/stop-demo.ps1 -SaveHistory, and deploy.ps1's history
# snapshot read and pass here, because a notebook gets no Azure Resource Manager token. Empty: none were passed, and
# section 7's T4 cell adds nothing.
POSTGRES_METRICS = ""

# T5, the Azure Cosmos DB account (2026-11-18, ADR 0027): its Azure Monitor metrics as compact JSON, passed the same way
# (the control's runbook, stop-demo.ps1 -SaveHistory, deploy.ps1's history snapshot, and configure-fabric.ps1
# -RunLiveActivity). Empty: none were passed, and section 7's T5 cell adds nothing.
COSMOS_METRICS = ""

# A demo session (2026-11-14, ADR 0026): NB_DatabaseHub_Control's Run all runs this notebook at the session's end,
# with its ID and its beats (SESSION_PLAN, JSON), which section 3 records in scenario_runs for the correlation; the
# Database Agent beat also has agent, regressUtc, and respondUtc (2026-11-18), for section 8's database_agent finding. A
# session whose history is already saved isn't saved twice. Empty: an analysis of the window above, with no session.
SESSION_ID = ""
SESSION_PLAN = ""

# The Incident RCA's evidence (2026-11-20, ADR 0031): the part lines of a session's evidence documents, which the observer
# on the VMs captured and the control's runbook or the operator's scripts exported (rca/README.md, Transport). The
# control notebook and the scripts stage them in this lakehouse's Files/rca/inbox and pass the file's path in
# RCA_EVIDENCE_FILE, which this notebook deletes once it has handled it; RCA_EVIDENCE can hold the same JSON instead, or
# as well: with both, their parts are merged, and each document is written once. Section 11 writes the seven detection
# tables and the answer key's table, rca_answer_key. Empty: none was passed, and no RCA rows are written. RCA_ONLY = True
# writes only that evidence, then ends the run, whatever SAVE_RESULTS says: an earlier session's, which the next
# session's start exported from the VMs. The control notebook passes SAVE_RESULTS = False with it, so a copy of this
# notebook from before the Incident RCA saves nothing.
RCA_EVIDENCE = ""
RCA_EVIDENCE_FILE = ""
RCA_ONLY = False

# The Incident RCA's sidecar (2026-11-23, rca/README.md, The sidecar): Query Store's evidence on T1 to T4 and who held
# T4's connections, in documents of their own. The scripts stage a save's sidecars in this lakehouse's Files/rca/inbox and
# pass the file's path here; section 2's last cells write its six tables (rca_query_* and rca_connections), each sidecar
# under its own session's run, acknowledge each by its ID in Files/rca/acks, and delete the file once every sidecar in it
# is written. Empty: none was passed, and no sidecar rows are written.
RCA_SIDECAR_FILE = ""

# Detected incident windows for the Power BI history report: consecutive bins over a threshold on one target. These are
# presenter heuristics, not product behavior (ADR 0014). Any blocked process, deadlock, or Log Rate Governor wait counts
# from one bin.
EPISODE_CPU_PERCENT = 80          # CPU at or above this (T3: percentage of the tier limit) for EPISODE_MIN_BINS bins
EPISODE_PAGEIOLATCH_SECONDS = 1   # Buffer IO (PAGEIOLATCH) waits per bin at or above this for EPISODE_MIN_BINS bins
EPISODE_WAIT_SECONDS = 5          # Buffer Latch, Memory, Parallelism, Tran Log IO, or Network IO waits per bin (Phase 25b)
EPISODE_MIN_BINS = 2

# Append results to the default lakehouse's Delta tables (scenario_runs, telemetry_snapshots, analysis_findings,
# incident_windows, and the RCA evidence's rca_* tables) and prepare the tables that the Power BI history report reads.
SAVE_RESULTS = True

## 2. Helpers

Run the next four cells before any other code cell. They validate the parameters and define the helpers:

- **Tokens:** `notebookutils.credentials.getToken` with documented audience keys only (`pbi` for the Fabric REST API, `kusto` for the telemetry endpoint). There's no Azure Resource Manager audience.
- **Telemetry KQL** follows the telemetry article's rules: filter on `SampleTimeUTC` first; take per-series deltas of cumulative counters and skip intervals with a reset (never sum them); branch on `CounterType` (`65792` gauge, `272696576` cumulative, `537003264` percentage); group by `ResourceID` and `SqlServerInstanceName`; exclude the `Idle` and `Unknown` wait categories. For per-bin timelines it keeps the first and last sample of each series in each bin, the article's pattern for queries that must scale.
- **T4's metrics:** `parse_postgres_metrics` checks the `POSTGRES_METRICS` document (version 1) and keeps only valid servers, series, and points; `record_postgres_metrics` turns it into this run's `telemetry_snapshots` rows and findings (analysis `postgresql`). Neither makes a network call.
- **T5's metrics:** `parse_cosmos_metrics` checks the `COSMOS_METRICS` document (version 1) the same way; `record_cosmos_metrics` turns it into this run's rows and findings (analysis `cosmosdb`, target `T5`), with each throttling episode. Neither makes a network call.
- **A session:** `parse_session_plan` checks the control's `SESSION_PLAN` and keeps only valid beats; `session_requests` turns the session and its beats into `scenario_runs` rows, and `session_saved` says whether a session is already in the history. None of them makes a network call.
- **The Incident RCA's evidence (part 4):** `read_rca_evidence` reads `RCA_EVIDENCE`, or the staged file in `Files/rca/inbox` that `RCA_EVIDENCE_FILE` names; `decode_rca_parts` groups the part lines by document, decodes them, checks each document's SHA-256 and its content against the contract (`check_rca_document`: `rca/evidence.schema.json` and `rca/history-tables.json`, version 1), and skips a document with a missing or bad part, never writing it in part; `save_rca_evidence` writes each document once to the RCA tables, the answer key only to `rca_answer_key`, and reports `RCA.EVIDENCE`. No network call; the staged file is read and deleted with `notebookutils.fs`.
- **Results:** findings use the demo's status model (**Pass**, **Fail**, **Warning**, **Manual action required**, **Not supported**, **Not selected**).

In [ ]:
# Helpers, part 1: parameter checks, tokens, Fabric REST, and telemetry queries. Run the three helper cells first.
# Tokens come from notebookutils with documented audience keys only. A token is used in one HTTP header and never
# printed, logged, returned, or saved.
import datetime as dt
import json
import re
import time
import uuid

import pandas as pd
import requests

NOTEBOOK_NAME = "NB_DatabaseHub_LiveActivity"
FABRIC_API = "https://api.fabric.microsoft.com/v1"
TELEMETRY_QUERY_URL = "https://adx.centralus.arcdataservices.com/kusto/v1/rest/query"
TELEMETRY_CONNECTION_URI = "https://adx.centralus.arcdataservices.com/kusto/"
TELEMETRY_DATABASE = "ArcSqlTelemetry"
ADX_WEB_UI = "https://dataexplorer.azure.com/"
DATABASE_HUB_URL = "https://powerbi.com/workloads/fdh/databaseHub"
# notebookutils.credentials.getToken supports only these audience keys. None of them is Azure Resource Manager.
ALLOWED_TOKEN_AUDIENCES = ("storage", "pbi", "keyvault", "kusto")
STATUSES = ("Pass", "Fail", "Warning", "Manual action required", "Not supported", "Not selected")
SCENARIO_SIGNALS = {
    "locking-blocking": ("blocking",),
    "cpu-pressure": ("cpu",),
    "io-pressure": ("io",),
    "inefficient-query": ("cpu", "log_rate"),
    "missing-index": ("cpu", "io"),
    "resource-governance": ("cpu",),
    "log-rate-pressure": ("log_rate",),
    "blocking-chain": ("blocking",),
    "deadlock-storm": ("deadlock", "blocking"),
    "tempdb-contention": ("latch",),
    "memory-grant-pressure": ("memory",),
    "parallel-scans": ("parallelism", "cpu"),
    "log-flush-waits": ("tran_log",),
    "slow-consumer": ("network",),
    "capacity-max": ("cpu", "io"),
    "plan-regression": ("cpu",),
    # T4's connection surge (F34): its active connections at the slots clients can use, never its failed connections,
    # which Azure Monitor didn't count its refused attempts in, live (2026-11-21).
    "connection-surge": ("connections",),
}
TARGET_COLORS = {"T1": "tab:blue", "T2": "tab:green", "T3": "tab:purple", "T4": "tab:red", "T4-replica": "tab:brown", "T5": "tab:olive"}
DOCS = {
    "notebookutils": "https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-credentials",
    "telemetry": "https://learn.microsoft.com/fabric/database/hub/microsoft-sql-query-performance-monitoring-telemetry",
    "monitor": "https://learn.microsoft.com/fabric/database/hub/monitor-sql",
    "listItems": "https://learn.microsoft.com/rest/api/fabric/core/items/list-items",
    "rtd": "https://learn.microsoft.com/fabric/real-time-intelligence/dashboard-real-time-create",
}
TELEMETRY_HTTP_HINTS = {
    400: "HTTP 400: the KQL has an error, or a table or column changed during the preview (check names with getschema).",
    401: "HTTP 401: the endpoint didn't accept the notebook's kusto token. A notebook kusto token isn't documented for this endpoint.",
    403: "HTTP 403: the signed-in user needs Reader on the subscription, and Microsoft.Sql must be registered on it.",
    500: "HTTP 500: the request body wasn't valid.",
}
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
FINDINGS = []
SNAPSHOT_ROWS = []
SESSION_REQUESTS = []
FRAMES = {}
# The tables this run saved: running the persist cell again replaces their rows (write_delta).
SAVED_TABLES = set()
TELEMETRY_STATE = {"checked": False, "available": False, "message": "The telemetry access check (section 4) hasn't run yet."}


def _require(pattern, value, name):
    if not isinstance(value, str) or not re.fullmatch(pattern, value):
        raise ValueError("Parameter %s has an unexpected value: %r" % (name, value))
    return value


def _flag(value):
    """Parameters can arrive as strings from a pipeline; only an explicit true value counts as True."""
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in ("true", "1", "yes")


SAVE_RESULTS = _flag(SAVE_RESULTS)
_GUID = r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}"
# Empty IDs are resolved from the session once runtime_context() is defined, below (ADR 0021).
WORKSPACE_ID = str(WORKSPACE_ID or "").strip()
LAKEHOUSE_ID = str(LAKEHOUSE_ID or "").strip()
_require(r"[A-Za-z0-9 _.-]{1,256}", LAKEHOUSE_NAME, "LAKEHOUSE_NAME")
# Names are checked against strict patterns before they're placed in KQL.
_require(r"[A-Za-z0-9._()-]{1,90}", RESOURCE_GROUP, "RESOURCE_GROUP")
_require(r"[A-Za-z0-9-]{1,64}", T1_VM_NAME, "T1_VM_NAME")
_require(r"[A-Za-z0-9._-]{1,64}", T2_MACHINE_NAME, "T2_MACHINE_NAME")
_require(r"[a-z0-9-]{1,63}", T3_SERVER_PREFIX, "T3_SERVER_PREFIX")
_require(r"[A-Za-z0-9_]{1,128}", DEMO_DATABASE, "DEMO_DATABASE")
# T4's metrics arrive as text, empty when none were passed. Section 7's T4 cell checks them (parse_postgres_metrics),
# not this cell, so a value it can't use never stops the SQL analysis.
POSTGRES_METRICS = str(POSTGRES_METRICS or "").strip()
# T5's, the same way (section 7's T5 cell, parse_cosmos_metrics).
COSMOS_METRICS = str(COSMOS_METRICS or "").strip()
# The Incident RCA's evidence (ADR 0031), as text: part 4 checks it (read_rca_evidence), not this cell, so evidence it
# can't use never stops the analysis.
RCA_EVIDENCE = str(RCA_EVIDENCE or "").strip()
RCA_EVIDENCE_FILE = str(RCA_EVIDENCE_FILE or "").strip()
RCA_ONLY = _flag(RCA_ONLY)
# A session from the control notebook (ADR 0026): its ID, checked here, and its plan, which section 3 reads.
SESSION_ID = str(SESSION_ID or "").strip().lower()
if SESSION_ID:
    _require(r"[0-9a-f]{32}", SESSION_ID, "SESSION_ID")
    # Every run that saves this session's history saves it under one run ID, so a run that a cancel cut off is completed
    # by the next one, not saved twice (write_delta, R907).
    RUN_ID = "session-" + SESSION_ID
SESSION_PLAN = str(SESSION_PLAN or "").strip()
SELECTED_TARGETS = [item.strip().upper() for item in str(TARGETS).split(",") if item.strip()]
if not SELECTED_TARGETS or any(item not in ("T1", "T2", "T3") for item in SELECTED_TARGETS):
    raise ValueError("TARGETS must list T1, T2, and/or T3, for example 'T1,T2,T3'.")
if not 5 <= int(WINDOW_MINUTES) <= 7 * 24 * 60:
    raise ValueError("WINDOW_MINUTES must be between 5 minutes and the 7-day retention.")
if not 0 <= int(BIN_MINUTES) <= 60 or not 0 <= int(TELEMETRY_LAG_MINUTES) <= 60:
    raise ValueError("BIN_MINUTES and TELEMETRY_LAG_MINUTES must be between 0 and 60.")
EPISODE_CPU_PERCENT = float(EPISODE_CPU_PERCENT)
EPISODE_PAGEIOLATCH_SECONDS = float(EPISODE_PAGEIOLATCH_SECONDS)
EPISODE_MIN_BINS = int(EPISODE_MIN_BINS)
EPISODE_WAIT_SECONDS = float(EPISODE_WAIT_SECONDS)
if (not 0 < EPISODE_CPU_PERCENT <= 100 or EPISODE_PAGEIOLATCH_SECONDS <= 0 or not 1 <= EPISODE_MIN_BINS <= 60
        or EPISODE_WAIT_SECONDS <= 0):
    raise ValueError("EPISODE_CPU_PERCENT must be above 0 and at most 100, EPISODE_PAGEIOLATCH_SECONDS and "
                     "EPISODE_WAIT_SECONDS above 0, and EPISODE_MIN_BINS between 1 and 60.")

_REDACTIONS = (
    (re.compile(r"eyJ[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]{5,}\.[A-Za-z0-9_-]*"), "[REDACTED:jwt]"),
    (re.compile(r"(?i)(bearer\s+)[A-Za-z0-9._~+/=-]{8,}"), r"\1[REDACTED]"),
    (re.compile(r"\b(?:gh[pousr]_[A-Za-z0-9]{20,}|github_pat_[A-Za-z0-9_]{20,})\b"), "[REDACTED:github-token]"),
    (re.compile(r"(?i)((?:^|[?&;\s\"'])sig=)[^&\s\"']+"), r"\1[REDACTED]"),
)


def redact(value):
    """Removes token-like strings from text before it's printed or saved."""
    text = "" if value is None else str(value)
    for pattern, replacement in _REDACTIONS:
        text = pattern.sub(replacement, text)
    return text


def utc_now():
    return dt.datetime.now(dt.timezone.utc)


def _notebookutils():
    try:
        return notebookutils  # noqa: F821 - Fabric provides notebookutils in every notebook session.
    except NameError:
        try:
            import notebookutils as module
            return module
        except ImportError:
            return None


def _spark():
    try:
        return spark  # noqa: F821 - Fabric provides the Spark session.
    except NameError:
        return None


def display_frame(frame):
    if REFERENCE_RUN:
        # One line and none of the table's values: nobody reads a reference run's output, and the history keeps the table.
        print("(%d row(s) of %s; the history report and the Insights app show them.)" % (len(frame), ", ".join(str(column) for column in frame.columns)))
        return
    try:
        display(frame)  # noqa: F821 - Fabric provides display().
    except NameError:
        print(frame.to_string(index=False))


def get_token(audience):
    """Returns a token for a documented notebookutils audience key. Never print, log, or save the result."""
    if audience not in ALLOWED_TOKEN_AUDIENCES:
        raise ValueError("notebookutils.credentials.getToken supports only %s." % ", ".join(ALLOWED_TOKEN_AUDIENCES))
    utils = _notebookutils()
    if utils is None:
        raise RuntimeError("notebookutils isn't available. Run this notebook in Microsoft Fabric.")
    return utils.credentials.getToken(audience)


def runtime_context():
    utils = _notebookutils()
    if utils is None:
        return {}
    try:
        return dict(utils.runtime.context or {})
    except Exception:
        return {}


def in_reference_run():
    """Whether another notebook runs this one in its own Spark session (notebookutils.notebook.run), as the control
    notebook's Run all does to save a session's history (isReferenceRun, NotebookUtils runtime context). Fabric then
    reads each cell's result itself, after its secret redaction: on 2026-11-16 the first chart's result stopped such a run
    ("Unrecognized token 'REDACTED'"), while the table before it passed. So this run prints one line for each table and
    chart instead: no table values, no NaN or infinity, and nothing taken from a secret."""
    return bool(runtime_context().get("isReferenceRun"))


REFERENCE_RUN = in_reference_run()


# The repo stores no Fabric IDs (ADR 0021). Empty IDs, as in a notebook imported by hand, are this session's own
# workspace and default lakehouse; the scripts' import fills in the ones it resolved by name.
if not WORKSPACE_ID or not LAKEHOUSE_ID:
    _session = runtime_context()
    WORKSPACE_ID = WORKSPACE_ID or str(_session.get("currentWorkspaceId") or "")
    LAKEHOUSE_ID = LAKEHOUSE_ID or str(_session.get("defaultLakehouseId") or "")
if WORKSPACE_ID:
    _require(_GUID, WORKSPACE_ID, "WORKSPACE_ID")
if LAKEHOUSE_ID:
    _require(_GUID, LAKEHOUSE_ID, "LAKEHOUSE_ID")


def _retry_after(response):
    try:
        return min(max(int(response.headers.get("Retry-After", "5")), 1), 60)
    except ValueError:
        return 5


def _error_text(response):
    try:
        body = response.json()
    except ValueError:
        return redact((response.text or "")[:300])
    if isinstance(body, dict):
        parts = [str(body.get(key)) for key in ("errorCode", "message", "requestId", "documentation_url") if body.get(key)]
        if isinstance(body.get("error"), dict):
            parts += [str(body["error"].get(key)) for key in ("code", "message") if body["error"].get(key)]
        return redact(" ".join(parts)[:300])
    return ""


def fabric_get(path_or_url):
    """GET from the Fabric REST API with the notebook user's pbi token. Returns (status, body, error)."""
    url = path_or_url if path_or_url.startswith(FABRIC_API + "/") else FABRIC_API + path_or_url
    if not url.startswith(FABRIC_API + "/"):
        raise ValueError("Refusing to send a Fabric token outside " + FABRIC_API)
    response = None
    for attempt in range(4):
        try:
            response = requests.get(url, headers={"Authorization": "Bearer " + get_token("pbi")}, timeout=60)
        except requests.RequestException as error:
            return 0, None, redact(error)
        except Exception as error:
            return 0, None, "notebookutils didn't return a pbi token: " + redact(error)
        if response.status_code == 429 and attempt < 3:
            time.sleep(_retry_after(response))
            continue
        break
    if response.ok:
        try:
            return response.status_code, response.json(), ""
        except ValueError:
            return response.status_code, None, "The response wasn't JSON."
    return response.status_code, None, _error_text(response)


def fabric_list_items(workspace_id, item_type):
    """Lists items of one type in a workspace, following continuation pages. Returns (items, error)."""
    items = []
    url = "%s/workspaces/%s/items?type=%s" % (FABRIC_API, workspace_id, item_type)
    for _ in range(50):
        if not url:
            return items, ""
        status, body, error = fabric_get(url)
        if status != 200 or body is None:
            return None, "HTTP %s %s" % (status, error)
        # A continuation URI can drop the type filter, so keep only items of the requested type.
        items.extend(item for item in body.get("value", []) if item.get("type") in (None, item_type))
        url = body.get("continuationUri")
        if url and not url.startswith(FABRIC_API + "/"):
            return None, "Unexpected continuation URI; stopped so the token stays inside the Fabric API."
    return (items, "") if not url else (None, "Stopped after 50 pages of results.")


# ---- Telemetry ----------------------------------------------------------------------------------------------

def _parse_utc(text):
    value = pd.Timestamp(str(text).strip())
    value = value.tz_localize("UTC") if value.tzinfo is None else value.tz_convert("UTC")
    return value.to_pydatetime()


def analysis_window():
    """Returns (start, end, bin_minutes) for the analysis, inside the telemetry's 7-day retention."""
    now = utc_now()
    end = _parse_utc(WINDOW_END_UTC) if str(WINDOW_END_UTC).strip() else now
    end = min(end, now)
    start = _parse_utc(WINDOW_START_UTC) if str(WINDOW_START_UTC).strip() else end - dt.timedelta(minutes=int(WINDOW_MINUTES))
    if start >= end:
        raise ValueError("The analysis window must start before it ends.")
    if now - start > dt.timedelta(days=7):
        print("Note: the window starts more than 7 days ago; telemetry retention is up to 7 days, so older data isn't available.")
    span_minutes = (end - start).total_seconds() / 60
    bin_minutes = int(BIN_MINUTES) or (1 if span_minutes <= 120 else 5 if span_minutes <= 720 else 15)
    return start, end, bin_minutes


def kql_string(value):
    return "'" + str(value).replace("\\", "\\\\").replace("'", "\\'") + "'"


def kql_datetime(value):
    return "datetime(" + value.astimezone(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ") + ")"


def kql_prelude(start, end, bin_minutes=1):
    """Time window, bin, names, and a function that maps telemetry rows to T1, T2, or T3 (empty for anything else)."""
    return "\n".join([
        "let _start = %s;" % kql_datetime(start),
        "let _end = %s;" % kql_datetime(end),
        "let _bin = %dm;" % int(bin_minutes),
        "let _rg = %s;" % kql_string(RESOURCE_GROUP),
        "let _db = %s;" % kql_string(DEMO_DATABASE),
        "let _t1 = %s;" % kql_string(T1_VM_NAME),
        "let _t2 = %s;" % kql_string(T2_MACHINE_NAME),
        "let _t3prefix = %s;" % kql_string(T3_SERVER_PREFIX),
        "let _t3suffix = %s;" % kql_string("/" + DEMO_DATABASE),
        "let _target = (resourceType: string, resourceName: string) {",
        "    case(resourceType == 'virtualMachines' and resourceName =~ _t1, 'T1',",
        "         resourceType == 'machines' and resourceName =~ _t2, 'T2',",
        "         resourceType == 'servers/databases' and resourceName startswith _t3prefix and resourceName endswith _t3suffix, 'T3',",
        "         '')",
        "};",
    ])


def _to_utc(series):
    try:
        return pd.to_datetime(series, utc=True, format="ISO8601")
    except (TypeError, ValueError):
        return pd.to_datetime(series, utc=True, errors="coerce")


def kusto_frame(body):
    """Turns a Kusto v1 response (Tables[0].Columns and Rows) into a DataFrame, mapping values by column name."""
    tables = body.get("Tables") or []
    if not tables:
        return pd.DataFrame()
    first = tables[0]
    columns = first.get("Columns", [])
    frame = pd.DataFrame(first.get("Rows", []), columns=[column.get("ColumnName") for column in columns])
    for column in columns:
        if str(column.get("ColumnType") or column.get("DataType")).lower() in ("datetime", "system.datetime"):
            frame[column["ColumnName"]] = _to_utc(frame[column["ColumnName"]])
    return frame


def _post_kql(kql):
    """Sends read-only KQL to the telemetry endpoint. Returns (status, frame, message)."""
    try:
        token = get_token("kusto")
    except Exception as error:
        return 0, None, "notebookutils didn't return a kusto token: " + redact(error)
    try:
        response = requests.post(
            TELEMETRY_QUERY_URL,
            json={"db": TELEMETRY_DATABASE, "csl": str(kql)},
            headers={"Authorization": "Bearer " + token, "Content-Type": "application/json"},
            timeout=180,
        )
    except requests.RequestException as error:
        return 0, None, "The telemetry endpoint couldn't be reached: " + redact(error)
    finally:
        token = None
    if response.status_code == 200:
        try:
            return 200, kusto_frame(response.json()), ""
        except ValueError:
            return 0, None, "The telemetry endpoint returned something other than Kusto v1 JSON."
    hint = TELEMETRY_HTTP_HINTS.get(response.status_code, "HTTP %d." % response.status_code)
    return response.status_code, None, (hint + " " + _error_text(response)).strip()


def print_kql_fallback(kql, title, reason):
    print("[%s] Not queried from the notebook: %s" % (title, redact(reason)))
    print("Run the KQL below yourself with the documented Azure Data Explorer web UI:")
    print("  1. Open %s and sign in with an account that can read the demo subscription." % ADX_WEB_UI)
    print("  2. Select Add > Connection, and enter the connection URI %s" % TELEMETRY_CONNECTION_URI)
    print("  3. Expand the connection, select the %s database, paste the query, and select Run." % TELEMETRY_DATABASE)
    print("----- KQL: %s -----" % title)
    print(kql)
    print("----- end of KQL -----")


def query_telemetry(kql, title):
    """Runs KQL with the notebook's kusto token. Returns a DataFrame, or None after printing the fallback."""
    if TELEMETRY_STATE["checked"] and not TELEMETRY_STATE["available"]:
        print_kql_fallback(kql, title, TELEMETRY_STATE["message"])
        return None
    status, frame, message = _post_kql(kql)
    if status == 200:
        return frame
    print_kql_fallback(kql, title, message)
    return None

In [ ]:
# Helpers, part 2: analysis results, findings, snapshots, and charts. Run it after part 1.


def _safe_number(value):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return 0.0
    return 0.0 if number != number else number


def to_series(frame, value_column, how="sum", where=None):
    """One value per target and bin, ready to chart: columns Target, Bucket, Value."""
    if frame is None or frame.empty:
        return None
    part = frame if where is None else frame[where(frame)]
    if part.empty:
        return None
    grouped = part.groupby(["Target", "Bucket"], as_index=False)[value_column].agg(how)
    return grouped.rename(columns={value_column: "Value"})


def _naive_utc(value):
    stamp = pd.Timestamp(value)
    return (stamp.tz_convert("UTC") if stamp.tzinfo is not None else stamp).tz_localize(None)


def plot_panels(title, panels, windows=()):
    """Draws one panel per (name, series, unit), one line per target, with scenario windows shaded."""
    panels = [(name, series, unit) for name, series, unit in panels if series is not None and not series.empty]
    if not panels:
        print("%s: nothing to chart." % title)
        return
    if REFERENCE_RUN:
        print("%s: %d panel(s), in the history report and the Insights app; a reference run draws no chart." % (title, len(panels)))
        return
    try:
        import matplotlib.dates as mdates
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib isn't available, so the charts are skipped.")
        return
    figure, axes = plt.subplots(len(panels), 1, figsize=(12, 2.6 * len(panels) + 0.8), sharex=True, squeeze=False)
    for index, (axis, (name, series, unit)) in enumerate(zip(axes[:, 0], panels)):
        for target in sorted(series["Target"].unique()):
            line = series[series["Target"] == target].sort_values("Bucket")
            axis.plot([_naive_utc(value) for value in line["Bucket"]], line["Value"], marker=".", linewidth=1.4,
                      label=target, color=TARGET_COLORS.get(target))
        for window in windows:
            axis.axvspan(_naive_utc(window["start"]), _naive_utc(window["end"]), color="tab:orange", alpha=0.12)
            if index == 0:
                axis.text(_naive_utc(window["start"]), 0.98, " " + window["label"], transform=axis.get_xaxis_transform(),
                          fontsize=7, va="top", color="tab:orange")
        axis.set_title(name, loc="left", fontsize=10)
        axis.set_ylabel(unit)
        axis.grid(alpha=0.3)
        axis.legend(loc="upper right", fontsize=8)
    axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    axes[-1, 0].set_xlabel("UTC (shaded: scenario windows from scenario_runs)")
    figure.suptitle(title)
    figure.tight_layout()
    plt.show()


def add_finding(analysis, target, status, finding, key, evidence=None, request_id=None, start=None, end=None):
    """Records one finding with the ADR 0012 status model. Re-running replaces a finding with the same key."""
    if status not in STATUSES:
        raise ValueError("Unknown status: " + status)
    finding_id = "%s.%s.%s" % (analysis, target, key)
    record = {
        "run_id": RUN_ID, "finding_id": finding_id, "source_notebook": NOTEBOOK_NAME, "captured_at_utc": utc_now(),
        "analysis": analysis, "target": target, "status": status, "finding": redact(finding),
        "evidence": redact(json.dumps(evidence or {}, default=str, sort_keys=True)), "request_id": request_id,
        "window_start_utc": start, "window_end_utc": end,
    }
    FINDINGS[:] = [item for item in FINDINGS if item["finding_id"] != finding_id]
    FINDINGS.append(record)
    print("%-24s [%s] %s" % (status, target, record["finding"]))
    return record


def add_snapshot(analysis, frame, start, end, metrics=None, metric_column=None, value_column=None, units=None):
    """Keeps analysis results in long form for telemetry_snapshots. Re-running replaces that analysis's rows."""
    SNAPSHOT_ROWS[:] = [row for row in SNAPSHOT_ROWS if row["analysis"] != analysis]
    if frame is None or frame.empty:
        return
    captured = utc_now()
    units = units or {}
    for record in frame.to_dict("records"):
        pairs = [(record.get(metric_column), record.get(value_column))] if metric_column else [(metric, record.get(metric)) for metric in metrics]
        for metric, value in pairs:
            if value is None or pd.isna(value):
                continue
            SNAPSHOT_ROWS.append({
                "run_id": RUN_ID, "source_notebook": NOTEBOOK_NAME, "captured_at_utc": captured, "analysis": analysis,
                "target": record.get("Target"), "resource_type": record.get("ResourceType"), "resource_name": record.get("ResourceName"),
                "instance_name": record.get("SqlServerInstanceName"), "database_name": record.get("DatabaseName"),
                "metric": str(metric), "bucket_utc": record.get("Bucket"), "value": float(value), "unit": units.get(metric, ""),
                "window_start_utc": start, "window_end_utc": end,
            })


# Signal families: the expected signal of each scenario (SCENARIO_SIGNALS) and the incident windows' signal_family.
SIGNAL_LABELS = {"blocking": "Blocking", "cpu": "CPU pressure", "io": "IO pressure", "log_rate": "Log-rate pressure",
                 "deadlock": "Deadlocks", "latch": "tempdb latch contention", "memory": "Memory grant pressure",
                 "parallelism": "Parallelism waits", "tran_log": "Log flush waits", "network": "Slow client (network waits)",
                 "activity": "Activity", "connections": "Connections at the limit"}
SIGNAL_UNITS = {"blocking": "processes", "cpu": "%", "io": "s", "log_rate": "s", "deadlock": "deadlocks", "latch": "s",
                "memory": "s", "parallelism": "s", "tran_log": "s", "network": "s", "activity": "requests",
                "connections": "connections"}
# The SqlServerWaitStats category behind each wait family (the telemetry article's WaitCategory values).
WAIT_FAMILY_CATEGORIES = {"log_rate": "Log Rate Governor", "latch": "Buffer Latch", "memory": "Memory",
                          "parallelism": "Parallelism", "tran_log": "Tran Log IO", "network": "Network IO"}
# Detected-episode rules for incident_windows: (signal family, threshold, strictly above it, minimum bins).
EPISODE_RULES = (
    ("blocking", 0.0, True, 1),
    ("cpu", EPISODE_CPU_PERCENT, False, EPISODE_MIN_BINS),
    ("io", EPISODE_PAGEIOLATCH_SECONDS, False, EPISODE_MIN_BINS),
    ("log_rate", 0.0, True, 1),
    ("deadlock", 0.0, True, 1),
    ("latch", EPISODE_WAIT_SECONDS, False, EPISODE_MIN_BINS),
    ("memory", EPISODE_WAIT_SECONDS, False, EPISODE_MIN_BINS),
    ("parallelism", EPISODE_WAIT_SECONDS, False, EPISODE_MIN_BINS),
    ("tran_log", EPISODE_WAIT_SECONDS, False, EPISODE_MIN_BINS),
    ("network", EPISODE_WAIT_SECONDS, False, EPISODE_MIN_BINS),
)


def expected_signals(profile):
    if profile.startswith("incident:"):
        scenario = profile.split(":", 1)[1]
        if scenario in SCENARIO_SIGNALS:
            return SCENARIO_SIGNALS[scenario]
        keywords = (("lock", "blocking"), ("block", "blocking"), ("cpu", "cpu"), ("io", "io"), ("log", "log_rate"), ("index", "cpu"))
        # A keyword starts one of the scenario's words: "io" in io-pressure, never inside "connection" or "transaction"
        # (F34: connection-surge and idle-in-transaction read as IO pressure).
        parts = scenario.split("-")
        return tuple(sorted({signal for word, signal in keywords if any(part.startswith(word) for part in parts)})) or ("activity",)
    return ("activity",)


def signal_episodes(series, threshold, bin_minutes, min_bins, strict=False):
    """Groups consecutive bins at or above a threshold (strictly above it with strict=True) into episodes, per target.
    series has the columns Target, Bucket, and Value (see to_series). Episodes shorter than min_bins bins are dropped;
    each ends one bin after its last bin. Returns dicts with target, start, last, end, peak, and bins."""
    if series is None or series.empty:
        return []
    above = series[series["Value"] > threshold] if strict else series[series["Value"] >= threshold]
    above = above.groupby(["Target", "Bucket"], as_index=False)["Value"].max().sort_values(["Target", "Bucket"])
    step = pd.Timedelta(minutes=int(bin_minutes))
    episodes = []
    for target, group in above.groupby("Target"):
        current = None
        for _, row in group.iterrows():
            if current is not None and row["Bucket"] - current["last"] <= step:
                current["last"] = row["Bucket"]
                current["peak"] = max(current["peak"], float(row["Value"]))
                current["bins"] += 1
            else:
                if current is not None:
                    episodes.append(current)
                current = {"target": target, "start": row["Bucket"], "last": row["Bucket"], "peak": float(row["Value"]), "bins": 1}
        if current is not None:
            episodes.append(current)
    kept = [episode for episode in episodes if episode["bins"] >= int(min_bins)]
    for episode in kept:
        episode["end"] = episode["last"] + step
    return kept


def episode_series(family):
    """The per-target, per-bin series that the episode rules scan, from the frames that sections 6 and 7 queried."""
    if family == "blocking":
        return to_series(FRAMES.get("gauges"), "MaxValue", "max", lambda f: f["CounterName"] == "Processes blocked")
    if family == "cpu":
        return to_series(FRAMES.get("cpu"), "AvgCPU", "max")
    if family == "deadlock":
        return to_series(FRAMES.get("rates"), "Delta", "sum", lambda f: f["CounterName"] == "Number of Deadlocks/sec")
    waits = FRAMES.get("waits")
    if family == "io":
        series = to_series(waits, "WaitMs", "sum", lambda f: f["PageIoLatch"].astype(bool))
    elif family in WAIT_FAMILY_CATEGORIES:
        category = WAIT_FAMILY_CATEGORIES[family]
        series = to_series(waits, "WaitMs", "sum", lambda f: f["WaitCategory"] == category)
    else:
        return None
    if series is not None:
        series["Value"] = series["Value"] / 1000.0
    return series


# ---- T4: PostgreSQL metrics from Azure Monitor (POSTGRES_METRICS, ADR 0025) ----------------------------------------
# This notebook can't read T4 or its read replica: it has no Azure Resource Manager token, and Database Hub's PostgreSQL
# dashboard charts Azure Monitor metrics, which aren't in the telemetry endpoint. The operator's scripts read them with the
# Azure CLI and pass them in POSTGRES_METRICS; these helpers check that document and turn it into this run's rows.
POSTGRES_ANALYSIS = "postgresql"
# The history's target for each server: T4 is the primary (pgflex01-<deploymentId>), T4-replica its read replica.
POSTGRES_TARGETS = {"primary": "T4", "replica": "T4-replica"}
POSTGRES_METRIC_UNITS = {
    "cpu_percent": "percent", "memory_percent": "percent", "active_connections": "connections",
    "connections_failed": "connections", "physical_replication_delay_in_seconds": "seconds",
    "physical_replication_delay_in_bytes": "bytes",
}
POSTGRES_AGGREGATIONS = ("Maximum", "Total", "Average", "Minimum")
# The scripts keep the document under about 24 KB; a longer text isn't theirs.
POSTGRES_MAX_LENGTH = 65536
# A presenter heuristic, not product behavior (ADR 0014): Read replica lag at or above this is a Warning.
REPLICA_LAG_WARNING_SECONDS = 60
# T4's connection slots for clients (F34): max_connections, which the template sets to 50 on every size
# (config/versions.json, postgresql.maxConnections), less the 15 that Azure keeps for physical replication and monitoring
# (reserved_connections plus superuser_reserved_connections:
# https://learn.microsoft.com/azure/postgresql/configure-maintain/concepts-limits). connection-surge's measured signal is
# active_connections at or above them: Azure Monitor counts the reserved slots' sessions too. Its refused attempts
# aren't in connections_failed, which stayed 0 in every interval live (2026-11-21).
POSTGRES_MAX_CONNECTIONS = 50
POSTGRES_RESERVED_CONNECTIONS = 15
POSTGRES_USABLE_CONNECTIONS = POSTGRES_MAX_CONNECTIONS - POSTGRES_RESERVED_CONNECTIONS
POSTGRES_FAILED_CAVEAT = ("They leave out a client refused at the connection limit (too many connections), so T4's active "
                          "connections show a connection surge.")
# Read replica lag right after a gap in its series of at least this many minutes (the replica was stopped, or it started
# after the window began) that reaches half the time since the series' last point is its catch-up after a restart, not
# replication lag: Azure Monitor reports the time since the replica last replayed a transaction (2026-11-16).
REPLICA_RESTART_GAP_MINUTES = 10
_PG_SERVER = re.compile(r"[a-z0-9][a-z0-9-]{1,61}[a-z0-9]")
_PG_SKU = re.compile(r"[A-Za-z0-9_]{0,64}")
_PG_UNIT = re.compile(r"[A-Za-z]{0,32}")
_PG_INSTANT = re.compile(r"\d{4}-\d\d-\d\dT\d\d:\d\d:\d\dZ")


def _pg_instant(text):
    """A yyyy-mm-ddThh:mm:ssZ text as an aware UTC datetime, or None."""
    if not isinstance(text, str) or not _PG_INSTANT.fullmatch(text):
        return None
    try:
        return dt.datetime.strptime(text, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=dt.timezone.utc)
    except ValueError:
        return None


def _pg_whole(value, low, high):
    """An integer from low to high, or None."""
    return value if isinstance(value, int) and not isinstance(value, bool) and low <= value <= high else None


def _pg_iso(value):
    return value.strftime("%Y-%m-%dT%H:%M:%SZ")


def _pg_text(value):
    return value.strftime("%Y-%m-%d %H:%M")


def parse_postgres_metrics(text):
    """Checks POSTGRES_METRICS (version 1) and keeps what's valid. Returns (capture, error): both empty for an empty text;
    capture None and the reason in error for a document it can't use; otherwise capture holds the window (start, end,
    capturedUtc, windowMinutes, intervalMinutes), the servers, the series with their points sorted by time, and skipped,
    how many entries it left out. No network calls."""
    text = str(text or "").strip()
    if not text:
        return None, ""
    if len(text) > POSTGRES_MAX_LENGTH:
        return None, "it's %d characters, more than the %d this notebook reads" % (len(text), POSTGRES_MAX_LENGTH)
    try:
        document = json.loads(text)
    except (ValueError, RecursionError) as error:
        return None, "it isn't JSON (%s)" % redact(error)
    if not isinstance(document, dict) or document.get("version") != 1:
        return None, "it isn't a version 1 document"
    captured = _pg_instant(document.get("capturedUtc"))
    window = _pg_whole(document.get("windowMinutes"), 1, 7 * 24 * 60)
    interval = _pg_whole(document.get("intervalMinutes"), 1, 24 * 60)
    if captured is None or window is None or interval is None:
        return None, "its capturedUtc, windowMinutes (1 to 10080), or intervalMinutes (1 to 1440) is missing or invalid"
    end = _pg_instant(document.get("endUtc")) or captured
    start = _pg_instant(document.get("startUtc")) or end - dt.timedelta(minutes=window)
    if not start < end:
        return None, "its window doesn't start before it ends"
    if not isinstance(document.get("servers"), list) or not isinstance(document.get("series"), list):
        return None, "it has no servers and series lists"
    skipped = 0
    servers = {}
    for item in document["servers"]:
        item = item if isinstance(item, dict) else {}
        role, name, sku = item.get("role"), item.get("resourceName"), item.get("sku") or ""
        if (not isinstance(role, str) or role not in POSTGRES_TARGETS or not isinstance(name, str) or not _PG_SERVER.fullmatch(name)
                or name in servers or not isinstance(sku, str) or not _PG_SKU.fullmatch(sku)
                or any(server["role"] == role for server in servers.values())):
            skipped += 1
            continue
        servers[name] = {"role": role, "target": POSTGRES_TARGETS[role], "resourceName": name, "sku": sku}
    lowest, highest = start - dt.timedelta(minutes=interval), end + dt.timedelta(minutes=interval)
    series, seen = [], set()
    for item in document["series"]:
        item = item if isinstance(item, dict) else {}
        name = item.get("resourceName")
        server = servers.get(name) if isinstance(name, str) else None
        metric, aggregation, unit = item.get("metric"), item.get("aggregation"), item.get("unit") or ""
        if (server is None or item.get("role") != server["role"] or not isinstance(metric, str) or metric not in POSTGRES_METRIC_UNITS
                or aggregation not in POSTGRES_AGGREGATIONS or not isinstance(unit, str) or not _PG_UNIT.fullmatch(unit)
                or not isinstance(item.get("points"), list) or (server["target"], metric) in seen):
            skipped += 1
            continue
        seen.add((server["target"], metric))
        points = {}
        for point in item["points"]:
            at = _pg_instant(point[0]) if isinstance(point, list) and len(point) == 2 else None
            value = point[1] if at is not None else None
            if (at is None or isinstance(value, bool) or not isinstance(value, (int, float)) or value != value
                    or abs(value) == float("inf") or not lowest <= at <= highest):
                skipped += 1
                continue
            points[at] = float(value)
        if points:
            series.append({"target": server["target"], "role": server["role"], "resourceName": server["resourceName"],
                           "metric": metric, "aggregation": aggregation, "unit": POSTGRES_METRIC_UNITS[metric],
                           "points": sorted(points.items())})
    return {"capturedUtc": captured, "start": start, "end": end, "windowMinutes": window, "intervalMinutes": interval,
            "servers": list(servers.values()), "series": series, "skipped": skipped}, ""


def postgres_snapshot_rows(capture, captured_at):
    """telemetry_snapshots rows (analysis postgresql), one per point of a parsed capture, with the capture's window."""
    return [{
        "run_id": RUN_ID, "source_notebook": NOTEBOOK_NAME, "captured_at_utc": captured_at, "analysis": POSTGRES_ANALYSIS,
        "target": series["target"], "resource_type": "flexibleServers", "resource_name": series["resourceName"],
        "instance_name": None, "database_name": None, "metric": series["metric"], "bucket_utc": at, "value": value,
        "unit": series["unit"], "window_start_utc": capture["start"], "window_end_utc": capture["end"],
    } for series in capture["series"] for at, value in series["points"]]


def _pg_series(capture, target, metric):
    return next((series for series in capture["series"] if series["target"] == target and series["metric"] == metric), None)


def _pg_span(capture, series):
    """From the series' first interval to the end of its last one."""
    return series["points"][0][0], series["points"][-1][0] + dt.timedelta(minutes=capture["intervalMinutes"])


def _pg_peak(series):
    """The series' highest point as (time, value); the earliest one when several share it."""
    return max(series["points"], key=lambda point: (point[1], -point[0].timestamp()))


def _pg_lag_catch_up(capture, series):
    """The read replica's lag points that are its catch-up after a restart, not replication lag, as (time, value): Azure
    Monitor reports the time since the replica last replayed a transaction, so a replica that was stopped reports at least
    the time since then until T4 writes again (live, 2026-11-13: 1,380 s at 01:00, before the scenario's 219 s). After a
    gap of REPLICA_RESTART_GAP_MINUTES (and two intervals) or more in the series, or after the window's start, the leading
    points that reach half the time since the last point before the gap are that catch-up."""
    interval = dt.timedelta(minutes=capture["intervalMinutes"])
    gap = max(dt.timedelta(minutes=REPLICA_RESTART_GAP_MINUTES), 2 * interval)
    anchor, resumed, caught = capture["start"], None, []
    for at, value in series["points"]:
        if at - anchor >= gap:
            resumed = anchor
        if resumed is not None and value >= 0.5 * (at + interval - resumed).total_seconds():
            caught.append((at, value))
        else:
            resumed = None
        anchor = at + interval
    return caught


def postgres_findings(capture):
    """The findings of a parsed capture, as dicts with target, key, status, finding, evidence, start, and end: what was
    captured, the read replica's lag, T4's CPU, T4's connections against the slots clients can use (connection-surge's
    signal, F34), and T4's failed connections. Each covers its series' time, or the window."""
    window = "%s to %s UTC" % (_pg_text(capture["start"]), _pg_text(capture["end"]))
    roles = {server["role"]: server for server in capture["servers"]}
    points = sum(len(series["points"]) for series in capture["series"])
    found = [{"target": POSTGRES_TARGETS["primary"], "key": "capture", "start": capture["start"], "end": capture["end"],
              "evidence": {"servers": capture["servers"], "series": len(capture["series"]), "points": points,
                           "skipped": capture["skipped"], "intervalMinutes": capture["intervalMinutes"],
                           "windowMinutes": capture["windowMinutes"], "capturedUtc": _pg_iso(capture["capturedUtc"])}}]
    if not capture["series"]:
        found[0].update(status="Warning", finding="POSTGRES_METRICS held no metric points for T4 or its read replica from %s." % window)
    else:
        described = " and ".join("%s %s%s" % ("T4" if server["role"] == "primary" else "its read replica", server["resourceName"],
                                              " (%s)" % server["sku"] if server["sku"] else "") for server in capture["servers"])
        found[0].update(status="Warning" if capture["skipped"] else "Pass", finding="Azure Monitor metrics of %s from %s, every %d minutes: %d series, %d points%s." % (
            described, window, capture["intervalMinutes"], len(capture["series"]), points,
            "; %d entries that weren't valid were left out" % capture["skipped"] if capture["skipped"] else ""))
    lag = _pg_series(capture, POSTGRES_TARGETS["replica"], "physical_replication_delay_in_seconds")
    if lag is not None:
        # The replica's catch-up after a restart isn't the session's lag: it's left out of the peak, and said (2026-11-16).
        caught = _pg_lag_catch_up(capture, lag)
        kept = [point for point in lag["points"] if point not in caught]
        start, end = _pg_span(capture, lag)
        evidence = {"points": len(lag["points"]), "thresholdSeconds": REPLICA_LAG_WARNING_SECONDS, "resourceName": lag["resourceName"],
                    "catchUpPoints": len(caught)}
        left = ""
        if caught:
            evidence.update(catchUpFromUtc=_pg_iso(caught[0][0]), catchUpPeakSeconds=max(value for _, value in caught))
            left = " Left out: %d point%s from %s UTC, up to %.1f s, the replica's catch-up after its restart (the time since its last replay until T4 wrote again), not replication lag." % (
                len(caught), "" if len(caught) == 1 else "s", _pg_text(caught[0][0]), evidence["catchUpPeakSeconds"])
        if kept:
            at, peak = _pg_peak({"points": kept})
            evidence.update(peakSeconds=peak, peakAtUtc=_pg_iso(at))
            status = "Warning" if peak >= REPLICA_LAG_WARNING_SECONDS else "Pass"
            finding = "Read replica lag on %s peaked at %.1f s at %s UTC (a Warning at %d s or more).%s" % (
                lag["resourceName"], peak, _pg_text(at), REPLICA_LAG_WARNING_SECONDS, left)
        else:
            status = "Pass"
            finding = "Read replica lag on %s: only the replica's catch-up after its restart, up to %.1f s from %s UTC (the time since its last replay until T4 wrote again), not replication lag." % (
                lag["resourceName"], evidence["catchUpPeakSeconds"], _pg_text(caught[0][0]))
        found.append({"target": POSTGRES_TARGETS["replica"], "key": "lag", "start": start, "end": end,
                      "status": status, "finding": finding, "evidence": evidence})
    elif "replica" in roles:
        found.append({"target": POSTGRES_TARGETS["replica"], "key": "lag", "start": capture["start"], "end": capture["end"],
                      "status": "Warning", "evidence": {"resourceName": roles["replica"]["resourceName"]},
                      "finding": "Azure Monitor had no Read replica lag for %s from %s." % (roles["replica"]["resourceName"], window)})
    else:
        found.append({"target": POSTGRES_TARGETS["replica"], "key": "lag", "start": capture["start"], "end": capture["end"],
                      "status": "Not selected", "evidence": {},
                      "finding": "No read replica in this capture, so no Read replica lag: T4's replica exists only during a session "
                                 "(start-demo.ps1 creates it; stop-demo.ps1 captures its metrics, then deletes it)."})
    cpu = _pg_series(capture, POSTGRES_TARGETS["primary"], "cpu_percent")
    if cpu is not None:
        at, peak = _pg_peak(cpu)
        start, end = _pg_span(capture, cpu)
        found.append({"target": POSTGRES_TARGETS["primary"], "key": "cpu", "start": start, "end": end,
                      "status": "Warning" if peak >= EPISODE_CPU_PERCENT else "Pass",
                      "finding": "T4's CPU on %s peaked at %.1f%% at %s UTC (a Warning at %g%% or more)." % (
                          cpu["resourceName"], peak, _pg_text(at), EPISODE_CPU_PERCENT),
                      "evidence": {"peakPercent": peak, "peakAtUtc": _pg_iso(at), "points": len(cpu["points"]),
                                   "thresholdPercent": EPISODE_CPU_PERCENT, "resourceName": cpu["resourceName"]}})
    # T4's connections against the slots clients can use (F34): connection-surge's measured signal. Azure Monitor counts the
    # reserved slots' sessions too, so a surge that fills the slots reads POSTGRES_USABLE_CONNECTIONS or more.
    connections = _pg_series(capture, POSTGRES_TARGETS["primary"], "active_connections")
    if connections is not None:
        at, peak = _pg_peak(connections)
        start, end = _pg_span(capture, connections)
        full = peak >= POSTGRES_USABLE_CONNECTIONS
        found.append({"target": POSTGRES_TARGETS["primary"], "key": "connections", "start": start, "end": end,
                      "status": "Warning" if full else "Pass",
                      "finding": "T4's active connections on %s peaked at %g at %s UTC, %s the %d slots clients can use "
                                 "(max_connections %d, less %d reserved; a Warning at %d or more).%s" % (
                                     connections["resourceName"], peak, _pg_text(at), "at or above" if full else "under",
                                     POSTGRES_USABLE_CONNECTIONS, POSTGRES_MAX_CONNECTIONS, POSTGRES_RESERVED_CONNECTIONS,
                                     POSTGRES_USABLE_CONNECTIONS,
                                     " At the limit, PostgreSQL refuses a new client (too many connections), and Azure Monitor's "
                                     "failed connections don't count that refusal." if full else ""),
                      "evidence": {"peak": peak, "peakAtUtc": _pg_iso(at), "points": len(connections["points"]),
                                   "usableConnections": POSTGRES_USABLE_CONNECTIONS, "maxConnections": POSTGRES_MAX_CONNECTIONS,
                                   "reservedConnections": POSTGRES_RESERVED_CONNECTIONS,
                                   "resourceName": connections["resourceName"]}})
    failed = _pg_series(capture, POSTGRES_TARGETS["primary"], "connections_failed")
    if failed is not None:
        total = int(round(sum(value for _, value in failed["points"])))
        at, peak = _pg_peak(failed)
        start, end = _pg_span(capture, failed)
        span = "%s to %s UTC" % (_pg_text(start), _pg_text(end))
        found.append({"target": POSTGRES_TARGETS["primary"], "key": "connections_failed", "start": start, "end": end,
                      "status": "Warning" if total > 0 else "Pass",
                      "finding": ("Failed connections on T4 (%s): %d from %s, at most %d in one %d-minute interval (at %s UTC). %s" % (
                          failed["resourceName"], total, span, int(peak), capture["intervalMinutes"], _pg_text(at), POSTGRES_FAILED_CAVEAT)
                                  if total > 0 else "Azure Monitor counted no failed connections on T4 (%s) from %s. %s" % (
                                      failed["resourceName"], span, POSTGRES_FAILED_CAVEAT)),
                      "evidence": {"total": total, "peak": peak, "peakAtUtc": _pg_iso(at), "points": len(failed["points"]),
                                   "resourceName": failed["resourceName"]}})
    return found


def record_postgres_metrics(text):
    """Adds POSTGRES_METRICS to this run's results: telemetry_snapshots rows and analysis_findings, analysis postgresql,
    replacing the ones an earlier run of the cell added. A document it can't use is a Fail finding and adds no rows.
    Returns the parsed capture, or None. Writes nothing itself: the persist cell saves the rows."""
    SNAPSHOT_ROWS[:] = [row for row in SNAPSHOT_ROWS if row["analysis"] != POSTGRES_ANALYSIS]
    FINDINGS[:] = [item for item in FINDINGS if item["analysis"] != POSTGRES_ANALYSIS]
    try:
        capture, error = parse_postgres_metrics(text)
    except Exception as failure:  # A document nobody foresaw must not stop the SQL history from being saved.
        capture, error = None, "it couldn't be read (%s)" % redact(failure)
    if capture is None:
        if error:
            add_finding(POSTGRES_ANALYSIS, POSTGRES_TARGETS["primary"], "Fail",
                        "T4's metrics in POSTGRES_METRICS weren't added: %s." % error, "capture",
                        evidence={"characters": len(str(text or ""))})
        return None
    SNAPSHOT_ROWS.extend(postgres_snapshot_rows(capture, utc_now()))
    for item in postgres_findings(capture):
        add_finding(POSTGRES_ANALYSIS, item["target"], item["status"], item["finding"], item["key"], evidence=item["evidence"],
                    start=item["start"], end=item["end"])
    return capture


def postgres_series_frame(metric):
    """This run's T4 rows of one metric as a chart series (columns Target, Bucket, Value), or None."""
    rows = [{"Target": row["target"], "Bucket": row["bucket_utc"], "Value": row["value"]}
            for row in SNAPSHOT_ROWS if row["analysis"] == POSTGRES_ANALYSIS and row["metric"] == metric]
    return pd.DataFrame(rows) if rows else None


# ---- T5: Azure Cosmos DB metrics from Azure Monitor (COSMOS_METRICS, ADR 0027) --------------------------------------
# This notebook can't read T5 either: Database Hub's Cosmos DB views chart Azure Monitor platform metrics, which aren't in
# the telemetry endpoint. The control's runbook and the operator's scripts read them and pass them in COSMOS_METRICS
# (2026-11-18); these helpers check that document and turn it into this run's rows.
COSMOS_ANALYSIS = "cosmosdb"
COSMOS_TARGET = "T5"
# The signals Database Hub's Cosmos DB views chart, and the gateway's server-side latency (the runner uses gateway mode, so
# ServerSideLatencyDirect stays empty), with the history's units. ThrottledRequests is TotalRequests with StatusCode 429.
COSMOS_METRIC_UNITS = {
    "NormalizedRUConsumption": "percent", "TotalRequestUnits": "RU", "TotalRequests": "requests",
    "ThrottledRequests": "requests", "ServerSideLatencyGateway": "ms", "ProvisionedThroughput": "RU/s",
    "ServiceAvailability": "percent",
}
COSMOS_AGGREGATIONS = ("Maximum", "Total", "Count", "Average", "Minimum")
# The scripts keep the document under about 24 KB; a longer text isn't theirs.
COSMOS_MAX_LENGTH = 65536
# Presenter heuristics, not product behavior (ADR 0014): throttled intervals less than this many minutes apart are one
# throttling episode (ru-storm's waves in one showcase slot), and at most this many episodes get a finding each.
COSMOS_EPISODE_GAP_MINUTES = 3
COSMOS_MAX_EPISODES = 24
_COSMOS_ACCOUNT = re.compile(r"[a-z0-9][a-z0-9-]{1,48}[a-z0-9]")


def _cosmos_series(capture, metric):
    return next((series for series in capture["series"] if series["metric"] == metric), None)


def _cosmos_sum(series, start=None, end=None):
    """The sum of a series' points, from start to before end when they're given."""
    points = (series or {}).get("points") or []
    return sum(value for at, value in points if (start is None or at >= start) and (end is None or at < end))


def _cosmos_count(value):
    return format(int(round(value)), ",")


def _cosmos_clock(value):
    return value.strftime("%H:%M")


def parse_cosmos_metrics(text):
    """Checks COSMOS_METRICS (version 1) and keeps what's valid. Returns (capture, error): both empty for an empty text;
    capture None and the reason in error for a document it can't use; otherwise capture holds the window (start, end,
    capturedUtc, windowMinutes, intervalMinutes), the account, the series with their points sorted by time (each with its
    own intervalMinutes: ProvisionedThroughput and ServiceAvailability are hourly), skipped, how many entries it left out,
    and zeroFilled: ThrottledRequests is 0 in each interval with requests and no 429 point, because Azure Monitor sends no
    point for a count of zero. No network calls."""
    text = str(text or "").strip()
    if not text:
        return None, ""
    if len(text) > COSMOS_MAX_LENGTH:
        return None, "it's %d characters, more than the %d this notebook reads" % (len(text), COSMOS_MAX_LENGTH)
    try:
        document = json.loads(text)
    except (ValueError, RecursionError) as error:
        return None, "it isn't JSON (%s)" % redact(error)
    if not isinstance(document, dict) or document.get("version") != 1:
        return None, "it isn't a version 1 document"
    captured = _pg_instant(document.get("capturedUtc"))
    window = _pg_whole(document.get("windowMinutes"), 1, 7 * 24 * 60)
    interval = _pg_whole(document.get("intervalMinutes"), 1, 24 * 60)
    if captured is None or window is None or interval is None:
        return None, "its capturedUtc, windowMinutes (1 to 10080), or intervalMinutes (1 to 1440) is missing or invalid"
    end = _pg_instant(document.get("endUtc")) or captured
    start = _pg_instant(document.get("startUtc")) or end - dt.timedelta(minutes=window)
    if not start < end:
        return None, "its window doesn't start before it ends"
    if not isinstance(document.get("accounts"), list) or not isinstance(document.get("series"), list):
        return None, "it has no accounts and series lists"
    skipped, account = 0, None
    for item in document["accounts"]:
        item = item if isinstance(item, dict) else {}
        name = item.get("resourceName")
        if (account is not None or item.get("target") != COSMOS_TARGET or not isinstance(name, str)
                or not _COSMOS_ACCOUNT.fullmatch(name)):
            skipped += 1
            continue
        account = {"target": COSMOS_TARGET, "resourceName": name}
    series, seen = [], set()
    for item in document["series"]:
        item = item if isinstance(item, dict) else {}
        metric, aggregation, unit = item.get("metric"), item.get("aggregation"), item.get("unit") or ""
        every = interval if item.get("intervalMinutes") is None else _pg_whole(item.get("intervalMinutes"), 1, 24 * 60)
        if (account is None or item.get("resourceName") != account["resourceName"] or not isinstance(metric, str)
                or metric not in COSMOS_METRIC_UNITS or metric in seen or aggregation not in COSMOS_AGGREGATIONS
                or not isinstance(unit, str) or not _PG_UNIT.fullmatch(unit) or every is None
                or item.get("statusCode") != ("429" if metric == "ThrottledRequests" else None)
                or not isinstance(item.get("points"), list)):
            skipped += 1
            continue
        seen.add(metric)
        lowest, highest = start - dt.timedelta(minutes=every), end + dt.timedelta(minutes=every)
        points = {}
        for point in item["points"]:
            at = _pg_instant(point[0]) if isinstance(point, list) and len(point) == 2 else None
            value = point[1] if at is not None else None
            if (at is None or isinstance(value, bool) or not isinstance(value, (int, float)) or value != value
                    or abs(value) == float("inf") or not lowest <= at <= highest):
                skipped += 1
                continue
            points[at] = float(value)
        series.append({"target": COSMOS_TARGET, "resourceName": account["resourceName"], "metric": metric,
                       "aggregation": aggregation, "unit": COSMOS_METRIC_UNITS[metric], "intervalMinutes": every,
                       "points": sorted(points.items())})
    filled = 0
    throttled, requests = _cosmos_series({"series": series}, "ThrottledRequests"), _cosmos_series({"series": series}, "TotalRequests")
    if throttled is not None and requests is not None and throttled["intervalMinutes"] == requests["intervalMinutes"]:
        known = dict(throttled["points"])
        for at, _ in requests["points"]:
            if at not in known:
                known[at] = 0.0
                filled += 1
        throttled["points"] = sorted(known.items())
    # A series without a point says nothing, except the throttled requests: read, and none throttled.
    series = [entry for entry in series if entry["points"] or entry["metric"] == "ThrottledRequests"]
    return {"capturedUtc": captured, "start": start, "end": end, "windowMinutes": window, "intervalMinutes": interval,
            "account": account, "series": series, "skipped": skipped, "zeroFilled": filled}, ""


def cosmos_snapshot_rows(capture, captured_at):
    """telemetry_snapshots rows (analysis cosmosdb, target T5), one per point of a parsed capture, with its window."""
    return [{
        "run_id": RUN_ID, "source_notebook": NOTEBOOK_NAME, "captured_at_utc": captured_at, "analysis": COSMOS_ANALYSIS,
        "target": series["target"], "resource_type": "databaseAccounts", "resource_name": series["resourceName"],
        "instance_name": None, "database_name": None, "metric": series["metric"], "bucket_utc": at, "value": value,
        "unit": series["unit"], "window_start_utc": capture["start"], "window_end_utc": capture["end"],
    } for series in capture["series"] for at, value in series["points"]]


def cosmos_throttle_episodes(capture):
    """The capture's throttling episodes: intervals with throttled requests (429s), those less than
    COSMOS_EPISODE_GAP_MINUTES apart joined into one (ru-storm's waves in one slot). Each is a dict with start, end (the
    end of its last throttled interval), throttled, requests, peakRu (the highest normalized RU in it, or None), stopped
    (a later interval had no 429s), and recoveredAt and recoveredRu: the first interval after it with normalized RU under
    100%, or None. Sorted by start."""
    throttled = _cosmos_series(capture, "ThrottledRequests")
    if throttled is None:
        return []
    every = dt.timedelta(minutes=throttled["intervalMinutes"])
    gap = dt.timedelta(minutes=COSMOS_EPISODE_GAP_MINUTES)
    episodes = []
    for at, value in throttled["points"]:
        if value <= 0:
            continue
        if episodes and at - episodes[-1]["end"] < gap:
            episodes[-1]["end"] = at + every
        else:
            episodes.append({"start": at, "end": at + every})
    requests = _cosmos_series(capture, "TotalRequests")
    normalized = (_cosmos_series(capture, "NormalizedRUConsumption") or {}).get("points") or []
    for episode in episodes:
        episode["throttled"] = _cosmos_sum(throttled, episode["start"], episode["end"])
        episode["requests"] = _cosmos_sum(requests, episode["start"], episode["end"]) if requests else None
        inside = [value for at, value in normalized if episode["start"] <= at < episode["end"]]
        episode["peakRu"] = max(inside) if inside else None
        episode["stopped"] = any(at >= episode["end"] for at, _ in throttled["points"])
        after = next(((at, value) for at, value in normalized if at >= episode["end"] and value < 100), None)
        episode["recoveredAt"], episode["recoveredRu"] = after if after else (None, None)
    return episodes


def cosmos_findings(capture):
    """The findings of a parsed capture, as dicts with target, key, status, finding, evidence, start, and end: what was
    captured; the throttling over the window (normalized RU and the throttled requests, 429s); each throttling episode (at
    most COSMOS_MAX_EPISODES), with when it stopped and normalized RU fell back; and, when T5 was throttled, whether it
    was back at its baseline, with no 429s, before the capture's end."""
    window = "%s to %s UTC" % (_pg_text(capture["start"]), _pg_text(capture["end"]))
    account = (capture["account"] or {}).get("resourceName") or ""
    points = sum(len(series["points"]) for series in capture["series"])
    found = [{"target": COSMOS_TARGET, "key": "capture", "start": capture["start"], "end": capture["end"],
              "evidence": {"account": account, "series": len(capture["series"]), "points": points, "skipped": capture["skipped"],
                           "zeroFilled": capture["zeroFilled"], "intervalMinutes": capture["intervalMinutes"],
                           "windowMinutes": capture["windowMinutes"], "capturedUtc": _pg_iso(capture["capturedUtc"])}}]
    if not points:
        found[0].update(status="Warning", finding="COSMOS_METRICS held no metric points for T5 from %s." % window)
        return found
    found[0].update(status="Warning" if capture["skipped"] else "Pass", finding="Azure Monitor metrics of T5 %s from %s, every %d minutes: %d series, %d points%s." % (
        account, window, capture["intervalMinutes"], len(capture["series"]), points,
        "; %d entries that weren't valid were left out" % capture["skipped"] if capture["skipped"] else ""))
    throttled = _cosmos_series(capture, "ThrottledRequests")
    if throttled is None:
        return found
    normalized = _cosmos_series(capture, "NormalizedRUConsumption")
    requests = _cosmos_series(capture, "TotalRequests")
    total = _cosmos_sum(throttled)
    served = _cosmos_sum(requests) if requests else 0
    peak = _pg_peak(normalized) if normalized and normalized["points"] else None
    episodes = cosmos_throttle_episodes(capture)
    evidence = {"throttled": total, "requests": served, "episodes": len(episodes), "resourceName": account,
                "throttledRatePercent": round(100.0 * total / served, 2) if served else None}
    if peak:
        evidence.update(peakNormalizedRuPercent=peak[1], peakAtUtc=_pg_iso(peak[0]))
    if total <= 0:
        found.append({"target": COSMOS_TARGET, "key": "throttling", "start": capture["start"], "end": capture["end"],
                      "status": "Pass", "evidence": evidence,
                      "finding": "No throttled requests (429s) on T5 %s from %s%s%s." % (
                          account, window, (", %s requests" % _cosmos_count(served)) if served else "",
                          ("; normalized RU consumption peaked at %.0f%% at %s UTC" % (peak[1], _pg_text(peak[0]))) if peak else "")})
        return found
    first, last = episodes[0], episodes[-1]
    found.append({"target": COSMOS_TARGET, "key": "throttling", "start": first["start"], "end": last["end"],
                  "status": "Warning", "evidence": evidence,
                  "finding": "Normalized RU consumption on T5 %s reached %s with %s throttled requests (429s)%s in %d episode%s between %s and %s UTC." % (
                      account, ("%.0f%%" % peak[1]) if peak else "its limit", _cosmos_count(total),
                      (" of %s requests (%.1f%%)" % (_cosmos_count(served), 100.0 * total / served)) if served else "",
                      len(episodes), "" if len(episodes) == 1 else "s", _pg_text(first["start"]), _pg_text(last["end"]))})
    for episode in episodes[:COSMOS_MAX_EPISODES]:
        stopped = episode["stopped"] or episode["recoveredAt"] is not None
        if episode["recoveredAt"] is not None:
            after = "; the throttling stopped at %s UTC, and normalized RU fell back to %.0f%% from %s UTC" % (
                _cosmos_clock(episode["end"]), episode["recoveredRu"], _cosmos_clock(episode["recoveredAt"]))
        elif stopped:
            after = "; the throttling stopped at %s UTC" % _cosmos_clock(episode["end"])
        else:
            after = "; it was still throttled at the capture's end"
        found.append({"target": COSMOS_TARGET, "key": "throttle.%s" % episode["start"].strftime("%Y%m%dT%H%M"),
                      "start": episode["start"], "end": episode["end"], "status": "Warning",
                      "finding": "Normalized RU consumption on T5 reached %s with %s throttled requests (429s)%s between %s and %s UTC%s." % (
                          ("%.0f%%" % episode["peakRu"]) if episode["peakRu"] is not None else "its limit",
                          _cosmos_count(episode["throttled"]),
                          (" of %s requests" % _cosmos_count(episode["requests"])) if episode["requests"] else "",
                          _pg_text(episode["start"]), _cosmos_clock(episode["end"]), after),
                      "evidence": {"throttled": episode["throttled"], "requests": episode["requests"],
                                   "peakNormalizedRuPercent": episode["peakRu"], "stopped": stopped,
                                   "recoveredAtUtc": _pg_iso(episode["recoveredAt"]) if episode["recoveredAt"] else None,
                                   "recoveredNormalizedRuPercent": episode["recoveredRu"]}})
    if last["recoveredAt"] is not None:
        status, text = "Pass", "T5 %s was back at its baseline before the capture's end: its last throttling stopped at %s UTC, and normalized RU consumption fell back to %.0f%% from %s UTC, with no 429s after it." % (
            account, _cosmos_clock(last["end"]), last["recoveredRu"], _cosmos_clock(last["recoveredAt"]))
    elif last["stopped"] and not (normalized and normalized["points"]):
        status, text = "Pass", "T5 %s's last throttling stopped at %s UTC, with no 429s after it before the capture's end." % (
            account, _cosmos_clock(last["end"]))
    elif last["stopped"]:
        status, text = "Warning", "T5 %s's last throttling stopped at %s UTC, but its normalized RU consumption stayed at 100%% until the capture's end, %s UTC." % (
            account, _cosmos_clock(last["end"]), _cosmos_clock(capture["end"]))
    else:
        status, text = "Warning", "T5 %s was still throttled (429s) at the capture's end, %s UTC: its last throttling began at %s UTC." % (
            account, _cosmos_clock(capture["end"]), _cosmos_clock(last["start"]))
    found.append({"target": COSMOS_TARGET, "key": "recovery", "start": last["start"], "end": capture["end"], "status": status,
                  "finding": text, "evidence": {"lastThrottledEndUtc": _pg_iso(last["end"]), "stopped": last["stopped"],
                                                "recoveredAtUtc": _pg_iso(last["recoveredAt"]) if last["recoveredAt"] else None,
                                                "recoveredNormalizedRuPercent": last["recoveredRu"]}})
    return found


def record_cosmos_metrics(text):
    """Adds COSMOS_METRICS to this run's results: telemetry_snapshots rows and analysis_findings, analysis cosmosdb and
    target T5, replacing the ones an earlier run of the cell added. A document it can't use is a Fail finding and adds no
    rows. Returns the parsed capture, or None. Writes nothing itself: the persist cell saves the rows."""
    SNAPSHOT_ROWS[:] = [row for row in SNAPSHOT_ROWS if row["analysis"] != COSMOS_ANALYSIS]
    FINDINGS[:] = [item for item in FINDINGS if item["analysis"] != COSMOS_ANALYSIS]
    try:
        capture, error = parse_cosmos_metrics(text)
    except Exception as failure:  # A document nobody foresaw must not stop the SQL history from being saved.
        capture, error = None, "it couldn't be read (%s)" % redact(failure)
    if capture is None:
        if error:
            add_finding(COSMOS_ANALYSIS, COSMOS_TARGET, "Fail", "T5's metrics in COSMOS_METRICS weren't added: %s." % error,
                        "capture", evidence={"characters": len(str(text or ""))})
        return None
    SNAPSHOT_ROWS.extend(cosmos_snapshot_rows(capture, utc_now()))
    for item in cosmos_findings(capture):
        add_finding(COSMOS_ANALYSIS, item["target"], item["status"], item["finding"], item["key"], evidence=item["evidence"],
                    start=item["start"], end=item["end"])
    return capture


def cosmos_series_frame(metric, label=None):
    """This run's T5 rows of one metric as a chart series (columns Target, Bucket, Value), each named label when it's
    given, or None."""
    rows = [{"Target": label or row["target"], "Bucket": row["bucket_utc"], "Value": row["value"]}
            for row in SNAPSHOT_ROWS if row["analysis"] == COSMOS_ANALYSIS and row["metric"] == metric]
    return pd.DataFrame(rows) if rows else None


In [ ]:
# Helpers, part 3: a demo session's scenario windows (the control notebook's SESSION_PLAN) and Delta tables. Run it after
# part 2. No network calls.
_SCENARIO_ID = re.compile(r"[a-z0-9][a-z0-9-]{0,47}")


def parse_session_plan(session_id, text):
    """SESSION_ID and SESSION_PLAN from NB_DatabaseHub_Control (ADR 0026): the session's window and its beats, each a
    target, a scenario, and a UTC window; the Database Agent beat also has when its run regresses and responds (agent,
    regress, respond; 2026-11-18). Returns None without a session ID; a plan it can't read, or a beat it can't use, is
    left out with a note, so it never stops the analysis."""
    if not session_id:
        return None
    session = {"id": session_id, "start": None, "end": None, "beats": [], "skipped": 0, "note": ""}
    if not text:
        session["note"] = "no beats were passed"
        return session
    try:
        plan = json.loads(text)
    except (ValueError, RecursionError):
        session["note"] = "SESSION_PLAN isn't JSON"
        return session
    if not isinstance(plan, dict) or plan.get("version") != 1 or plan.get("sessionId") not in (None, session_id):
        session["note"] = "SESSION_PLAN isn't a version 1 plan of this session"
        return session

    def instant(value):
        try:
            return _parse_utc(value) if isinstance(value, str) and value.strip() else None
        except (TypeError, ValueError):
            return None

    session["start"], session["end"] = instant(plan.get("startUtc")), instant(plan.get("endUtc"))
    for beat in plan.get("beats") if isinstance(plan.get("beats"), list) else []:
        beat = beat if isinstance(beat, dict) else {}
        start, end = instant(beat.get("startUtc")), instant(beat.get("endUtc"))
        scenario = beat.get("scenario")
        if (beat.get("target") not in ("T1", "T2", "T3", "T4", "T5") or not isinstance(scenario, str) or not _SCENARIO_ID.fullmatch(scenario)
                or start is None or end is None or not start < end):
            session["skipped"] += 1
            continue
        item = {"target": beat["target"], "scenario": scenario, "start": start, "end": end}
        if beat.get("agent") is True:
            # The Database Agent beat: its times count only inside its window, in order.
            regress, respond = instant(beat.get("regressUtc")), instant(beat.get("respondUtc"))
            if regress is not None and respond is not None and start <= regress < respond <= end:
                item.update(agent=True, regress=regress, respond=respond)
        session["beats"].append(item)
    return session


def session_requests(session):
    """The session's scenario_runs rows: one for the session (profile "session", every target) and one per beat on the
    targets this notebook analyzes (SELECTED_TARGETS; T4's and T5's beats show in their own metrics, section 7). Each beat is an
    "incident:<scenario>" window, so section 8 correlates it and section 11 records it in incident_windows."""
    now = utc_now()
    base = {"run_id": RUN_ID, "source_notebook": NOTEBOOK_NAME, "requested_at_utc": session["start"] or now, "environment": "",
            "concurrency": None, "intensity": None, "reset_before": False, "auto_cleanup": True, "dispatch_mode": "session",
            "status": "Pass", "github_run_id": None, "github_run_url": None}
    rows = []
    if session["start"] and session["end"]:
        rows.append(dict(base, request_id="session:%s" % session["id"], target="all", profile="session",
                         duration_minutes=int((session["end"] - session["start"]).total_seconds() // 60),
                         message="Session %s: NB_DatabaseHub_Control's Run all (ADR 0026)." % session["id"][:8],
                         window_start_utc=session["start"], window_end_utc=session["end"]))
    for request_id, beat in session_beats(session):
        if beat["target"] not in SELECTED_TARGETS:
            continue
        message = "Session %s's showcase beat." % session["id"][:8]
        if beat.get("agent"):
            message = "Session %s's Database Agent beat: the plan regresses at %s UTC; the run's response forces the last good plan at %s UTC." % (
                session["id"][:8], beat["regress"].strftime("%H:%M"), beat["respond"].strftime("%H:%M"))
        rows.append(dict(base, request_id=request_id, target=beat["target"],
                         profile="incident:%s" % beat["scenario"], duration_minutes=int((beat["end"] - beat["start"]).total_seconds() // 60),
                         message=message, window_start_utc=beat["start"], window_end_utc=beat["end"]))
    return rows


def session_beats(session):
    """The session's beats in start order, each with its scenario_runs request ID ("session:<id>:<NN>")."""
    ordered = sorted(session["beats"], key=lambda item: (item["start"], item["target"]))
    return [("session:%s:%02d" % (session["id"], index), beat) for index, beat in enumerate(ordered)]


def agent_beat_finding(session_id, beat, cpu_points, covered_until):
    """The database_agent finding of a session's Database Agent beat (2026-11-18): the target's average CPU from 8 to 1
    minutes before the run's response and from 1 to 6 minutes after it, as NB_DatabaseHub_Control's live view checks it.
    cpu_points: (bin start, average CPU percent) pairs for the beat's target; covered_until: the end of the telemetry this
    run reads. Pass when CPU fell by a quarter or 10 points; a Warning when it didn't, or the telemetry doesn't cover the
    response. Returns (status, finding, evidence)."""
    respond = beat["respond"]
    windows = {"before": (respond - dt.timedelta(minutes=8), respond - dt.timedelta(minutes=1)),
               "after": (respond + dt.timedelta(minutes=1), respond + dt.timedelta(minutes=6))}
    average = {}
    for name, (start, end) in windows.items():
        values = [float(value) for bucket, value in cpu_points
                  if start <= bucket <= end and value is not None and float(value) == float(value)]
        average[name] = round(sum(values) / len(values), 1) if values else None
    before, after = average["before"], average["after"]

    def text(moment):
        return moment.strftime("%Y-%m-%dT%H:%M:%SZ")

    evidence = {"sessionId": session_id, "regressUtc": text(beat["regress"]), "respondUtc": text(respond),
                "cpuBeforePercent": before, "cpuAfterPercent": after,
                "beforeStartUtc": text(windows["before"][0]), "beforeEndUtc": text(windows["before"][1]),
                "afterStartUtc": text(windows["after"][0]), "afterEndUtc": text(windows["after"][1]),
                "response": "Query Store's last good plan forced; FORCE_LAST_GOOD_PLAN back on"}
    target = beat["target"]
    head = "Database Agent loop on %s: the plan regressed at %s UTC and the run's response forced the last good plan at %s UTC" % (
        target, beat["regress"].strftime("%H:%M"), respond.strftime("%H:%M"))
    if before is None or after is None or (covered_until is not None and windows["after"][1] > covered_until):
        return ("Warning", "%s; %s's CPU around it isn't in the telemetry this run read, so its drop isn't checked here. "
                "Database Hub's Performance page shows it." % (head, target), evidence)
    if after <= before * 0.75 or before - after >= 10:
        return ("Pass", "%s; %s's CPU fell from %.0f%% to %.0f%%." % (head, target, before, after), evidence)
    return ("Warning", "%s, but %s's CPU didn't fall (%.0f%% before, %.0f%% after), so the regression didn't load it enough "
            "to show the drop. The agent's issue still shows in Database Hub." % (head, target, before, after), evidence)


def session_saved(session_id):
    """Whether scenario_runs already has this session (its "session:<id>" row, written last by section 11). False when
    the table can't be read, so the history is saved rather than missed."""
    session = _spark()
    if session is None or not runtime_context().get("defaultLakehouseId"):
        return False
    try:
        if not session.catalog.tableExists("scenario_runs"):
            return False
        return session.table("scenario_runs").filter("request_id = 'session:%s'" % session_id).limit(1).count() > 0
    except Exception as error:
        print("scenario_runs couldn't be read, so this session's history is saved again if it's there: " + redact(error))
        return False


def saved_earlier(table):
    """Whether an earlier run already saved this session's rows in table (R907, 2026-11-19). Every run of one session saves
    under its run ID, session-<id>, and a cancel can cut a run off after some tables, before its scenario_runs row says
    the session is saved: the app's Stop now cancels the control's run, which may be saving the history. The next run
    (Stop now's, or the next Run all's) then completes the save, table by table, and keeps what the earlier run saved,
    such as the findings that the session's plan gave it, which a run without the plan can't make. False without a
    session, for a table this run saved itself (running the persist cell again replaces its rows), and when the table
    can't be read, so the rows are saved rather than missed."""
    if not SESSION_ID or table in SAVED_TABLES:
        return False
    session = _spark()
    if session is None or not runtime_context().get("defaultLakehouseId"):
        return False
    try:
        if not session.catalog.tableExists(table):
            return False
        return session.table(table).filter("run_id = '%s'" % RUN_ID).limit(1).count() > 0
    except Exception as error:
        print("%s couldn't be read, so this run saves this session's rows there: %s" % (table, redact(error)))
        return False

TABLE_COLUMNS = {
    "validation_results": [
        ("run_id", "string"), ("source_notebook", "string"), ("check_id", "string"), ("area", "string"),
        ("target", "string"), ("status", "string"), ("message", "string"), ("evidence", "string"),
        ("remediation", "string"), ("doc_url", "string"), ("evaluation_only", "boolean"), ("timestamp_utc", "timestamp"),
        ("report_date_utc", "date"),
    ],
    "scenario_runs": [
        ("request_id", "string"), ("run_id", "string"), ("source_notebook", "string"), ("requested_at_utc", "timestamp"),
        ("environment", "string"), ("target", "string"), ("profile", "string"), ("duration_minutes", "integer"),
        ("concurrency", "integer"), ("intensity", "integer"), ("reset_before", "boolean"), ("auto_cleanup", "boolean"),
        ("dispatch_mode", "string"), ("status", "string"), ("message", "string"), ("github_run_id", "long"),
        ("github_run_url", "string"), ("window_start_utc", "timestamp"), ("window_end_utc", "timestamp"),
        ("report_date_utc", "date"),
    ],
    "telemetry_snapshots": [
        ("run_id", "string"), ("source_notebook", "string"), ("captured_at_utc", "timestamp"), ("analysis", "string"),
        ("target", "string"), ("resource_type", "string"), ("resource_name", "string"), ("instance_name", "string"),
        ("database_name", "string"), ("metric", "string"), ("bucket_utc", "timestamp"), ("value", "double"),
        ("unit", "string"), ("window_start_utc", "timestamp"), ("window_end_utc", "timestamp"), ("report_date_utc", "date"),
    ],
    "analysis_findings": [
        ("run_id", "string"), ("finding_id", "string"), ("source_notebook", "string"), ("captured_at_utc", "timestamp"),
        ("analysis", "string"), ("target", "string"), ("status", "string"), ("finding", "string"), ("evidence", "string"),
        ("request_id", "string"), ("window_start_utc", "timestamp"), ("window_end_utc", "timestamp"),
        ("report_date_utc", "date"),
    ],
    # One row per incident window: a recorded incident request, or a detected signal episode (prepare_report_tables).
    "incident_windows": [
        ("incident_key", "string"), ("incident_label", "string"), ("incident_sort", "long"), ("source", "string"),
        ("target", "string"), ("scenario", "string"), ("signal_family", "string"), ("request_id", "string"),
        ("window_start_utc", "timestamp"), ("window_end_utc", "timestamp"), ("duration_minutes", "double"),
        ("peak_value", "double"), ("peak_unit", "string"), ("correlation_status", "string"),
        ("correlation_evidence", "string"), ("run_id", "string"), ("captured_at_utc", "timestamp"),
        ("report_date_utc", "date"),
    ],
}


# The UTC calendar date of each row's primary time (the first one that isn't null). The Power BI history report
# (fabric/powerbi) relates every table to its Date table through this column.
REPORT_DATE_SOURCES = {
    "validation_results": ("timestamp_utc",),
    "scenario_runs": ("window_start_utc", "requested_at_utc"),
    "telemetry_snapshots": ("bucket_utc", "window_end_utc", "captured_at_utc"),
    "analysis_findings": ("window_start_utc", "captured_at_utc"),
    "incident_windows": ("window_start_utc",),
}


def _utc_date(value):
    """UTC calendar date of a timestamp; naive values are already UTC."""
    if value is None:
        return None
    stamp = pd.Timestamp(value)
    if pd.isna(stamp):
        return None
    return (stamp.tz_convert("UTC") if stamp.tzinfo is not None else stamp).date()


def report_date(table, row):
    """report_date_utc for one row: the UTC date of the first primary time in REPORT_DATE_SOURCES that isn't null."""
    for name in REPORT_DATE_SOURCES.get(table, ()):
        value = _utc_date(row.get(name))
        if value is not None:
            return value
    return None


def _struct(table):
    from pyspark.sql import types as spark_types
    mapping = {
        "string": spark_types.StringType(), "boolean": spark_types.BooleanType(), "timestamp": spark_types.TimestampType(),
        "double": spark_types.DoubleType(), "long": spark_types.LongType(), "integer": spark_types.IntegerType(),
        "date": spark_types.DateType(),
    }
    return spark_types.StructType([spark_types.StructField(name, mapping[kind], True) for name, kind in TABLE_COLUMNS[table]])


def _spark_value(value):
    if value is None:
        return None
    if isinstance(value, pd.Timestamp):
        return None if pd.isna(value) else value.to_pydatetime(warn=False)
    if isinstance(value, float) and value != value:
        return None
    if hasattr(value, "item") and not isinstance(value, (str, bytes)):
        return value.item()
    return value


def _rows_frame(session, table, rows):
    """A Spark DataFrame with the TABLE_COLUMNS schema; report_date_utc comes from REPORT_DATE_SOURCES."""
    names = [name for name, _ in TABLE_COLUMNS[table]]
    data = [tuple(_spark_value(report_date(table, row) if name == "report_date_utc" else row.get(name)) for name in names)
            for row in rows]
    return session.createDataFrame(data, schema=_struct(table))


def write_delta(table, rows, replace_run=True):
    """Appends rows to a Delta table in the default lakehouse. Lakehouse data only: no Azure or Fabric resource
    changes. With replace_run, rows from this RUN_ID are deleted first, so re-running a cell doesn't duplicate them. A
    table that an earlier run of this session already saved keeps its rows (saved_earlier), so a save that a cancel cut
    off is completed by the next run, not repeated (R907)."""
    if not rows:
        print("%s: nothing to save." % table)
        return 0
    session = _spark()
    if session is None:
        print("%s: Spark isn't available, so nothing was saved." % table)
        return 0
    if not runtime_context().get("defaultLakehouseId"):
        print("%s: no default lakehouse is attached, so nothing was saved. Attach %s and run the cell again." % (table, LAKEHOUSE_NAME))
        return 0
    if saved_earlier(table):
        print("%s: an earlier run already saved session %s's rows here, so this run keeps them rather than saving them twice." % (table, SESSION_ID[:8]))
        return 0
    frame = _rows_frame(session, table, rows)
    if replace_run and session.catalog.tableExists(table):
        session.sql("DELETE FROM %s WHERE run_id = '%s'" % (table, RUN_ID))
    # One Parquet file per write keeps every table far below Direct Lake's limit of 1,000 files per table (F16).
    frame.coalesce(1).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table)
    SAVED_TABLES.add(table)
    print("%s: saved %d row(s) for run %s." % (table, len(rows), RUN_ID))
    return len(rows)


def scenario_windows(start, end):
    """Scenario windows that overlap [start, end]: this session's requests plus the scenario_runs table."""
    found = {record["request_id"]: record for record in SESSION_REQUESTS}
    session = _spark()
    if session is not None and runtime_context().get("defaultLakehouseId"):
        try:
            if session.catalog.tableExists("scenario_runs"):
                from pyspark.sql import functions as spark_functions
                rows = session.table("scenario_runs").select(
                    "request_id", "profile", "target", "status",
                    spark_functions.unix_timestamp("window_start_utc").alias("start_epoch"),
                    spark_functions.unix_timestamp("window_end_utc").alias("end_epoch")).collect()
                for row in rows:
                    if row["request_id"] not in found and row["start_epoch"] is not None and row["end_epoch"] is not None:
                        found[row["request_id"]] = {
                            "request_id": row["request_id"], "profile": row["profile"], "target": row["target"], "status": row["status"],
                            "window_start_utc": dt.datetime.fromtimestamp(row["start_epoch"], tz=dt.timezone.utc),
                            "window_end_utc": dt.datetime.fromtimestamp(row["end_epoch"], tz=dt.timezone.utc)}
        except Exception as error:
            print("scenario_runs couldn't be read: " + redact(error))
    windows = []
    for record in found.values():
        if record["window_end_utc"] < start or record["window_start_utc"] > end or record["status"] == "Fail":
            continue
        target = record["target"]
        windows.append({
            "request_id": record["request_id"], "profile": record["profile"], "status": record["status"],
            "targets": ["T1", "T2", "T3"] if target == "all" else [target],
            "start": max(record["window_start_utc"], start), "end": min(record["window_end_utc"], end),
            "window_start_utc": record["window_start_utc"], "window_end_utc": record["window_end_utc"],
            "label": "%s (%s)" % (record["profile"], target),
        })
    return sorted(windows, key=lambda item: item["start"])


# ---- Report tables for the Power BI history report (fabric/powerbi; Direct Lake on OneLake) --------------------

SPARK_DDL_TYPES = {"string": "STRING", "boolean": "BOOLEAN", "timestamp": "TIMESTAMP", "double": "DOUBLE", "long": "BIGINT",
                   "integer": "INT", "date": "DATE"}
# OPTIMIZE compacts a table above this many files, far below Direct Lake's limit of 1,000 files per table (F16).
OPTIMIZE_FILE_THRESHOLD = 200
_INCIDENT_KEY = re.compile(r"(request|detected):[A-Za-z0-9:._-]{1,120}")


def _as_utc(value):
    stamp = pd.Timestamp(value)
    return (stamp.tz_localize("UTC") if stamp.tzinfo is None else stamp.tz_convert("UTC")).to_pydatetime()


def _epoch_utc(value):
    return None if value is None else dt.datetime.fromtimestamp(value, tz=dt.timezone.utc)


def _json_object(text):
    """Parses stored evidence JSON back into a dict; anything else is kept as text."""
    try:
        value = json.loads(text or "{}")
    except ValueError:
        return {"raw": text}
    return value if isinstance(value, dict) else {"value": value}


def _sql_in_utc(session, statement):
    """Runs one SQL statement with the session time zone set to UTC, so to_date() returns the UTC calendar date."""
    previous = session.conf.get("spark.sql.session.timeZone")
    session.conf.set("spark.sql.session.timeZone", "UTC")
    try:
        return session.sql(statement)
    finally:
        session.conf.set("spark.sql.session.timeZone", previous)


def _incident_row(key, source, target, family, start, end, peak, scenario=None, request_id=None, status="Pass", evidence=None):
    """One incident_windows row. The label reads like "2026-10-21 14:05 UTC \u00b7 T1 \u00b7 Blocking (detected)", and
    incident_sort is the negative epoch minute of the start, so the report's picker lists the newest first."""
    start, end = _as_utc(start), _as_utc(end)
    origin = "detected" if source == "Detected signal" else "%s, requested" % scenario
    return {
        "incident_key": key,
        "incident_label": "%s UTC \u00b7 %s \u00b7 %s (%s)" % (start.strftime("%Y-%m-%d %H:%M"), target, SIGNAL_LABELS.get(family, "Activity"), origin),
        "incident_sort": -int(start.timestamp() // 60), "source": source, "target": target, "scenario": scenario,
        "signal_family": family, "request_id": request_id, "window_start_utc": start, "window_end_utc": end,
        "duration_minutes": round((end - start).total_seconds() / 60.0, 1), "peak_value": peak,
        "peak_unit": SIGNAL_UNITS.get(family, ""), "correlation_status": status,
        "correlation_evidence": redact(json.dumps(evidence or {}, default=str, sort_keys=True)),
        "run_id": RUN_ID, "captured_at_utc": utc_now(),
    }


def _series_peak(family, target, start, end):
    series = episode_series(family)
    if series is None or series.empty:
        return None
    part = series[(series["Target"] == target) & (series["Bucket"] >= start) & (series["Bucket"] < end)]
    return None if part.empty else round(_safe_number(part["Value"].max()), 3)


def incident_window_rows():
    """This run's incident windows, as rows (writes nothing). Recorded: each incident request in scenario_runs that
    wasn't a Fail, with its signal family, its peak, and section 8's correlation verdict. Detected: each episode that
    EPISODE_RULES find in the telemetry frames; one that overlaps a recorded window on the same target (within
    TELEMETRY_LAG_MINUTES) and matches one of its expected signals becomes evidence for that window instead of a row."""
    lag = dt.timedelta(minutes=int(TELEMETRY_LAG_MINUTES))
    recorded = []
    for window in scenario_windows(WINDOW_START, WINDOW_END):
        if not window["profile"].startswith("incident:"):
            continue
        family = expected_signals(window["profile"])[0]
        start, end = window["window_start_utc"], window["window_end_utc"]
        for target in window["targets"]:
            finding = next((item for item in FINDINGS if item["analysis"] == "correlation" and item["target"] == target
                            and item["request_id"] == window["request_id"]), None)
            if finding is not None:
                status, evidence = finding["status"], _json_object(finding["evidence"])
            else:
                status = "Not selected" if target not in SELECTED_TARGETS else "Manual action required"
                evidence = {"note": "No correlation verdict in this run: run section 8, Correlation."}
            key = "request:%s" % window["request_id"] + ("" if len(window["targets"]) == 1 else ":" + target)
            row = _incident_row(key, "Recorded request", target, family, start, end, _series_peak(family, target, start, end + lag),
                                scenario=window["profile"].split(":", 1)[1], request_id=window["request_id"], status=status)
            row["_evidence"], row["_profile"] = evidence, window["profile"]
            recorded.append(row)
    detected = []
    for family, threshold, strict, min_bins in EPISODE_RULES:
        for episode in signal_episodes(episode_series(family), threshold, BIN, min_bins, strict):
            summary = {"family": family, "start": episode["start"], "end": episode["end"], "peak": round(episode["peak"], 3),
                       "bins": episode["bins"]}
            explained = [row for row in recorded if row["target"] == episode["target"] and family in expected_signals(row["_profile"])
                         and episode["start"] <= row["window_end_utc"] + lag and episode["end"] >= row["window_start_utc"]]
            for row in explained:
                row["_evidence"].setdefault("detectedEpisodes", []).append(summary)
                if family == row["signal_family"] and (row["peak_value"] is None or episode["peak"] > row["peak_value"]):
                    row["peak_value"] = round(episode["peak"], 3)
            if explained:
                continue
            rule = {"threshold": threshold, "strictlyAbove": strict, "minBins": min_bins, "binMinutes": BIN}
            detected.append(_incident_row("detected:%s:%s:%s" % (episode["target"], family, episode["start"].strftime("%Y%m%dT%H%M")),
                                          "Detected signal", episode["target"], family, episode["start"], episode["end"],
                                          round(episode["peak"], 3), evidence=dict(summary, rule=rule)))
    for row in recorded:
        row["correlation_evidence"] = redact(json.dumps(row.pop("_evidence"), default=str, sort_keys=True))
        row.pop("_profile")
    return recorded, detected


def reconcile_recorded(recorded, stored):
    """Recorded requests already in incident_windows (stored: incident_key to a dict with correlation_status,
    correlation_evidence, and peak_value) keep a stored Pass verdict when a later run sees only part of the window,
    and keep the higher peak. Returns the rows to upsert."""
    for row in recorded:
        previous = stored.get(row["incident_key"])
        if previous is None:
            continue
        if previous["correlation_status"] == "Pass" and row["correlation_status"] != "Pass":
            row["correlation_status"], row["correlation_evidence"] = previous["correlation_status"], previous["correlation_evidence"]
        if previous["peak_value"] is not None and (row["peak_value"] is None or previous["peak_value"] > row["peak_value"]):
            row["peak_value"] = previous["peak_value"]
    return recorded


def union_episodes(stored, detected):
    """Unions this run's detected rows with stored detected episodes (dicts with key, target, family, start, end, and
    peak) of the same target and family that overlap or touch them, keeping the earliest stored key, so a re-run never
    duplicates an episode. Returns (rows to upsert, stored keys that another row absorbed)."""
    fresh = [{"kind": "new", "key": row["incident_key"], "target": row["target"], "family": row["signal_family"],
              "start": _as_utc(row["window_start_utc"]), "end": _as_utc(row["window_end_utc"]), "peak": row["peak_value"], "row": row}
             for row in detected]
    old = [dict(item, kind="stored", start=_as_utc(item["start"]), end=_as_utc(item["end"])) for item in stored]
    rows, absorbed = [], []
    for target, family in sorted({(item["target"], item["family"]) for item in fresh}):
        items = sorted((item for item in old + fresh if item["target"] == target and item["family"] == family),
                       key=lambda item: (item["start"], item["kind"] != "stored", item["key"]))
        clusters = []
        for item in items:
            if clusters and item["start"] <= clusters[-1]["end"]:
                clusters[-1]["end"] = max(clusters[-1]["end"], item["end"])
                clusters[-1]["items"].append(item)
            else:
                clusters.append({"start": item["start"], "end": item["end"], "items": [item]})
        for cluster in clusters:
            new_items = [item for item in cluster["items"] if item["kind"] == "new"]
            if not new_items:
                continue
            kept = [item for item in cluster["items"] if item["kind"] == "stored"]
            key = kept[0]["key"] if kept else new_items[0]["key"]
            absorbed += [item["key"] for item in kept[1:]]
            peaks = [item["peak"] for item in cluster["items"] if item["peak"] is not None]
            evidence = _json_object(new_items[0]["row"]["correlation_evidence"])
            if len(cluster["items"]) > 1:
                evidence["mergedFrom"] = sorted({item["key"] for item in cluster["items"]})
            rows.append(_incident_row(key, "Detected signal", target, family, cluster["start"], cluster["end"],
                                      max(peaks) if peaks else None, evidence=evidence))
    return rows, absorbed


def merge_incident_windows(session, recorded, detected):
    """Upserts this run's incident windows into incident_windows (MERGE on incident_key). Detected episodes are first
    unioned with the stored detected rows of the same target and family that overlap or touch them, keeping the
    earliest stored key, so a re-run never duplicates an episode. A recorded request keeps a stored Pass verdict
    when a later run sees only part of its window, and the higher peak. Lakehouse data only."""
    from pyspark.sql import functions as spark_functions
    table = session.table("incident_windows")
    upserts, absorbed = {}, []
    if recorded:
        stored = {row["incident_key"]: row.asDict() for row in table.where(spark_functions.col("incident_key").isin([row["incident_key"] for row in recorded]))
                  .select("incident_key", "correlation_status", "correlation_evidence", "peak_value").collect()}
        upserts.update((row["incident_key"], row) for row in reconcile_recorded(recorded, stored))
    if detected:
        step = dt.timedelta(minutes=int(BIN))
        nearby = table.where((spark_functions.col("source") == "Detected signal")
                             & (spark_functions.col("window_end_utc") >= spark_functions.lit(WINDOW_START - step))
                             & (spark_functions.col("window_start_utc") <= spark_functions.lit(WINDOW_END + step)))
        stored = [{"key": row["incident_key"], "target": row["target"], "family": row["signal_family"],
                   "start": _epoch_utc(row["start_epoch"]), "end": _epoch_utc(row["end_epoch"]), "peak": row["peak_value"]}
                  for row in nearby.select("incident_key", "target", "signal_family", "peak_value",
                                           spark_functions.unix_timestamp("window_start_utc").alias("start_epoch"),
                                           spark_functions.unix_timestamp("window_end_utc").alias("end_epoch")).collect()
                  if row["start_epoch"] is not None and row["end_epoch"] is not None]
        rows, absorbed = union_episodes(stored, detected)
        upserts.update((row["incident_key"], row) for row in rows)
    if upserts:
        _rows_frame(session, "incident_windows", list(upserts.values())).createOrReplaceTempView("incident_windows_updates")
        session.sql("MERGE INTO incident_windows AS target USING incident_windows_updates AS source "
                    "ON target.incident_key = source.incident_key WHEN MATCHED THEN UPDATE SET * WHEN NOT MATCHED THEN INSERT *")
    absorbed = sorted(key for key in set(absorbed) - set(upserts) if _INCIDENT_KEY.fullmatch(key))
    if absorbed:
        session.sql("DELETE FROM incident_windows WHERE incident_key IN (%s)" % ", ".join("'%s'" % key for key in absorbed))
    return len(upserts), len(absorbed)


def prepare_report_tables():
    """Readies the tables that the Power BI history report reads (SM_DatabaseHub_History, Direct Lake on OneLake), the five
    history tables and the Incident RCA's eight (RCA_TABLES),
    which can't frame a missing table or column: CREATE TABLE IF NOT EXISTS for each, report_date_utc added to tables
    written before it existed and backfilled once, this run's incident windows merged (unless an earlier run of this
    session merged them: saved_earlier), and OPTIMIZE for any table with more than OPTIMIZE_FILE_THRESHOLD files.
    Lakehouse data only: no Azure or Fabric resource changes."""
    session = _spark()
    if session is None or not runtime_context().get("defaultLakehouseId"):
        print("Report tables: Spark or the default lakehouse isn't available, so nothing was prepared. Attach %s." % LAKEHOUSE_NAME)
        return {}
    summary = {"created": [], "columnAdded": [], "backfilled": {}, "optimized": []}
    for table, columns in TABLE_COLUMNS.items():
        if not session.catalog.tableExists(table):
            summary["created"].append(table)
        session.sql("CREATE TABLE IF NOT EXISTS %s (%s) USING DELTA" % (table, ", ".join("%s %s" % (name, SPARK_DDL_TYPES[kind]) for name, kind in columns)))
        if "report_date_utc" not in {field.name for field in session.table(table).schema.fields}:
            session.sql("ALTER TABLE %s ADD COLUMNS (report_date_utc DATE)" % table)
            summary["columnAdded"].append(table)
        missing = int(session.sql("SELECT COUNT(*) AS n FROM %s WHERE report_date_utc IS NULL" % table).collect()[0]["n"])
        if missing:
            # Rows written before the column existed (also when an append with mergeSchema added it): a one-time backfill.
            _sql_in_utc(session, "UPDATE %s SET report_date_utc = to_date(coalesce(%s)) WHERE report_date_utc IS NULL"
                        % (table, ", ".join(REPORT_DATE_SOURCES[table])))
            summary["backfilled"][table] = missing
    if saved_earlier("incident_windows"):
        # Not again: a run without the session's plan (the control's re-save of a session whose status lost it) would
        # take its beats' signals for detected episodes, beside the windows that the earlier run recorded for them (R907).
        print("incident_windows: an earlier run already merged session %s's incident windows, so this run keeps them." % SESSION_ID[:8])
        summary["incidentWindowsMerged"], summary["incidentWindowsAbsorbed"] = 0, 0
    else:
        recorded, detected = incident_window_rows()
        summary["incidentWindowsMerged"], summary["incidentWindowsAbsorbed"] = merge_incident_windows(session, recorded, detected)
        SAVED_TABLES.add("incident_windows")
    for table in TABLE_COLUMNS:
        files = session.sql("DESCRIBE DETAIL %s" % table).collect()[0]["numFiles"]
        if files and int(files) > OPTIMIZE_FILE_THRESHOLD:
            session.sql("OPTIMIZE %s" % table)
            summary["optimized"].append(table)
    print("Report tables ready for the Power BI history report: " + json.dumps(summary, sort_keys=True))
    return summary


WINDOW_START, WINDOW_END, BIN = analysis_window()
WINDOWS = []
print("Helpers ready. Run ID: %s. Targets: %s." % (RUN_ID, ", ".join(SELECTED_TARGETS)))

In [ ]:
# Helpers, part 4: the Incident RCA's evidence (ADR 0031). Run it after part 3. It reads RCA_EVIDENCE, or the file in this
# lakehouse's Files/rca/inbox that RCA_EVIDENCE_FILE names, decodes its part lines into evidence documents, checks each
# document against the contract (rca/evidence.schema.json and rca/history-tables.json, version 1), and writes the seven
# detection tables and the answer key's own table, rca_answer_key, which nothing in this notebook reads back. Lakehouse
# data only, and no network call: the staged file is read and deleted with notebookutils.fs
# (https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-file-system) by its OneLake path
# (https://learn.microsoft.com/fabric/onelake/onelake-access-api).
import base64
import hashlib
import zlib

# The eight tables, their columns in the contract's order, the required ones, and the primary time that report_date_utc
# comes from (rca/history-tables.json, version 1; tests/unit/Fabric.Notebooks.Tests.ps1 checks them against it). Each
# table also has the notebook's own columns: run_id and document_id first, report_date_utc last.
RCA_TABLES = {
    "rca_sessions": {
        "role": "detection", "primary": "captured_at_utc",
        "columns": [
            ("captured_at_utc", "timestamp"), ("target", "string"), ("session_id", "long"), ("blocked_by", "string"),
            ("status", "string"), ("wait_type", "string"), ("wait_ms", "long"), ("wait_resource", "string"),
            ("lock_resource_type", "string"), ("lock_object", "string"), ("lock_index", "string"),
            ("lock_mode_wanted", "string"), ("lock_mode_held", "string"), ("open_transactions", "integer"),
            ("transaction_age_ms", "long"), ("idle_ms", "long"), ("login_name", "string"), ("host_name", "string"),
            ("program_name", "string"), ("run_token", "string"), ("module_name", "string"),
            ("statement_text", "string"), ("input_buffer", "string"), ("backend_type", "string"),
            ("is_demo", "boolean"),
        ],
        "required": ("captured_at_utc", "target", "session_id", "is_demo"),
    },
    "rca_waits": {
        "role": "detection", "primary": "window_start_utc",
        "columns": [
            ("target", "string"), ("episode_id", "string"), ("phase", "string"), ("window_start_utc", "timestamp"),
            ("window_end_utc", "timestamp"), ("wait_type", "string"), ("wait_category", "string"), ("wait_ms", "long"),
            ("waiting_tasks", "long"), ("signal_ms", "long"), ("max_wait_ms", "long"),
        ],
        "required": ("target", "phase", "window_start_utc", "window_end_utc", "wait_type", "wait_ms"),
    },
    "rca_events": {
        "role": "detection", "primary": "event_time_utc",
        "columns": [
            ("event_time_utc", "timestamp"), ("target", "string"), ("kind", "string"), ("event_name", "string"),
            ("session_id", "long"), ("program_name", "string"), ("run_token", "string"), ("login_name", "string"),
            ("host_name", "string"), ("module_name", "string"), ("code", "string"), ("severity", "integer"),
            ("intercepted", "boolean"), ("event_count", "long"), ("object_name", "string"), ("lock_mode", "string"),
            ("detail", "string"), ("deadlock_json", "string"),
        ],
        "required": ("event_time_utc", "target", "kind", "event_count", "detail"),
    },
    "rca_queries": {
        "role": "detection", "primary": "interval_start_utc",
        "columns": [
            ("target", "string"), ("episode_id", "string"), ("phase", "string"), ("interval_start_utc", "timestamp"),
            ("interval_end_utc", "timestamp"), ("query_id", "long"), ("plan_id", "long"), ("query_hash", "string"),
            ("module_name", "string"), ("statement_text", "string"), ("executions", "long"), ("cpu_ms", "double"),
            ("duration_ms", "double"), ("logical_reads", "double"), ("physical_reads", "double"),
            ("log_bytes", "double"), ("max_used_memory_kb", "double"), ("max_dop", "long"),
            ("plans_for_query", "integer"), ("is_forced_plan", "boolean"), ("plan_forcing_type", "string"),
        ],
        "required": ("target", "phase", "interval_start_utc", "interval_end_utc", "query_id", "plan_id", "executions"),
    },
    "rca_resource": {
        "role": "detection", "primary": "at_utc",
        "columns": [
            ("at_utc", "timestamp"), ("target", "string"), ("source", "string"), ("metric", "string"),
            ("value", "double"), ("unit", "string"),
        ],
        "required": ("at_utc", "target", "source", "metric", "value"),
    },
    "rca_replica": {
        "role": "detection", "primary": "at_utc",
        "columns": [
            ("at_utc", "timestamp"), ("target", "string"), ("side", "string"), ("standby_name", "string"),
            ("since_last_replayed_s", "double"), ("received_not_replayed_bytes", "double"), ("replay_lag_s", "double"),
            ("write_lag_s", "double"), ("flush_lag_s", "double"), ("send_gap_bytes", "double"),
            ("wal_bytes_per_s", "double"), ("startup_wait", "string"), ("conflict_relation", "string"),
            ("conflict_wanted_mode", "string"), ("conflict_wait_start_utc", "timestamp"),
            ("conflict_holder_session", "long"), ("conflict_holder_mode", "string"),
            ("conflict_holder_program", "string"), ("conflict_holder_run_token", "string"),
            ("conflict_holder_statement", "string"), ("conflicts_cancelled", "long"),
            ("max_standby_streaming_delay_s", "double"),
        ],
        "required": ("at_utc", "target", "side"),
    },
    "rca_coverage": {
        "role": "detection", "primary": "window_start_utc",
        "columns": [
            ("target", "string"), ("window_start_utc", "timestamp"), ("window_end_utc", "timestamp"),
            ("captured", "boolean"), ("sources", "string"), ("gaps", "string"),
        ],
        "required": ("target", "window_start_utc", "window_end_utc", "captured"),
    },
    "rca_answer_key": {
        "role": "answerKey", "primary": "event_time_utc",
        "columns": [
            ("kind", "string"), ("target", "string"), ("event_time_utc", "timestamp"), ("end_time_utc", "timestamp"),
            ("scenario", "string"), ("scenario_run_id", "string"), ("run_token", "string"), ("event_name", "string"),
            ("outcome", "string"), ("detail", "string"),
        ],
        "required": ("kind", "target", "event_time_utc", "scenario", "scenario_run_id", "run_token"),
    },
}
for _table, _spec in RCA_TABLES.items():
    TABLE_COLUMNS[_table] = [("run_id", "string"), ("document_id", "string")] + _spec["columns"] + [("report_date_utc", "date")]
    REPORT_DATE_SOURCES[_table] = (_spec["primary"],)
RCA_DETECTION_TABLES = tuple(name for name, spec in RCA_TABLES.items() if spec["role"] == "detection")
# rca_coverage goes last: a document's coverage rows say that all its rows are written (rca_documents_saved).
RCA_WRITE_ORDER = tuple(name for name in RCA_TABLES if name != "rca_coverage") + ("rca_coverage",)
RCA_TARGETS = ("T1", "T2", "T3", "T4", "T4-replica", "T5")
RCA_PART_LINE = re.compile(r"DBHUBDEMO-RCA 1 ([0-9]{8}T[0-9]{6}Z-[a-z0-9]+(?:-[a-z0-9]+)*-[0-9]{4}) ([1-9][0-9]?)/([1-9][0-9]?) "
                           r"([0-9a-f]{64}) ([A-Za-z0-9+/]+={0,2})")
RCA_DOCUMENT_ID = re.compile(r"[0-9]{8}T[0-9]{6}Z-[a-z0-9]+(?:-[a-z0-9]+)*-[0-9]{4}")
RCA_UTC = re.compile(r"[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}(?:\.[0-9]{1,7})?Z")
RCA_SESSION = re.compile(r"[0-9a-f]{32}")
RCA_HOST = re.compile(r"[a-z0-9][a-z0-9-]{0,39}")
# RCA_EVIDENCE_FILE: a file in the inbox, as rca/transport.schema.json's evidenceFile says (no other folder, no
# traversal). The control notebook and the scripts name theirs <when it was staged, UTC>-<name>.json, so a file that a
# failed run left is found by its age (RCA_INBOX_FILE).
RCA_INBOX = "Files/rca/inbox"
RCA_EVIDENCE_FILE_PATTERN = re.compile(r"Files/rca/inbox/[A-Za-z0-9][A-Za-z0-9._-]{0,127}\.json")
RCA_INBOX_FILE = re.compile(r"Files/rca/inbox/([0-9]{8}T[0-9]{6}Z)-[a-z0-9-]{1,64}\.json")
RCA_INBOX_KEEP_HOURS = 24
# The contract's limits (rca/README.md, Limits), and what a document may hold uncompressed, so a corrupt stream can't
# fill the driver's memory.
RCA_LIMITS = {"parts": 64, "partChars": 3400, "rows": 20000, "strings": 20000, "stringChars": 4000, "message": 500,
              "fileBytes": 262144, "documentBytes": 32 * 1024 * 1024}
_RCA_INT64, _RCA_INT32 = (-2 ** 63, 2 ** 63 - 1), (-2 ** 31, 2 ** 31 - 1)
_RCA_MEMBERS = ("kind", "version", "documentId", "sessionId", "producer", "exportedUtc", "sequence", "final", "windowStartUtc",
                "windowEndUtc", "targets", "epochUtc", "strings", "tables", "answerKey")
# This run's RCA.EVIDENCE: saved once per run (save_rca_evidence), whichever cell gets there first.
RCA_STATE = {"done": False, "passed": False, "status": "Not selected", "message": "", "documents": [], "rows": {}, "skipped": []}


def _rca_path(relative):
    """The OneLake path of a file in this notebook's lakehouse, by the workspace's and the lakehouse's IDs."""
    return "abfss://%s@onelake.dfs.fabric.microsoft.com/%s/%s" % (WORKSPACE_ID, LAKEHOUSE_ID, relative)


def _rca_json(text):
    """JSON without NaN or Infinity, which isn't JSON and no column may hold."""
    def refuse(name):
        raise ValueError("%s isn't JSON" % name)
    return json.loads(text, parse_constant=refuse)


def _rca_int(value, bounds=_RCA_INT64):
    return isinstance(value, int) and not isinstance(value, bool) and bounds[0] <= value <= bounds[1]


def rca_parameter_problem(value):
    """Why value isn't a version 1 RCA_EVIDENCE document (rca/transport.schema.json, parameter), or "". A part line it
    can't read is left to decode_rca_parts, which skips its document and says so."""
    if not isinstance(value, dict) or value.get("kind") != "ContosoCare.RcaEvidenceParts":
        return "it isn't a ContosoCare.RcaEvidenceParts document"
    if value.get("version") != 1:
        return "its version %s is newer than this notebook's 1 (./scripts/deploy.ps1 updates the notebook)" % str(value.get("version"))[:20]
    extra = sorted(str(name) for name in set(value) - {"kind", "version", "sessionId", "parts", "status", "message"})
    if extra:
        return "it has members that version 1 doesn't have (%s)" % ", ".join(extra)[:200]
    session = value.get("sessionId", "")
    if session is not None and not (isinstance(session, str) and RCA_SESSION.fullmatch(session)):
        return "its sessionId isn't a session ID or null"
    parts = value.get("parts")
    if not isinstance(parts, list) or not 1 <= len(parts) <= RCA_LIMITS["parts"]:
        return "it must hold 1 to %d part lines" % RCA_LIMITS["parts"]
    if any(not isinstance(part, str) or len(part) > RCA_LIMITS["partChars"] for part in parts):
        return "a part line isn't text of at most %d characters" % RCA_LIMITS["partChars"]
    if "status" in value and value["status"] not in STATUSES:
        return "its status isn't one of ADR 0012's"
    if "message" in value and not (isinstance(value["message"], str) and len(value["message"]) <= RCA_LIMITS["message"]):
        return "its message isn't text of at most %d characters" % RCA_LIMITS["message"]
    return ""


def _rca_evidence_value(text, staged):
    """One RCA_EVIDENCE document from its text, checked whole: (the document, "") or (None, why it can't be used)."""
    if not text:
        return None, "the staged file is empty" if staged else "it's empty"
    try:
        value = _rca_json(text)
    except (ValueError, RecursionError):
        return None, "it isn't JSON, or it's larger than %d bytes" % RCA_LIMITS["fileBytes"]
    problem = rca_parameter_problem(value)
    return (None if problem else value), problem


def read_rca_evidence():
    """The RCA_EVIDENCE documents passed: the staged file that RCA_EVIDENCE_FILE names, and the inline RCA_EVIDENCE, each
    checked whole. With both, their parts are merged (rca/README.md, RCA_EVIDENCE_FILE: both parameters), and
    decode_rca_parts still decodes each document once. Returns (the part lines, [] when neither can be used; (where, why)
    for each one that can't be used; whether the staged file stays for a later run, because it couldn't be read). Nothing
    here stops the analysis."""
    parts, problems, keep = [], [], False
    if RCA_EVIDENCE_FILE:
        value, why, utils = None, "", _notebookutils()
        if not RCA_EVIDENCE_FILE_PATTERN.fullmatch(RCA_EVIDENCE_FILE):
            why = "RCA_EVIDENCE_FILE must name a file in %s, as the control notebook and the scripts stage it" % RCA_INBOX
        elif utils is None or not WORKSPACE_ID or not LAKEHOUSE_ID:
            why, keep = "notebookutils or the lakehouse's IDs aren't available, so the staged file wasn't read", True
        else:
            try:
                text = utils.fs.head(_rca_path(RCA_EVIDENCE_FILE), RCA_LIMITS["fileBytes"])
            except Exception as error:
                text, why, keep = "", "the staged file couldn't be read (%s)" % redact(error)[:300], True
            if not why:
                value, why = _rca_evidence_value(text, True)
        if value is None:
            problems.append((RCA_EVIDENCE_FILE, why))
        else:
            parts.extend(value["parts"])
    if RCA_EVIDENCE:
        value, why = _rca_evidence_value(RCA_EVIDENCE, False)
        if value is None:
            problems.append(("RCA_EVIDENCE", why))
        else:
            parts.extend(value["parts"])
    return list(dict.fromkeys(parts)), problems, keep


def _rca_gunzip(stream):
    """A document's gzip stream as UTF-8 text, at most RCA_LIMITS["documentBytes"]: (text, "") or ("", why)."""
    inflater = zlib.decompressobj(16 + zlib.MAX_WBITS)
    try:
        data = inflater.decompress(stream, RCA_LIMITS["documentBytes"] + 1)
    except zlib.error:
        return "", "it isn't gzip"
    if len(data) > RCA_LIMITS["documentBytes"] or inflater.unconsumed_tail:
        return "", "it's larger than %d bytes uncompressed" % RCA_LIMITS["documentBytes"]
    if not inflater.eof or inflater.unused_data:
        return "", "its gzip stream is cut short or has bytes after its end"
    try:
        return data.decode("utf-8"), ""
    except UnicodeDecodeError:
        return "", "it isn't UTF-8"


def _rca_strings_problem(strings, section):
    if not isinstance(strings, list) or len(strings) > RCA_LIMITS["strings"]:
        return "%s's strings must be a list of at most %d" % (section, RCA_LIMITS["strings"])
    if any(not isinstance(item, str) or len(item) > RCA_LIMITS["stringChars"] for item in strings):
        return "a string in %s's strings isn't text of at most %d characters" % (section, RCA_LIMITS["stringChars"])
    return ""


def _rca_table_problem(name, table, strings):
    """Why table isn't a version 1 table name with strings as its dictionary (rca/README.md, Evidence documents), or ""."""
    spec = RCA_TABLES[name]
    if not isinstance(table, dict) or set(table) != {"rows", "columns"}:
        return "%s must hold rows and columns only" % name
    rows, columns = table["rows"], table["columns"]
    if not _rca_int(rows, (1, RCA_LIMITS["rows"])) or not isinstance(columns, dict):
        return "%s's rows must be 1 to %d, with its columns" % (name, RCA_LIMITS["rows"])
    kinds = dict(spec["columns"])
    unknown = sorted(str(column) for column in set(columns) - set(kinds))
    if unknown:
        return "%s has columns that version 1 doesn't have (%s)" % (name, ", ".join(unknown)[:200])
    for column in spec["required"]:
        if column not in columns:
            return "%s lacks its required column %s" % (name, column)
    for column, values in columns.items():
        if not isinstance(values, list) or len(values) != rows:
            return "%s.%s doesn't hold %d values" % (name, column, rows)
        kind, required = kinds[column], column in spec["required"]
        for value in values:
            if value is None:
                if required:
                    return "%s.%s is required, but holds a null" % (name, column)
                continue
            if kind == "string":
                good = _rca_int(value, (0, len(strings) - 1))
            elif kind in ("timestamp", "long"):
                good = _rca_int(value)
            elif kind == "integer":
                good = _rca_int(value, _RCA_INT32)
            elif kind == "double":
                good = isinstance(value, (int, float)) and not isinstance(value, bool)
            else:
                good = isinstance(value, bool)
            if not good:
                return "%s.%s holds a value that isn't a %s%s" % (name, column, kind, " index" if kind == "string" else "")
    return ""


def check_rca_document(document):
    """Why document isn't a version 1 evidence document this notebook can write (rca/evidence.schema.json, and the rules
    rca/README.md adds), or "". A document is checked whole before any of its rows is written."""
    if not isinstance(document, dict) or document.get("kind") != "ContosoCare.RcaEvidence":
        return "it isn't a ContosoCare.RcaEvidence document"
    if document.get("version") != 1:
        return "its version %s is newer than this notebook's 1 (./scripts/deploy.ps1 updates the notebook)" % str(document.get("version"))[:20]
    missing = [name for name in _RCA_MEMBERS if name not in document]
    extra = sorted(str(name) for name in set(document) - set(_RCA_MEMBERS) - {"$schema"})
    if missing or extra:
        return "it lacks %s or has members that version 1 doesn't have (%s)" % (", ".join(missing) or "nothing", ", ".join(extra)[:200] or "none")
    if not isinstance(document["documentId"], str) or len(document["documentId"]) > 80 or not RCA_DOCUMENT_ID.fullmatch(document["documentId"]):
        return "its documentId isn't one"
    session = document["sessionId"]
    if session is not None and not (isinstance(session, str) and RCA_SESSION.fullmatch(session)):
        return "its sessionId isn't a session ID or null"
    producer = document["producer"]
    if (not isinstance(producer, dict) or set(producer) != {"name", "version", "host", "lane"} or producer["name"] != "ContosoCare.Workload"
            or producer["lane"] != "observer" or not isinstance(producer["version"], str) or not 1 <= len(producer["version"]) <= 64
            or not isinstance(producer["host"], str) or not RCA_HOST.fullmatch(producer["host"])):
        return "its producer isn't the workload's observer"
    for name in ("exportedUtc", "windowStartUtc", "windowEndUtc", "epochUtc"):
        if not isinstance(document[name], str) or not RCA_UTC.fullmatch(document[name]):
            return "its %s isn't a UTC time" % name
    if not _rca_int(document["sequence"], (1, 9999)) or not isinstance(document["final"], bool):
        return "its sequence or final isn't one"
    targets = document["targets"]
    if (not isinstance(targets, list) or not targets or any(not isinstance(target, str) or target not in RCA_TARGETS for target in targets)
            or len(set(targets)) != len(targets)):
        return "its targets must be T1 to T5 or T4-replica, each once"
    problem = _rca_strings_problem(document["strings"], "the document")
    if problem:
        return problem
    tables = document["tables"]
    if not isinstance(tables, dict) or "rca_coverage" not in tables:
        return "its tables must include rca_coverage"
    for name, table in tables.items():
        if name not in RCA_DETECTION_TABLES:
            return "its tables hold %s, which version 1's detection tables don't have" % str(name)[:60]
        problem = _rca_table_problem(name, table, document["strings"])
        if problem:
            return problem
    key = document["answerKey"]
    if not isinstance(key, dict) or set(key) != {"strings", "tables"} or not isinstance(key["tables"], dict):
        return "its answerKey must hold strings and tables only"
    if set(key["tables"]) - {"rca_answer_key"}:
        return "its answerKey holds a table other than rca_answer_key"
    problem = _rca_strings_problem(key["strings"], "the answer key")
    if not problem and "rca_answer_key" in key["tables"]:
        problem = _rca_table_problem("rca_answer_key", key["tables"]["rca_answer_key"], key["strings"])
    return problem


def decode_rca_parts(parts):
    """Groups the part lines by documentId and decodes each document as rca/README.md (Part lines) says: each part's base64
    decoded on its own, concatenated in index order, the SHA-256 of the whole gzip stream checked, then gunzipped, parsed,
    and checked (check_rca_document). Returns ({documentId: document}, [(documentId, why it was skipped)]). A document with
    a missing, bad, or conflicting part is skipped, never written in part."""
    groups, skipped, unreadable = {}, [], 0
    for line in parts:
        match = RCA_PART_LINE.fullmatch(line)
        if not match:
            unreadable += 1
            continue
        document_id, index, count, digest, data = match.group(1), int(match.group(2)), int(match.group(3)), match.group(4), match.group(5)
        group = groups.setdefault(document_id, {"count": count, "sha256": digest, "parts": {}, "conflict": False})
        if group["count"] != count or group["sha256"] != digest or index > count or group["parts"].get(index, data) != data:
            group["conflict"] = True
        group["parts"][index] = data
    if unreadable:
        skipped.append(("", "%d part line(s) that part format 1 doesn't describe (a newer runner? ./scripts/deploy.ps1 updates the notebook)" % unreadable))
    documents = {}
    for document_id, group in sorted(groups.items()):
        missing = [str(index) for index in range(1, group["count"] + 1) if index not in group["parts"]]
        if group["conflict"]:
            why = "its parts disagree on their count, their SHA-256, or a part's content"
        elif missing:
            why = "part(s) %s of %d are missing" % (", ".join(missing[:5]), group["count"])
        else:
            why = ""
            try:
                stream = b"".join(base64.b64decode(group["parts"][index], validate=True) for index in range(1, group["count"] + 1))
            except ValueError:
                stream, why = b"", "a part isn't base64"
            if not why and hashlib.sha256(stream).hexdigest() != group["sha256"]:
                why = "its SHA-256 doesn't match its parts"
            if not why:
                text, why = _rca_gunzip(stream)
            if not why:
                try:
                    document = _rca_json(text)
                except (ValueError, RecursionError):
                    document, why = None, "it isn't JSON"
            if not why:
                why = check_rca_document(document)
            if not why and document["documentId"] != document_id:
                why = "its documentId isn't the one its parts name"
        if why:
            skipped.append((document_id, why))
        else:
            documents[document_id] = document
    return documents, skipped


def rca_rows(document):
    """The document's rows by table, decoded: strings from their section's dictionary, through redact(); times from
    epochUtc; doubles as floats; the notebook's run_id and document_id added (report_date_utc comes with the write). The
    answer key's rows come only from its own section and its own dictionary, so no detection row carries any of it."""
    epoch = pd.Timestamp(document["epochUtc"]).tz_convert("UTC").to_pydatetime(warn=False)
    sections = [(name, table, document["strings"]) for name, table in document["tables"].items()]
    sections += [(name, table, document["answerKey"]["strings"]) for name, table in document["answerKey"]["tables"].items()]
    rows_by_table = {}
    for name, table, strings in sections:
        kinds = dict(RCA_TABLES[name]["columns"])
        decoded = [redact(text) for text in strings]
        rows = [{"run_id": RUN_ID, "document_id": document["documentId"]} for _ in range(table["rows"])]
        for column, values in table["columns"].items():
            kind = kinds[column]
            for row, value in zip(rows, values):
                if value is None:
                    row[column] = None
                elif kind == "string":
                    row[column] = decoded[value]
                elif kind == "timestamp":
                    row[column] = epoch + dt.timedelta(milliseconds=value)
                elif kind == "double":
                    row[column] = float(value)
                else:
                    row[column] = value
        rows_by_table[name] = rows
    return rows_by_table


def rca_documents_saved(document_ids):
    """The documents whose rows are all in the history already: their rca_coverage rows, which write_rca_documents writes
    last, say so. A document that a failed write cut off before them is written again, whole. Empty when rca_coverage
    can't be read, so the evidence is written rather than missed."""
    session = _spark()
    if not document_ids or session is None or not session.catalog.tableExists("rca_coverage"):
        return set()
    try:
        # documentIds match RCA_DOCUMENT_ID (letters, digits, and dashes), so they're safe in the statement.
        rows = session.sql("SELECT DISTINCT document_id FROM rca_coverage WHERE document_id IN (%s)" % ", ".join(
            "'%s'" % document_id for document_id in sorted(document_ids))).collect()
        return {row["document_id"] for row in rows}
    except Exception as error:
        print("rca_coverage couldn't be read, so this run writes its RCA evidence again: " + redact(error)[:200])
        return set()


def write_rca_documents(rows_by_document):
    """Writes the documents' rows ({documentId: rows by table}) to the eight RCA tables, rca_coverage last: any rows of
    theirs from an earlier write that was cut off are deleted first (by document_id), so nothing is ever there twice.
    Any of the eight tables that's missing is created, so the history model frames them all. Lakehouse data only.
    Returns the rows written by table; an error is raised to the caller, which reports it."""
    session = _spark()
    # documentIds match RCA_DOCUMENT_ID (letters, digits, and dashes), so they're safe in the statement.
    ids = ", ".join("'%s'" % document_id for document_id in sorted(rows_by_document))
    written = {}
    for table in RCA_WRITE_ORDER:
        rows = [row for tables in rows_by_document.values() for row in tables.get(table, [])]
        if session.catalog.tableExists(table):
            session.sql("DELETE FROM %s WHERE document_id IN (%s)" % (table, ids))
        else:
            session.sql("CREATE TABLE IF NOT EXISTS %s (%s) USING DELTA" % (
                table, ", ".join("%s %s" % (name, SPARK_DDL_TYPES[kind]) for name, kind in TABLE_COLUMNS[table])))
        if rows:
            # One Parquet file per write, as the other tables (F16).
            frame = _rows_frame(session, table, rows)
            frame.coalesce(1).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table)
        written[table] = len(rows)
    return written


def clean_rca_inbox(handled=""):
    """Deletes the staged file this run handled, and any file in Files/rca/inbox staged more than RCA_INBOX_KEEP_HOURS ago,
    which a run that failed before its cleanup left; then the inbox and Files/rca once they're empty (a removal without
    recurse leaves a folder that still holds a file). Only files in the inbox are ever deleted (the handled one by
    RCA_EVIDENCE_FILE_PATTERN, old ones by RCA_INBOX_FILE), never a file that a wrong RCA_EVIDENCE_FILE names elsewhere.
    Returns the files it deleted. Lakehouse files only."""
    files = getattr(_notebookutils(), "fs", None)
    deleted = []
    if files is None or not WORKSPACE_ID or not LAKEHOUSE_ID:
        return deleted
    try:
        if handled and RCA_EVIDENCE_FILE_PATTERN.fullmatch(handled) and files.exists(_rca_path(handled)):
            files.rm(_rca_path(handled), False)
            deleted.append(handled)
        if not files.exists(_rca_path(RCA_INBOX)):
            return deleted
        cutoff = utc_now() - dt.timedelta(hours=RCA_INBOX_KEEP_HOURS)
        for entry in files.ls(_rca_path(RCA_INBOX)):
            name = "%s/%s" % (RCA_INBOX, str(getattr(entry, "name", "")).rstrip("/"))
            match = RCA_INBOX_FILE.fullmatch(name)
            if match and dt.datetime.strptime(match.group(1), "%Y%m%dT%H%M%SZ").replace(tzinfo=dt.timezone.utc) < cutoff:
                files.rm(_rca_path(name), False)
                deleted.append(name)
        for folder in (RCA_INBOX, "Files/rca"):
            if files.exists(_rca_path(folder)) and not list(files.ls(_rca_path(folder))):
                files.rm(_rca_path(folder), False)
    except Exception as error:
        print("Files/rca/inbox wasn't cleaned up completely (%s); the next run, or the deployment notebook's purge, finishes it." % redact(error)[:200])
    return deleted


def _rca_report(status, message, documents=(), rows=None, skipped=()):
    """Records this run's RCA.EVIDENCE once: printed, kept in RCA_STATE, and, when evidence was passed, saved as a finding
    (analysis rca_evidence) with the other findings."""
    RCA_STATE.update(done=True, status=status, message=redact(message), documents=list(documents), rows=dict(rows or {}),
                     skipped=["%s: %s" % (document_id or "parts", why) for document_id, why in skipped])
    if RCA_STATE["passed"]:
        add_finding("rca_evidence", "all", status, "RCA.EVIDENCE: " + RCA_STATE["message"], "evidence",
                    evidence={"documents": RCA_STATE["documents"], "rows": RCA_STATE["rows"], "skipped": RCA_STATE["skipped"]})
    else:
        print("RCA.EVIDENCE %s: %s" % (status, RCA_STATE["message"]))
    return RCA_STATE


def rca_exit_value():
    """RCA.EVIDENCE as one line, for a run's exit value, which the control notebook prints (notebookutils.notebook.run
    returns it)."""
    return ("RCA.EVIDENCE %s: %s" % (RCA_STATE["status"], RCA_STATE["message"]))[:1000]


def save_rca_evidence():
    """Writes this run's RCA evidence, once per run, and reports RCA.EVIDENCE (rca/README.md, RCA_EVIDENCE): a document
    is written when its sessionId is this run's SESSION_ID, or either is empty, and skipped for another session. Pass when
    every document was written, or was in the history already (each is written once: rca_documents_saved); Warning
    with why one wasn't; Not selected when none was passed, or SAVE_RESULTS is False outside RCA_ONLY. The staged file
    is deleted only once its evidence is in the history, written now or before, so a deleted file means written for the
    control notebook and the scripts (the review of 37a546bc); one that nothing was written from (unusable, unread,
    every document skipped, or a write that failed) stays, and clean_rca_inbox deletes it a day later at the latest."""
    if RCA_STATE["done"]:
        return RCA_STATE
    RCA_STATE["passed"] = bool(RCA_EVIDENCE or RCA_EVIDENCE_FILE)
    if not RCA_STATE["passed"]:
        clean_rca_inbox()
        return _rca_report("Not selected", "No RCA evidence was passed (RCA_EVIDENCE and RCA_EVIDENCE_FILE are empty), so no RCA rows were written.")
    parts, problems, keep = read_rca_evidence()
    if not parts:
        # Nothing to write from: the staged file stays, so its caller sees it left (the review of 37a546bc).
        clean_rca_inbox()
        return _rca_report("Warning", " ".join("The RCA evidence in %s wasn't used: %s." % item for item in problems))
    documents, skipped = decode_rca_parts(parts)
    # With both parameters, one that can't be used is skipped, and the other's documents are still written.
    skipped = [(where, "not used, because " + why) for where, why in problems] + skipped
    wanted, hosts, targets, starts, ends = {}, set(), set(), [], []
    for document_id, document in sorted(documents.items()):
        other = document["sessionId"]
        if SESSION_ID and other and other != SESSION_ID:
            skipped.append((document_id, "it's session %s's evidence, not this run's (%s)" % (other[:8], SESSION_ID[:8])))
            continue
        try:
            wanted[document_id] = rca_rows(document)
        except (OverflowError, ValueError, TypeError, IndexError, KeyError) as error:
            skipped.append((document_id, "its values couldn't be decoded (%s)" % type(error).__name__))
            continue
        hosts.add(document["producer"]["host"])
        targets.update(document["targets"])
        starts.append(_parse_utc(document["windowStartUtc"]))
        ends.append(_parse_utc(document["windowEndUtc"]))
    if not SAVE_RESULTS and not RCA_ONLY:
        return _rca_report("Not selected", "SAVE_RESULTS is False, so the RCA evidence (%d document(s)) wasn't written." % len(wanted), skipped=skipped)
    written, already = {}, sorted(rca_documents_saved(set(wanted)))
    for document_id in already:
        wanted.pop(document_id)
    if wanted:
        if _spark() is None or not runtime_context().get("defaultLakehouseId"):
            return _rca_report("Warning", "Spark or the default lakehouse isn't available, so the RCA evidence wasn't written. Attach %s and run again." % LAKEHOUSE_NAME,
                               skipped=skipped)
        try:
            written = write_rca_documents(wanted)
        except Exception as error:
            return _rca_report("Warning", "The RCA evidence couldn't be written (%s); the session's next save writes it again." % redact(error)[:300],
                               skipped=skipped)
    # A staged file goes only once its evidence is in the history: one whose every document was skipped stays.
    clean_rca_inbox("" if keep or not (wanted or already) else RCA_EVIDENCE_FILE)
    detection = sum(count for table, count in written.items() if table in RCA_DETECTION_TABLES)
    where = ("from %s (%s; %s to %s UTC)" % (", ".join(sorted(hosts)), ", ".join(item for item in RCA_TARGETS if item in targets),
                                              min(starts).strftime("%Y-%m-%d %H:%M"), max(ends).strftime("%H:%M"))) if starts else ""
    if wanted:
        what = "Wrote %d evidence document(s) %s: %d detection row(s)%s" % (
            len(wanted), where, detection, ("; %d more was in the history already" % len(already)) if already else "")
    elif already:
        what = "The %d evidence document(s) %s were in the history already: each is written once" % (len(already), where)
    else:
        what = "Wrote no RCA evidence"
    reasons = "; ".join("%s: %s" % (document_id or "parts", why) for document_id, why in skipped[:3]) + (
        "; and %d more" % (len(skipped) - 3) if len(skipped) > 3 else "")
    if not skipped:
        return _rca_report("Pass", what + ".", sorted(wanted) + already, written)
    return _rca_report("Warning", "%s, and skipped %d document(s) or line(s): %s." % (what, len(skipped), reasons),
                       sorted(wanted) + already, written, skipped)

print("RCA helpers ready: %d tables (rca/history-tables.json, version 1)." % len(RCA_TABLES))

### The Incident RCA's sidecar (2026-11-23)

Query Store's evidence on T1 to T4 (each plan's runtime statistics per 15-minute interval, its waits, the plans themselves, forcings, Query Store's state, and what the capture couldn't read) and who held T4's connections during a surge travel beside the evidence in documents of their own kind, **sidecars** (`rca/README.md`, The sidecar), so the evidence above, and everything that reads it, stays as it was. The scripts stage a save's sidecars in `Files/rca/inbox` and pass the file in `RCA_SIDECAR_FILE`. The next two cells handle it here, before section 3, which ends a run that saves only the evidence (`RCA_ONLY`) or a session already saved:

- **Decode and check:** `read_rca_sidecar` reads the file with `notebookutils.fs.head` (262,144 bytes); `decode_rca_sidecar_parts` takes only `DBHUBDEMO-RCAS` part lines, checks each sidecar's SHA-256 and its content against the contract (`check_rca_sidecar`: `rca/sidecar.schema.json` and `rca/history-tables.json`), and skips a sidecar with a missing or bad part, never writing it in part.
- **Write once, under its session:** the six tables `rca_query_stats`, `rca_query_waits`, `rca_query_plans`, `rca_query_events`, `rca_connections`, and `rca_query_coverage` last, which marks a sidecar written. A sidecar's rows go under its own session's run, `session-<id>`, whichever run reads it, so a session's evidence that a later save fetched lands with that session; a sidecar with no session goes under this run's `RUN_ID`.
- **The latest read wins:** Query Store's interval can be read more than once (open at a capture's end, then whole; or by two sessions' baselines), and a later read holds the same plan's counts or more. So `rca_query_stats` and `rca_query_waits` keep only the latest read of each target, interval, login, query, and plan: an earlier read in the history is replaced, deleted only once the later read is written (a write cut off between the two leaves both, which the history model's measures count once), and a read older than the history's isn't written.
- **Acknowledge:** once every sidecar in the file is committed, `Files/rca/acks/<the file's name without .json>.ack.json` says what this run did with each, by its ID: `written`, `already` (an earlier run wrote it), or `skipped` with why. The scripts read it, never the file's disappearance. A file this run can't read gets none, so its sidecars stay pending. The file is deleted only when every sidecar in it is written or already was.
- **Clean up:** a file the scripts staged (`sidecar-<time>[-<name>].json`, which part 4's day-old sweep never matches) that holds a skipped sidecar stays until later runs have written every sidecar in it, or for 30 days; an acknowledgement goes a day after its file does, or after 30 days, but never in the run that wrote it, and a `-runall-` one only after 30 days, since it waits for the next Run all start. The deployment notebook's purge removes `Files/rca/inbox` and `Files/rca/acks` with the tables.
- **`RCA.SIDECAR`** (analysis `rca_sidecar`): **Pass** when every sidecar was written or already was, **Warning** with why one wasn't, **Not selected** when none was passed, or `SAVE_RESULTS` is False outside `RCA_ONLY`, which writes, acknowledges, and deletes nothing. Without a sidecar, section 11 still creates the six tables, so the history model frames them.

No network call: the file is read, written, and deleted with `notebookutils.fs`.

In [ ]:
# Helpers, part 5: the Incident RCA's sidecar (2026-11-23, rca/README.md, The sidecar). Run it after part 4. It reads the
# file in this lakehouse's Files/rca/inbox that RCA_SIDECAR_FILE names, decodes its DBHUBDEMO-RCAS part lines into
# sidecars, checks each against the contract (rca/sidecar.schema.json and rca/history-tables.json, document sidecar),
# writes the sidecar's six tables under each sidecar's own session's run, with Query Store's latest read of each interval
# winning, acknowledges each sidecar by its ID in Files/rca/acks, and deletes the file only when every sidecar in it is in
# the history. Part 4's evidence is untouched: its decoder never takes a DBHUBDEMO-RCAS line, and its day-old sweep never
# matches a sidecar file's name. Lakehouse data only, and no network call: files are read, written, and deleted with
# notebookutils.fs (https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-file-system) by their
# OneLake paths (https://learn.microsoft.com/fabric/onelake/onelake-access-api).

# The sidecar's six tables, their columns in the contract's order, the required ones, and the primary time that
# report_date_utc comes from (rca/history-tables.json, document sidecar; tests/unit/Fabric.Notebooks.Tests.ps1 checks
# them against it). As the evidence's, each table also has the notebook's own columns: run_id and document_id first,
# report_date_utc last.
RCA_SIDECAR_TABLES = {
    "rca_query_stats": {
        "primary": "interval_start_utc",
        "columns": [
            ("captured_at_utc", "timestamp"), ("target", "string"), ("interval_start_utc", "timestamp"),
            ("interval_end_utc", "timestamp"), ("query_id", "long"), ("plan_id", "long"), ("login_name", "string"),
            ("query_hash", "string"), ("plan_hash", "string"), ("module_name", "string"), ("statement_text", "string"),
            ("executions", "long"), ("aborted_executions", "long"), ("cpu_ms", "double"), ("duration_ms", "double"),
            ("max_duration_ms", "double"), ("logical_reads", "double"), ("physical_reads", "double"),
            ("row_count", "double"), ("log_bytes", "double"), ("tempdb_kb", "double"), ("max_used_memory_kb", "double"),
            ("max_dop", "long"), ("wait_ms", "double"), ("first_execution_utc", "timestamp"),
            ("last_execution_utc", "timestamp"),
        ],
        "required": ("captured_at_utc", "target", "interval_start_utc", "interval_end_utc", "query_id", "plan_id", "executions"),
    },
    "rca_query_waits": {
        "primary": "interval_start_utc",
        "columns": [
            ("captured_at_utc", "timestamp"), ("target", "string"), ("interval_start_utc", "timestamp"),
            ("interval_end_utc", "timestamp"), ("query_id", "long"), ("plan_id", "long"), ("login_name", "string"),
            ("wait_category", "string"), ("wait_event", "string"), ("wait_ms", "double"), ("max_wait_ms", "double"),
            ("samples", "long"),
        ],
        "required": ("captured_at_utc", "target", "interval_start_utc", "interval_end_utc", "query_id", "plan_id",
                     "wait_category", "wait_ms"),
    },
    "rca_query_plans": {
        "primary": "captured_at_utc",
        "columns": [
            ("captured_at_utc", "timestamp"), ("target", "string"), ("query_id", "long"), ("plan_id", "long"),
            ("query_hash", "string"), ("plan_hash", "string"), ("module_name", "string"), ("statement_text", "string"),
            ("compiled_utc", "timestamp"), ("last_compiled_utc", "timestamp"), ("last_execution_utc", "timestamp"),
            ("compiles", "long"), ("is_forced_plan", "boolean"), ("forcing_type", "string"), ("force_failures", "long"),
            ("last_force_failure", "string"), ("plans_for_query", "integer"), ("estimated_cost", "double"),
            ("is_parallel", "boolean"), ("plan_summary", "string"),
        ],
        "required": ("captured_at_utc", "target", "query_id", "plan_id"),
    },
    "rca_query_events": {
        "primary": "event_time_utc",
        "columns": [
            ("event_time_utc", "timestamp"), ("time_from_utc", "timestamp"), ("target", "string"), ("kind", "string"),
            ("query_id", "long"), ("plan_id", "long"), ("forcing_type", "string"), ("state", "string"),
            ("previous_state", "string"), ("detail", "string"),
        ],
        "required": ("event_time_utc", "target", "kind"),
    },
    "rca_query_coverage": {
        "primary": "window_start_utc",
        "columns": [
            ("target", "string"), ("window_start_utc", "timestamp"), ("window_end_utc", "timestamp"), ("state", "string"),
            ("capture_mode", "string"), ("interval_minutes", "integer"), ("read_from_utc", "timestamp"),
            ("read_to_utc", "timestamp"), ("pending_from_utc", "timestamp"), ("pending_to_utc", "timestamp"),
            ("backfill", "boolean"), ("sources", "string"), ("gaps", "string"),
        ],
        "required": ("target", "window_start_utc", "window_end_utc", "backfill"),
    },
    "rca_connections": {
        "primary": "captured_at_utc",
        "columns": [
            ("captured_at_utc", "timestamp"), ("target", "string"), ("program_name", "string"), ("run_token", "string"),
            ("login_name", "string"), ("state", "string"), ("connections", "long"), ("waiting_lock", "long"),
            ("oldest_state_s", "double"), ("oldest_connection_s", "double"), ("is_demo", "boolean"),
        ],
        "required": ("captured_at_utc", "target", "state", "connections", "is_demo"),
    },
}
for _table, _spec in RCA_SIDECAR_TABLES.items():
    TABLE_COLUMNS[_table] = [("run_id", "string"), ("document_id", "string")] + _spec["columns"] + [("report_date_utc", "date")]
    REPORT_DATE_SOURCES[_table] = (_spec["primary"],)
# rca_query_coverage goes last: a sidecar's coverage rows say that all its rows are written (rca_sidecars_saved).
RCA_SIDECAR_WRITE_ORDER = tuple(name for name in RCA_SIDECAR_TABLES if name != "rca_query_coverage") + ("rca_query_coverage",)
# Query Store's latest read wins (rca/README.md, Query Store): a later read of an interval holds the same plan's counts or
# more, so these two tables keep only the rows of the latest read of each target, interval, login, query, and plan, once
# each by the contract's key (history-tables.json, key, less run_id).
RCA_LATEST_READ_TABLES = ("rca_query_stats", "rca_query_waits")
RCA_READ_KEY = ("target", "interval_start_utc", "login_name", "query_id", "plan_id")
RCA_ROW_KEY = {"rca_query_stats": RCA_READ_KEY + ("captured_at_utc",),
               "rca_query_waits": RCA_READ_KEY + ("captured_at_utc", "wait_category", "wait_event")}
RCA_SIDECAR_ID = re.compile(r"[0-9]{8}T[0-9]{6}Z-[a-z0-9]+(?:-[a-z0-9]+)*-[0-9]{4}-sidecar")
RCA_SIDECAR_PART_LINE = re.compile(r"DBHUBDEMO-RCAS 1 (%s) ([1-9][0-9]?)/([1-9][0-9]?) ([0-9a-f]{64}) ([A-Za-z0-9+/]+={0,2})"
                                   % RCA_SIDECAR_ID.pattern)
# A DBHUBDEMO-RCAS line that names a sidecar in any part format: what the acknowledgement and the sweep can say about a
# line this notebook can't decode.
RCA_SIDECAR_NAMED = re.compile(r"DBHUBDEMO-RCAS \S+ (%s)(?: .*)?" % RCA_SIDECAR_ID.pattern)
# RCA_SIDECAR_FILE: a file in the inbox named as rca/transport.schema.json's sidecarFile pins it,
# sidecar-<when it was staged, UTC>[-<name>].json, which part 4's day-old sweep never matches: a file that holds a skipped
# sidecar stays until later runs have written every sidecar in it, or for RCA_SIDECAR_KEEP_DAYS, and this cell's sweep
# finds it by that name (RCA_SIDECAR_INBOX_FILE). The pattern is the contract's, whose (?![\s\S]) ends the input, as
# fullmatch does.
RCA_SIDECAR_FILE_PATTERN = re.compile(r"Files/rca/inbox/sidecar-[0-9]{8}T[0-9]{6}Z(-[A-Za-z0-9][A-Za-z0-9._-]{0,63})?\.json(?![\s\S])")
RCA_SIDECAR_INBOX_FILE = re.compile(r"Files/rca/inbox/(sidecar-([0-9]{8}T[0-9]{6}Z)(?:-[A-Za-z0-9][A-Za-z0-9._-]{0,63})?)\.json")
# A file's acknowledgement: Files/rca/acks/<the file's name without .json>.ack.json (rca/transport.schema.json, ack).
RCA_ACKS = "Files/rca/acks"
RCA_ACK_FILE = re.compile(r"Files/rca/acks/(sidecar-([0-9]{8}T[0-9]{6}Z)(?:-[A-Za-z0-9][A-Za-z0-9._-]{0,63})?)\.ack\.json")
RCA_SIDECAR_KEEP_DAYS = 30
# Run all's files (sidecar-<UTC>-runall-<8 hex>.json): their acknowledgement waits for the next Run all start to send its
# IDs, however long that takes, and the control notebook deletes it then, so this cell's sweep deletes one only
# RCA_SIDECAR_KEEP_DAYS after its file was staged (rca/README.md, Acknowledgement).
RCA_RUNALL_PAYLOAD = re.compile(r"sidecar-[0-9]{8}T[0-9]{6}Z-runall-[0-9a-f]{8}")
# How many kept files one run's sweep reads, oldest first, to see whether later runs have written every sidecar in them.
RCA_SIDECAR_SWEEP_READS = 16
# A sidecar's header times as the contract's utc says (rca/sidecar.schema.json): UTC with exactly three fractional digits,
# so its rows stay on the millisecond grid that _rca_keep_latest compares. Part 4's RCA_UTC, as released, takes more.
RCA_SIDECAR_UTC = re.compile(r"[0-9]{4}-[0-9]{2}-[0-9]{2}T[0-9]{2}:[0-9]{2}:[0-9]{2}\.[0-9]{3}Z")
_RCA_SIDECAR_MEMBERS = ("kind", "version", "documentId", "sessionId", "producer", "exportedUtc", "sequence", "final",
                        "windowStartUtc", "windowEndUtc", "targets", "epochUtc", "strings", "tables")
_RCA_UNIX = dt.datetime(1970, 1, 1, tzinfo=dt.timezone.utc)
# The sidecar arrives as text, empty when none was passed: this cell checks it, so a file it can't use never stops the
# analysis.
RCA_SIDECAR_FILE = str(RCA_SIDECAR_FILE or "").strip()
# This run's RCA.SIDECAR: saved once per run (save_rca_sidecar).
RCA_SIDECAR_STATE = {"done": False, "passed": False, "status": "Not selected", "message": "", "documents": [], "rows": {},
                     "skipped": [], "replaced": 0, "superseded": 0, "ack": ""}


def _rca_ms(value):
    """A UTC time as integer milliseconds after 1970-01-01, as Spark's unix_millis returns a stored one."""
    return (value - _RCA_UNIX) // dt.timedelta(milliseconds=1)


def rca_sidecar_parameter_problem(value):
    """Why value isn't a version 1 ContosoCare.RcaSidecarParts document (rca/transport.schema.json, sidecarParameter), or
    "". A part line it can't read is left to decode_rca_sidecar_parts, which skips its sidecar and says so."""
    if not isinstance(value, dict) or value.get("kind") != "ContosoCare.RcaSidecarParts":
        return "it isn't a ContosoCare.RcaSidecarParts document"
    if value.get("version") != 1:
        return "its version %s is newer than this notebook's 1 (./scripts/deploy.ps1 updates the notebook)" % str(value.get("version"))[:20]
    extra = sorted(str(name) for name in set(value) - {"kind", "version", "sessionId", "parts", "status", "message"})
    if extra:
        return "it has members that version 1 doesn't have (%s)" % ", ".join(extra)[:200]
    session = value.get("sessionId", "")
    if session is not None and not (isinstance(session, str) and RCA_SESSION.fullmatch(session)):
        return "its sessionId isn't a session ID or null"
    parts = value.get("parts")
    if not isinstance(parts, list) or not 1 <= len(parts) <= RCA_LIMITS["parts"]:
        return "it must hold 1 to %d part lines" % RCA_LIMITS["parts"]
    if any(not isinstance(part, str) or len(part) > RCA_LIMITS["partChars"] for part in parts):
        return "a part line isn't text of at most %d characters" % RCA_LIMITS["partChars"]
    if "status" in value and value["status"] not in STATUSES:
        return "its status isn't one of ADR 0012's"
    if "message" in value and not (isinstance(value["message"], str) and len(value["message"]) <= RCA_LIMITS["message"]):
        return "its message isn't text of at most %d characters" % RCA_LIMITS["message"]
    return ""


def _rca_sidecar_named(lines):
    """The sidecars that part lines name, in any part format, each once, and how many lines name none."""
    named, unnamed = [], 0
    for line in lines:
        match = RCA_SIDECAR_NAMED.fullmatch(line) if isinstance(line, str) else None
        if match:
            named.append(match.group(1))
        else:
            unnamed += 1
    return list(dict.fromkeys(named)), unnamed


def read_rca_sidecar():
    """The file RCA_SIDECAR_FILE names, checked whole: (its part lines, [] when it can't be used; why it can't be used, or
    ""; the sidecars its lines name, which the acknowledgement of a file it can't use lists as skipped; whether it was
    read). Nothing here stops the analysis."""
    utils = _notebookutils()
    if not RCA_SIDECAR_FILE_PATTERN.fullmatch(RCA_SIDECAR_FILE):
        return [], "RCA_SIDECAR_FILE must name a file %s/sidecar-<time>[-<name>].json, as the scripts stage it" % RCA_INBOX, [], False
    if utils is None or not WORKSPACE_ID or not LAKEHOUSE_ID:
        return [], "notebookutils or the lakehouse's IDs aren't available, so the staged file wasn't read", [], False
    try:
        text = utils.fs.head(_rca_path(RCA_SIDECAR_FILE), RCA_LIMITS["fileBytes"])
    except Exception as error:
        return [], "the staged file couldn't be read (%s)" % redact(error)[:300], [], False
    if not text:
        return [], "the staged file is empty", [], True
    try:
        value = _rca_json(text)
    except (ValueError, RecursionError):
        return [], "it isn't JSON, or it's larger than %d bytes" % RCA_LIMITS["fileBytes"], [], True
    lines = value.get("parts") if isinstance(value, dict) and isinstance(value.get("parts"), list) else []
    problem = rca_sidecar_parameter_problem(value)
    return ([] if problem else list(dict.fromkeys(value["parts"]))), problem, _rca_sidecar_named(lines)[0], True


def _rca_sidecar_table_problem(name, table, strings):
    """Why table isn't the version 1 sidecar table name with strings as its dictionary, as part 4's _rca_table_problem
    checks an evidence table (the sidecar keeps the evidence document's encoding: rca/README.md, The sidecar), or ""."""
    spec = RCA_SIDECAR_TABLES[name]
    if not isinstance(table, dict) or set(table) != {"rows", "columns"}:
        return "%s must hold rows and columns only" % name
    rows, columns = table["rows"], table["columns"]
    if not _rca_int(rows, (1, RCA_LIMITS["rows"])) or not isinstance(columns, dict):
        return "%s's rows must be 1 to %d, with its columns" % (name, RCA_LIMITS["rows"])
    kinds = dict(spec["columns"])
    unknown = sorted(str(column) for column in set(columns) - set(kinds))
    if unknown:
        return "%s has columns that version 1 doesn't have (%s)" % (name, ", ".join(unknown)[:200])
    for column in spec["required"]:
        if column not in columns:
            return "%s lacks its required column %s" % (name, column)
    for column, values in columns.items():
        if not isinstance(values, list) or len(values) != rows:
            return "%s.%s doesn't hold %d values" % (name, column, rows)
        kind, required = kinds[column], column in spec["required"]
        for value in values:
            if value is None:
                if required:
                    return "%s.%s is required, but holds a null" % (name, column)
                continue
            if kind == "string":
                good = _rca_int(value, (0, len(strings) - 1))
            elif kind in ("timestamp", "long"):
                good = _rca_int(value)
            elif kind == "integer":
                good = _rca_int(value, _RCA_INT32)
            elif kind == "double":
                good = isinstance(value, (int, float)) and not isinstance(value, bool)
            else:
                good = isinstance(value, bool)
            if not good:
                return "%s.%s holds a value that isn't a %s%s" % (name, column, kind, " index" if kind == "string" else "")
    return ""


def check_rca_sidecar(document):
    """Why document isn't a version 1 sidecar this notebook can write (rca/sidecar.schema.json, and the rules rca/README.md
    adds), or "". A sidecar is checked whole before any of its rows is written."""
    if not isinstance(document, dict) or document.get("kind") != "ContosoCare.RcaSidecar":
        return "it isn't a ContosoCare.RcaSidecar document"
    if document.get("version") != 1:
        return "its version %s is newer than this notebook's 1 (./scripts/deploy.ps1 updates the notebook)" % str(document.get("version"))[:20]
    missing = [name for name in _RCA_SIDECAR_MEMBERS if name not in document]
    extra = sorted(str(name) for name in set(document) - set(_RCA_SIDECAR_MEMBERS) - {"$schema"})
    if missing or extra:
        return "it lacks %s or has members that version 1 doesn't have (%s)" % (", ".join(missing) or "nothing", ", ".join(extra)[:200] or "none")
    if not isinstance(document["documentId"], str) or len(document["documentId"]) > 80 or not RCA_SIDECAR_ID.fullmatch(document["documentId"]):
        return "its documentId isn't a sidecar's"
    session = document["sessionId"]
    if session is not None and not (isinstance(session, str) and RCA_SESSION.fullmatch(session)):
        return "its sessionId isn't a session ID or null"
    producer = document["producer"]
    if (not isinstance(producer, dict) or set(producer) != {"name", "version", "host", "lane"} or producer["name"] != "ContosoCare.Workload"
            or producer["lane"] != "observer" or not isinstance(producer["version"], str) or not 1 <= len(producer["version"]) <= 64
            or not isinstance(producer["host"], str) or not RCA_HOST.fullmatch(producer["host"])):
        return "its producer isn't the workload's observer"
    for name in ("exportedUtc", "windowStartUtc", "windowEndUtc", "epochUtc"):
        if not isinstance(document[name], str) or not RCA_SIDECAR_UTC.fullmatch(document[name]):
            return "its %s isn't a UTC time with milliseconds" % name
        try:
            _parse_utc(document[name])
        except (ValueError, OverflowError):
            # A date that doesn't exist, or on pandas 1.x one past 2262 (OutOfBoundsDatetime): refused here, never a
            # failed run.
            return "its %s isn't a time this notebook can read" % name
    if not _rca_int(document["sequence"], (1, 9999)) or not isinstance(document["final"], bool):
        return "its sequence or final isn't one"
    targets = document["targets"]
    if (not isinstance(targets, list) or not targets or any(not isinstance(target, str) or target not in RCA_TARGETS for target in targets)
            or len(set(targets)) != len(targets)):
        return "its targets must be T1 to T5 or T4-replica, each once"
    problem = _rca_strings_problem(document["strings"], "the sidecar")
    if problem:
        return problem
    tables = document["tables"]
    if not isinstance(tables, dict) or "rca_query_coverage" not in tables:
        return "its tables must include rca_query_coverage"
    for name, table in tables.items():
        if name not in RCA_SIDECAR_TABLES:
            return "its tables hold %s, which version 1's sidecar tables don't have" % str(name)[:60]
        problem = _rca_sidecar_table_problem(name, table, document["strings"])
        if problem:
            return problem
    return ""


def decode_rca_sidecar_parts(parts):
    """Groups the DBHUBDEMO-RCAS part lines by documentId and decodes each sidecar as part 4's decode_rca_parts decodes an
    evidence document (rca/README.md, Part lines), checking it with check_rca_sidecar. Returns ({documentId: sidecar},
    [(documentId, why it was skipped)]): a sidecar with a missing, bad, or conflicting part is skipped, never written in
    part, and so is one that a line this notebook can't read names; ("", why) counts the lines that name no sidecar."""
    groups, skipped, unread = {}, [], {}
    for line in parts:
        match = RCA_SIDECAR_PART_LINE.fullmatch(line)
        if not match:
            named = RCA_SIDECAR_NAMED.fullmatch(line)
            name = named.group(1) if named else ""
            unread[name] = unread.get(name, 0) + 1
            continue
        document_id, index, count, digest, data = match.group(1), int(match.group(2)), int(match.group(3)), match.group(4), match.group(5)
        group = groups.setdefault(document_id, {"count": count, "sha256": digest, "parts": {}, "conflict": False})
        if group["count"] != count or group["sha256"] != digest or index > count or group["parts"].get(index, data) != data:
            group["conflict"] = True
        group["parts"][index] = data
    if unread.get(""):
        skipped.append(("", "%d line(s) that aren't a sidecar's part lines" % unread[""]))
    for document_id in sorted(name for name in unread if name and name not in groups):
        skipped.append((document_id, "its part lines are in a part format this notebook doesn't read (a newer runner? ./scripts/deploy.ps1 updates the notebook)"))
    documents = {}
    for document_id, group in sorted(groups.items()):
        missing = [str(index) for index in range(1, group["count"] + 1) if index not in group["parts"]]
        if group["conflict"]:
            why = "its parts disagree on their count, their SHA-256, or a part's content"
        elif document_id in unread:
            why = "%d of its part line(s) can't be read" % unread[document_id]
        elif missing:
            why = "part(s) %s of %d are missing" % (", ".join(missing[:5]), group["count"])
        else:
            why = ""
            try:
                stream = b"".join(base64.b64decode(group["parts"][index], validate=True) for index in range(1, group["count"] + 1))
            except ValueError:
                stream, why = b"", "a part isn't base64"
            if not why and hashlib.sha256(stream).hexdigest() != group["sha256"]:
                why = "its SHA-256 doesn't match its parts"
            if not why:
                text, why = _rca_gunzip(stream)
            if not why:
                try:
                    document = _rca_json(text)
                except (ValueError, RecursionError):
                    document, why = None, "it isn't JSON"
            if not why:
                why = check_rca_sidecar(document)
            if not why and document["documentId"] != document_id:
                why = "its documentId isn't the one its parts name"
        if why:
            skipped.append((document_id, why))
        else:
            documents[document_id] = document
    return documents, skipped


def rca_sidecar_run_id(document):
    """The run a sidecar's rows go under (rca/README.md, The sidecar): its own session's, session-<sessionId>, whichever
    run reads it, so a session's Query Store evidence that a later save fetched lands with that session; this run's
    RUN_ID for a sidecar with no session."""
    return "session-" + document["sessionId"] if document["sessionId"] else RUN_ID


def rca_sidecar_rows(document):
    """The sidecar's rows by table, decoded as part 4's rca_rows decodes an evidence document's: strings from its
    dictionary, through redact(); times from epochUtc; doubles as floats; its run_id (rca_sidecar_run_id) and
    document_id added (report_date_utc comes with the write)."""
    epoch = pd.Timestamp(document["epochUtc"]).tz_convert("UTC").to_pydatetime(warn=False)
    decoded = [redact(text) for text in document["strings"]]
    run_id = rca_sidecar_run_id(document)
    rows_by_table = {}
    for name, table in document["tables"].items():
        kinds = dict(RCA_SIDECAR_TABLES[name]["columns"])
        rows = [{"run_id": run_id, "document_id": document["documentId"]} for _ in range(table["rows"])]
        for column, values in table["columns"].items():
            kind = kinds[column]
            for row, value in zip(rows, values):
                if value is None:
                    row[column] = None
                elif kind == "string":
                    row[column] = decoded[value]
                elif kind == "timestamp":
                    row[column] = epoch + dt.timedelta(milliseconds=value)
                elif kind == "double":
                    row[column] = float(value)
                else:
                    row[column] = value
        rows_by_table[name] = rows
    return rows_by_table


def _rca_read_key(row):
    return tuple(row.get(column) for column in RCA_READ_KEY)


def latest_rca_reads(rows_by_document):
    """Query Store's latest read wins among the sidecars passed (rca/README.md, Query Store): in RCA_LATEST_READ_TABLES,
    only the rows of each target, interval, login, query, and plan's latest read stay, each once by RCA_ROW_KEY. Returns
    how many earlier reads it left out."""
    left = set()
    for table in RCA_LATEST_READ_TABLES:
        latest = {}
        for tables in rows_by_document.values():
            for row in tables.get(table, []):
                key = _rca_read_key(row)
                if key not in latest or row["captured_at_utc"] > latest[key]:
                    latest[key] = row["captured_at_utc"]
        seen = set()
        for tables in rows_by_document.values():
            if table in tables:
                kept = []
                for row in tables[table]:
                    key = tuple(row.get(column) for column in RCA_ROW_KEY[table])
                    if row["captured_at_utc"] != latest[_rca_read_key(row)]:
                        left.add((table, _rca_read_key(row), row["captured_at_utc"]))
                    elif key not in seen:
                        seen.add(key)
                        kept.append(row)
                tables[table] = kept
    return len(left)


def _rca_keep_latest(session, table, rows):
    """Query Store's latest read wins against the history (rca/README.md, Query Store), in one of RCA_LATEST_READ_TABLES:
    of each target, interval, login, query, and plan in rows (one read each: latest_rca_reads), the rows here aren't
    written when the history holds as late a read or a later one, and a read that's later than the history's replaces it
    once it's written (_rca_drop_earlier). Returns (the rows to write, one row of each read that replaces an earlier one,
    the reads not written)."""
    reads = {}
    for row in rows:
        reads.setdefault(_rca_read_key(row), row)
    starts = [_rca_ms(key[1]) for key in reads]
    history = {}
    # Only integers in the statement: the intervals' bounds in UTC milliseconds (unix_millis), whatever the session's time
    # zone.
    for found in session.sql("SELECT target, unix_millis(interval_start_utc) AS interval_ms, login_name, query_id, plan_id, "
                             "unix_millis(captured_at_utc) AS captured_ms FROM %s WHERE unix_millis(interval_start_utc) "
                             "BETWEEN %d AND %d" % (table, min(starts), max(starts))).collect():
        key = (found["target"], found["interval_ms"], found["login_name"], found["query_id"], found["plan_id"])
        history[key] = max(history.get(key, found["captured_ms"]), found["captured_ms"])
    keep, replacing = set(), []
    for key, row in reads.items():
        stored = history.get((key[0], _rca_ms(key[1])) + key[2:])
        if stored is None or stored < _rca_ms(row["captured_at_utc"]):
            keep.add(key)
            if stored is not None:
                replacing.append(row)
    return [row for row in rows if _rca_read_key(row) in keep], replacing, len(reads) - len(keep)


def _rca_drop_earlier(session, table, later):
    """Deletes the history's earlier reads of the reads in later (one row each, from _rca_keep_latest) with MERGE ... WHEN
    MATCHED THEN DELETE, once the later reads are written: a write cut off between the two commits leaves both reads,
    which the history model's measures count once, never neither."""
    if later:
        _rows_frame(session, table, later).createOrReplaceTempView("rca_sidecar_reads")
        session.sql("MERGE INTO %s AS t USING rca_sidecar_reads AS s ON t.target = s.target AND t.interval_start_utc = "
                    "s.interval_start_utc AND t.login_name <=> s.login_name AND t.query_id = s.query_id AND t.plan_id = "
                    "s.plan_id AND t.captured_at_utc < s.captured_at_utc WHEN MATCHED THEN DELETE" % table)


def rca_sidecars_saved(document_ids):
    """The sidecars whose rows are all in the history already: their rca_query_coverage rows, which write_rca_sidecars
    writes last, say so. A sidecar that a failed write cut off before them is written again, whole. Empty when
    rca_query_coverage can't be read, so the sidecar is written rather than missed."""
    session = _spark()
    if not document_ids or session is None or not session.catalog.tableExists("rca_query_coverage"):
        return set()
    try:
        # Sidecar IDs match RCA_SIDECAR_ID (letters, digits, and dashes), so they're safe in the statement.
        rows = session.sql("SELECT DISTINCT document_id FROM rca_query_coverage WHERE document_id IN (%s)" % ", ".join(
            "'%s'" % document_id for document_id in sorted(document_ids))).collect()
        return {row["document_id"] for row in rows}
    except Exception as error:
        print("rca_query_coverage couldn't be read, so this run writes its sidecars again: " + redact(error)[:200])
        return set()


def write_rca_sidecars(rows_by_document):
    """Writes the sidecars' rows ({documentId: rows by table}) to the six tables, rca_query_coverage last: any rows of
    theirs from an earlier write that was cut off are deleted first (by document_id), so a sidecar's rows are there once,
    and in rca_query_stats and rca_query_waits the latest read of each interval wins over the history's
    (_rca_keep_latest): an earlier read is deleted only after the later one is written (_rca_drop_earlier). Any of the six
    that's missing is created, so the history model frames them all. Lakehouse data only. Returns (the rows written by
    table, the earlier reads deleted, the reads not written); an error is raised to the caller, which reports it."""
    session = _spark()
    # Sidecar IDs match RCA_SIDECAR_ID (letters, digits, and dashes), so they're safe in the statement.
    ids = ", ".join("'%s'" % document_id for document_id in sorted(rows_by_document))
    written, replaced, superseded = {}, 0, 0
    for table in RCA_SIDECAR_WRITE_ORDER:
        rows = [row for tables in rows_by_document.values() for row in tables.get(table, [])]
        if session.catalog.tableExists(table):
            session.sql("DELETE FROM %s WHERE document_id IN (%s)" % (table, ids))
        else:
            session.sql("CREATE TABLE IF NOT EXISTS %s (%s) USING DELTA" % (
                table, ", ".join("%s %s" % (name, SPARK_DDL_TYPES[kind]) for name, kind in TABLE_COLUMNS[table])))
        later = []
        if rows and table in RCA_LATEST_READ_TABLES:
            rows, later, older = _rca_keep_latest(session, table, rows)
            replaced, superseded = replaced + len(later), superseded + older
        if rows:
            # One Parquet file per write, as the other tables (F16).
            frame = _rows_frame(session, table, rows)
            frame.coalesce(1).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(table)
        _rca_drop_earlier(session, table, later)
        written[table] = len(rows)
    return written, replaced, superseded


def rca_ack_path(payload):
    """A staged file's acknowledgement: Files/rca/acks/<the file's name without .json>.ack.json."""
    return "%s/%s.ack.json" % (RCA_ACKS, payload.rsplit("/", 1)[-1][:-len(".json")])


def write_rca_ack(payload, documents):
    """Writes the acknowledgement of one staged file (rca/transport.schema.json, ack): what this run did with each sidecar
    in it, by its ID. The scripts read it, never the file's disappearance. Returns (its path, "") or ("", why it
    couldn't be written)."""
    utils, path = _notebookutils(), rca_ack_path(payload)
    ack = {"kind": "ContosoCare.RcaAck", "version": 1, "payload": payload, "runId": RUN_ID[:200],
           "documents": documents[:RCA_LIMITS["parts"]]}
    try:
        utils.fs.mkdirs(_rca_path(RCA_ACKS))
        utils.fs.put(_rca_path(path), json.dumps(ack, separators=(",", ":")), True)
    except Exception as error:
        return "", "its acknowledgement couldn't be written (%s)" % redact(error)[:200]
    return path, ""


def _rca_sidecars_all_saved(files, name):
    """Whether every sidecar that a kept file's lines name is in the history now, so the file can go: False when it can't
    be read, or a line in it names no sidecar."""
    try:
        value = _rca_json(files.head(_rca_path(name), RCA_LIMITS["fileBytes"]))
    except Exception:
        return False
    lines = value.get("parts") if isinstance(value, dict) else None
    if not isinstance(lines, list) or not lines:
        return False
    named, unnamed = _rca_sidecar_named(lines)
    return bool(named) and not unnamed and set(named) <= rca_sidecars_saved(set(named))


def clean_rca_sidecars(handled="", keep_handled=True):
    """Deletes the staged file this run handled unless it holds a sidecar that isn't in the history (keep_handled), then
    sweeps what earlier runs kept: a file named as the scripts stage one (RCA_SIDECAR_INBOX_FILE) once later runs have
    written every sidecar in it, or RCA_SIDECAR_KEEP_DAYS after it was staged; and an acknowledgement (RCA_ACK_FILE) a day
    after its file was staged once that file is gone, or RCA_SIDECAR_KEEP_DAYS after, but never the handled file's, which
    the scripts read next however long ago they staged the file, and a Run all file's (RCA_RUNALL_PAYLOAD) only
    RCA_SIDECAR_KEEP_DAYS after, since it waits for the next Run all start, and the control notebook deletes it before
    then (rca/README.md, Acknowledgement); then Files/rca/acks, Files/rca/inbox, and Files/rca once they're empty (a
    removal without recurse leaves a folder that still holds a file). Only the handled file (RCA_SIDECAR_FILE_PATTERN)
    and files with those names are ever deleted, and never one whose name's time doesn't exist (a 30th of February),
    which the sweep goes past and leaves for the deployment notebook's purge. Returns the files it deleted. Lakehouse
    files only."""
    files = getattr(_notebookutils(), "fs", None)
    deleted = []
    if files is None or not WORKSPACE_ID or not LAKEHOUSE_ID:
        return deleted
    now = utc_now()

    def staged(stamp):
        try:
            return dt.datetime.strptime(stamp, "%Y%m%dT%H%M%SZ").replace(tzinfo=dt.timezone.utc)
        except ValueError:
            return None

    try:
        if handled and not keep_handled and RCA_SIDECAR_FILE_PATTERN.fullmatch(handled) and files.exists(_rca_path(handled)):
            files.rm(_rca_path(handled), False)
            deleted.append(handled)
        if files.exists(_rca_path(RCA_INBOX)):
            kept = []
            for entry in files.ls(_rca_path(RCA_INBOX)):
                name = "%s/%s" % (RCA_INBOX, str(getattr(entry, "name", "")).rstrip("/"))
                match = RCA_SIDECAR_INBOX_FILE.fullmatch(name)
                when = staged(match.group(2)) if match else None
                if when is None or name == handled:
                    continue
                if when < now - dt.timedelta(days=RCA_SIDECAR_KEEP_DAYS):
                    files.rm(_rca_path(name), False)
                    deleted.append(name)
                else:
                    kept.append((match.group(2), name))
            for _, name in sorted(kept)[:RCA_SIDECAR_SWEEP_READS]:
                if _rca_sidecars_all_saved(files, name):
                    files.rm(_rca_path(name), False)
                    deleted.append(name)
        if files.exists(_rca_path(RCA_ACKS)):
            own = rca_ack_path(handled) if handled else ""
            for entry in files.ls(_rca_path(RCA_ACKS)):
                name = "%s/%s" % (RCA_ACKS, str(getattr(entry, "name", "")).rstrip("/"))
                match = RCA_ACK_FILE.fullmatch(name)
                when = staged(match.group(2)) if match else None
                if when is None or name == own:
                    continue
                age = now - when
                # Run all's waits for the next Run all start, however long that takes: only its age takes it here.
                waiting = RCA_RUNALL_PAYLOAD.fullmatch(match.group(1)) is not None
                gone = not waiting and not files.exists(_rca_path("%s/%s.json" % (RCA_INBOX, match.group(1))))
                if age > dt.timedelta(days=RCA_SIDECAR_KEEP_DAYS) or (gone and age > dt.timedelta(hours=RCA_INBOX_KEEP_HOURS)):
                    files.rm(_rca_path(name), False)
                    deleted.append(name)
        for folder in (RCA_ACKS, RCA_INBOX, "Files/rca"):
            if files.exists(_rca_path(folder)) and not list(files.ls(_rca_path(folder))):
                files.rm(_rca_path(folder), False)
    except Exception as error:
        print("Files/rca wasn't cleaned up completely (%s); a later run, or the deployment notebook's purge, finishes it." % redact(error)[:200])
    return deleted


def _rca_sidecar_report(status, message, documents=(), rows=None, skipped=(), replaced=0, superseded=0, ack=""):
    """Records this run's RCA.SIDECAR once: printed, kept in RCA_SIDECAR_STATE, and, when a sidecar was passed, saved as a
    finding (analysis rca_sidecar) with the other findings."""
    RCA_SIDECAR_STATE.update(done=True, status=status, message=redact(message), documents=list(documents), rows=dict(rows or {}),
                             skipped=["%s: %s" % (document_id or "lines", why) for document_id, why in skipped],
                             replaced=replaced, superseded=superseded, ack=ack)
    if RCA_SIDECAR_STATE["passed"]:
        add_finding("rca_sidecar", "all", status, "RCA.SIDECAR: " + RCA_SIDECAR_STATE["message"], "sidecar",
                    evidence={key: RCA_SIDECAR_STATE[key] for key in ("documents", "rows", "skipped", "replaced", "superseded", "ack")})
    else:
        print("RCA.SIDECAR %s: %s" % (status, RCA_SIDECAR_STATE["message"]))
    return RCA_SIDECAR_STATE


def save_rca_sidecar():
    """Writes this run's sidecars, once per run, acknowledges them, and reports RCA.SIDECAR (rca/README.md,
    RCA_SIDECAR_FILE). Each sidecar is written once, by its documentId, under its own session's run (rca_sidecar_run_id).
    Pass when every sidecar was written, or was in the history already (rca_sidecars_saved), and acknowledged; Warning
    with why one wasn't; Not selected when none was passed, or SAVE_RESULTS is False outside RCA_ONLY, which writes,
    acknowledges, and deletes nothing. The file is deleted only when every sidecar in it is in the history and its
    acknowledgement is written (rca/README.md, Acknowledgement); one with a skipped sidecar stays, and clean_rca_sidecars
    deletes it once later runs have written them all, or RCA_SIDECAR_KEEP_DAYS later."""
    if RCA_SIDECAR_STATE["done"]:
        return RCA_SIDECAR_STATE
    RCA_SIDECAR_STATE["passed"] = bool(RCA_SIDECAR_FILE)
    saving = SAVE_RESULTS or RCA_ONLY
    if not RCA_SIDECAR_FILE:
        if saving:
            clean_rca_sidecars()
        return _rca_sidecar_report("Not selected", "No sidecar was passed (RCA_SIDECAR_FILE is empty), so no sidecar rows were written.")
    parts, problem, named, read = read_rca_sidecar()
    documents, skipped = decode_rca_sidecar_parts(parts)
    if not saving:
        return _rca_sidecar_report("Not selected", "SAVE_RESULTS is False, so the %d sidecar(s) in %s weren't written or acknowledged, and the file stays."
                                   % (len(documents), RCA_SIDECAR_FILE), skipped=skipped)
    if problem:
        skipped = [(document_id, "not used, because " + problem) for document_id in named]
    wanted, hosts, targets, starts, ends, runs = {}, set(), set(), [], [], set()
    for document_id, document in sorted(documents.items()):
        try:
            wanted[document_id] = rca_sidecar_rows(document)
        except (OverflowError, ValueError, TypeError, IndexError, KeyError) as error:
            skipped.append((document_id, "its values couldn't be decoded (%s)" % type(error).__name__))
            continue
        hosts.add(document["producer"]["host"])
        targets.update(document["targets"])
        starts.append(_parse_utc(document["windowStartUtc"]))
        ends.append(_parse_utc(document["windowEndUtc"]))
        runs.add(rca_sidecar_run_id(document))
    already = sorted(rca_sidecars_saved(set(wanted)))
    for document_id in already:
        wanted.pop(document_id)
    written, replaced, superseded, left = {}, 0, 0, 0
    if wanted:
        failure = ""
        if _spark() is None or not runtime_context().get("defaultLakehouseId"):
            failure = "Spark or the default lakehouse isn't available, so it wasn't written (attach %s and run again)" % LAKEHOUSE_NAME
        else:
            left = latest_rca_reads(wanted)
            try:
                written, replaced, superseded = write_rca_sidecars(wanted)
            except Exception as error:
                failure = "the write failed (%s); the next save writes it again" % redact(error)[:300]
        if failure:
            skipped += [(document_id, failure) for document_id in sorted(wanted)]
            wanted, written, replaced, superseded, left = {}, {}, 0, 0, 0
    ack, unacknowledged = "", ""
    entries = [{"documentId": document_id, "kind": "ContosoCare.RcaSidecar", "status": "written"} for document_id in sorted(wanted)]
    entries += [{"documentId": document_id, "kind": "ContosoCare.RcaSidecar", "status": "already"} for document_id in already]
    entries += [{"documentId": document_id, "kind": "ContosoCare.RcaSidecar", "status": "skipped", "reason": redact(reason)[:500]}
                for document_id, reason in skipped if document_id]
    if read and entries:
        # Once every sidecar in the file is committed, and listing exactly them; none for a file that names no sidecar,
        # which is one this run couldn't read (rca/README.md, Acknowledgement).
        ack, unacknowledged = write_rca_ack(RCA_SIDECAR_FILE, entries)
    # The file goes only when every sidecar in it is in the history, and its acknowledgement says so.
    clean_rca_sidecars(RCA_SIDECAR_FILE, keep_handled=bool(problem or skipped or unacknowledged or not ack))
    if problem:
        message = "The sidecar file %s wasn't used: %s" % (RCA_SIDECAR_FILE, problem)
    else:
        where = ""
        if starts:
            first, last = min(starts), max(ends)
            where = "from %s (%s; %s to %s UTC)" % (", ".join(sorted(hosts)), ", ".join(item for item in RCA_TARGETS if item in targets),
                                                     first.strftime("%Y-%m-%d %H:%M"), last.strftime("%H:%M" if last.date() == first.date() else "%Y-%m-%d %H:%M"))
        if wanted:
            message = "Wrote %d sidecar(s) %s: %d row(s)%s" % (len(wanted), where, sum(written.values()),
                                                              ("; %d more already in the history" % len(already)) if already else "")
        elif already:
            message = "The %d sidecar(s) %s were in the history already: each is written once" % (len(already), where)
        else:
            message = "Wrote no sidecar"
        if wanted or already:
            message += ", under %s" % " and ".join(
                ["session %s's run" % run[len("session-"):][:8] for run in sorted(runs) if run.startswith("session-") and run != RUN_ID]
                + (["this run's"] if any(run == RUN_ID for run in runs) else []))
        latest = [text % count for text, count in (("%d earlier read(s) in the history replaced", replaced),
                                                   ("%d not written because the history holds a later read", superseded),
                                                   ("%d earlier read(s) among these sidecars left out", left)) if count]
        if latest:
            message += "; Query Store's latest read of each interval kept (%s)" % ", ".join(latest)
        if skipped:
            message += "; and skipped %d sidecar(s) or line(s): %s" % (len(skipped), "; ".join(
                "%s: %s" % (document_id or "lines", why) for document_id, why in skipped[:3]) + ("; and %d more" % (len(skipped) - 3) if len(skipped) > 3 else ""))
    if unacknowledged:
        message += ". The scripts keep its sidecars pending, because " + unacknowledged
    elif ack:
        message += ". Acknowledged in %s%s" % (ack, ", and the file stays" if problem or skipped else "")
    if problem or skipped or unacknowledged:
        return _rca_sidecar_report("Warning", message + ".", sorted(wanted) + already, written, skipped or [(RCA_SIDECAR_FILE, problem)], replaced,
                                   superseded, ack)
    return _rca_sidecar_report("Pass", message + ".", sorted(wanted) + already, written, (), replaced, superseded, ack)

print("RCA sidecar helpers ready: %d tables (rca/history-tables.json, document sidecar)." % len(RCA_SIDECAR_TABLES))

In [ ]:
# The Incident RCA's sidecar (helpers, part 5): writes the sidecars in the file RCA_SIDECAR_FILE names to their six tables,
# each under its own session's run, acknowledges each by its ID in Files/rca/acks, deletes the file once every sidecar in
# it is in the history, and sweeps what earlier runs kept. It runs here, before section 3, which ends a run that saves
# only the evidence (RCA_ONLY) or a session already saved, so a sidecar is written in those runs too. Lakehouse data
# only, and only with SAVE_RESULTS or RCA_ONLY.
save_rca_sidecar()

## 3. This session

At a demo session's end, `NB_DatabaseHub_Control`'s Run all (ADR 0026) runs this notebook in its own Spark session (a reference run, with this workspace's lakehouse as its default; as a job in this workspace when it can't) with the session's ID (`SESSION_ID`), its beats (`SESSION_PLAN`; the Database Agent beat with when its run regresses and responds), its window, T4's and its replica's metrics (`POSTGRES_METRICS`), and T5's (`COSMOS_METRICS`). The next cell records the session and its beats on the targets this notebook analyzes as this run's scenario windows, so section 8 correlates each beat with the telemetry and section 11 saves them in `scenario_runs`, last, after the analysis. **A session already in `scenario_runs` was saved by an earlier run**, so this run writes only its Incident RCA evidence, when some was passed (each document is written once), stops at that cell, and saves nothing twice; that's how the control's next Run all saves an earlier session's history safely when its own end didn't. **In a reference run, each table and chart is one printed line** (its rows and columns, or its panels), because Fabric reads each cell's result itself, and a chart's stopped such a run on 2026-11-16; a job, or a run by hand, shows them in full.

With `RCA_ONLY = True`, the run writes only the Incident RCA's evidence and ends there: the control's next session exports a session's evidence that its own end didn't (when the nightly stop or the VMs' auto-shutdown ended it), and saves it this way under that session's ID.

Without `SESSION_ID`, this notebook analyzes the window in the parameters cell, which `deploy.ps1`'s history snapshot and `./scripts/stop-demo.ps1 -SaveHistory` use. It never changes the demo's activity and uses no GitHub Actions: `NB_DatabaseHub_Control` starts and stops a session, and `./scripts/demo-control.ps1` sets one activity from a PC.

In [ ]:
# This session: records the control's session and its beats as this run's scenario windows. No network calls; it reads
# scenario_runs, and ends the run when this session's history is already saved, or when the run saves only the Incident
# RCA's evidence (RCA_ONLY), once it has written that evidence (notebookutils.notebook.exit:
# https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-notebook-run).
SESSION = parse_session_plan(SESSION_ID, SESSION_PLAN)
if RCA_ONLY:
    # An earlier session's evidence, which the next session's start exported from the VMs (ADR 0031): the RCA tables only.
    print("RCA_ONLY: this run writes only the Incident RCA's evidence%s." % (" of session %s" % SESSION_ID[:8] if SESSION_ID else ""))
    # Whatever SAVE_RESULTS says: the control notebook passes it False with RCA_ONLY, so a copy of this notebook from before
    # the Incident RCA, which doesn't know RCA_ONLY, saves nothing.
    save_rca_evidence()
    SAVE_RESULTS = False
    if _notebookutils() is not None:
        _notebookutils().notebook.exit(rca_exit_value())
elif SESSION is None:
    print("No session: this run analyzes the window in the parameters cell.")
elif session_saved(SESSION["id"]):
    print("Session %s's history is already saved in scenario_runs, so this run saves nothing twice." % SESSION["id"][:8])
    # Its RCA evidence all the same, when some was passed: each document is written once, by its documentId (ADR 0031).
    save_rca_evidence()
    SAVE_RESULTS = False
    if _notebookutils() is not None:
        _notebookutils().notebook.exit("Session %s is already saved." % SESSION["id"] + (" " + rca_exit_value() if RCA_STATE["passed"] else ""))
else:
    SESSION_REQUESTS.extend(session_requests(SESSION))
    beats = [row for row in SESSION_REQUESTS if row["profile"].startswith("incident:")]
    window = ("%s to %s UTC" % (SESSION["start"].strftime("%Y-%m-%d %H:%M"), SESSION["end"].strftime("%H:%M"))) if SESSION["start"] and SESSION["end"] else "its window unknown"
    notes = [SESSION["note"]] if SESSION["note"] else []
    if SESSION["skipped"]:
        notes.append("%d beat(s) it couldn't use" % SESSION["skipped"])
    others = len(SESSION["beats"]) - len(beats)
    if others:
        notes.append("%d beat(s) on T4, T5, or another target this run doesn't analyze, which their Azure Monitor metrics show (section 7)" % others)
    print("Session %s, %s: %d beat(s) on %s recorded for the correlation.%s" % (
        SESSION["id"][:8], window, len(beats), ", ".join(sorted({row["target"] for row in beats})) or "no target",
        (" Left out: %s." % "; ".join(notes)) if notes else ""))

## 4. Telemetry access

The analysis uses the documented telemetry REST endpoint (`POST https://adx.centralus.arcdataservices.com/kusto/v1/rest/query`, database `ArcSqlTelemetry`), the same data as Database Hub **Performance**. Its documented token audience is `https://kusto.kusto.windows.net`; a notebook can only ask for its `kusto` audience key, and **whether the endpoint accepts that token isn't documented**. The next cell runs the article's connectivity query once:

- **Pass:** the analysis below runs in this notebook.
- **Manual action required:** every analysis cell prints its KQL instead. Paste it into the Azure Data Explorer web UI (connection `https://adx.centralus.arcdataservices.com/kusto/`, database `ArcSqlTelemetry`), or create a Real-Time Dashboard over the same endpoint (section 10).
- **Fail:** HTTP 403. You need Reader on the subscription, and `Microsoft.Sql` must be registered on it.

It also sets the analysis window. Telemetry keeps up to 7 days and arrives 3–15 minutes after the activity.

In [ ]:
# Telemetry access: validates the undocumented notebook kusto-token path once, and sets the analysis window. Read-only.
WINDOW_START, WINDOW_END, BIN = analysis_window()
print("Analysis window: %s to %s UTC, %d-minute bins." % (WINDOW_START.strftime("%Y-%m-%d %H:%M"), WINDOW_END.strftime("%Y-%m-%d %H:%M"), BIN))

PROBE_KQL = """SqlServerCPUUtilization
| where SampleTimeUTC > ago(15m)
| summarize Resources = dcount(ResourceID) by ResourceType"""
probe_status, probe_frame, probe_message = _post_kql(PROBE_KQL)
TELEMETRY_STATE.update(checked=True, available=(probe_status == 200), message=probe_message)
if probe_status == 200:
    add_finding("telemetry_access", "fabric", "Pass",
                "The telemetry endpoint accepted the notebook's kusto token, so the analysis runs here.", "token")
elif probe_status == 403:
    add_finding("telemetry_access", "fabric", "Fail",
                probe_message + " Ask for Reader on the subscription (P8) and register Microsoft.Sql (P2).", "token")
else:
    add_finding("telemetry_access", "fabric", "Manual action required",
                "The notebook can't query the telemetry endpoint directly (%s). Each analysis cell prints its KQL for the Azure Data Explorer web UI." % probe_message,
                "token", evidence={"httpStatus": probe_status})

## 5. Estate pulse

Which demo resources are reporting, how many telemetry tables each sends, and how old the last sample is (the telemetry article's query 1, scoped to `RESOURCE_GROUP` and the demo's names). A resource that's missing either isn't running, hasn't reported yet, or doesn't have monitoring enabled: an empty result never means healthy.

In [ ]:
# Estate pulse: resources reporting and the age of their last sample. Read-only.
PULSE_KQL = kql_prelude(WINDOW_START, WINDOW_END, BIN) + """
union withsource=TableName SqlServer*
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize Rows = count(), Tables = dcount(TableName), LastSampleUTC = max(SampleTimeUTC), Instances = dcount(SqlServerInstanceName)
        by Target, ResourceType, ResourceName
| order by Target asc"""

pulse = query_telemetry(PULSE_KQL, "Estate pulse")
FRAMES["pulse"] = pulse
if pulse is not None:
    if not pulse.empty:
        pulse["LastSampleAgeMinutes"] = ((WINDOW_END - pulse["LastSampleUTC"]).dt.total_seconds() / 60).round(1)
        display_frame(pulse[["Target", "ResourceType", "ResourceName", "Tables", "Rows", "Instances", "LastSampleAgeMinutes"]])
    for target in SELECTED_TARGETS:
        rows = pulse[pulse["Target"] == target] if not pulse.empty else pulse
        if rows.empty:
            add_finding("estate_pulse", target, "Warning",
                        "%s sent no telemetry in the window: not running, not reporting yet (3-15 minutes), or monitoring isn't enabled." % target, "reporting")
        else:
            age = float(rows["LastSampleAgeMinutes"].min())
            status = "Pass" if age <= 15 else "Warning"
            add_finding("estate_pulse", target, status,
                        "%s (%s) reports %d table(s); the last sample is %.1f minutes old." % (target, rows["ResourceName"].iloc[0], int(rows["Tables"].max()), age),
                        "reporting", evidence={"tables": int(rows["Tables"].max()), "rows": int(rows["Rows"].sum()), "lastSampleAgeMinutes": age})
    add_snapshot("estate_pulse", pulse, WINDOW_START, WINDOW_END, metrics=["Rows", "Tables", "LastSampleAgeMinutes"],
                 units={"Rows": "rows", "Tables": "tables", "LastSampleAgeMinutes": "minutes"})

## 6. Locks and blocking

| Signal | Source | How it's computed |
|---|---|---|
| `Processes blocked` timeline | `SqlServerPerformanceCountersCommon`, `CounterType` 65792 (gauge) | Maximum per bin |
| Lock-category waits | `SqlServerWaitStats` (cumulative) | Per-series deltas, resets skipped, summed per bin |
| `Active Transactions` | Performance counters (gauge, per database) | Maximum across databases per bin |
| Active sessions in `ContosoCare` | `SqlServerActiveSessions` (gauge) | Distinct sessions per sample, maximum per bin |
| Deadlocks | `Number of Deadlocks/sec`, `CounterType` 272696576 (cumulative) | Per-series deltas per bin (`_Total` instance when present) |
| **Incidents detected** | `Processes blocked` | Consecutive bins with at least one blocked process |

An incident inside a recorded scenario window for the same target is expected (**Pass**); one outside any window is unexpected (**Warning**). The telemetry describes resources, not queries: for the blocking chain itself, use SSMS or the `diagnose-blocking-and-deadlocks` skill on the target.

In [ ]:
# Locks and blocking: gauges, cumulative counters and waits as per-bin deltas, sessions, incidents, and a chart. Read-only.
prelude = kql_prelude(WINDOW_START, WINDOW_END, BIN)

GAUGES_KQL = prelude + """
SqlServerPerformanceCountersCommon
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where CounterType == 65792
| where CounterName in ('Processes blocked', 'Active Transactions', 'User Connections')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize MaxValue = max(CounterValue), AvgValue = avg(CounterValue)
        by Target, ResourceName, SqlServerInstanceName, CounterName, Bucket = bin(SampleTimeUTC, _bin)
| order by Bucket asc
| take 50000"""

# Cumulative counters: keep the first and last sample per series and bin, then sum increases between adjacent
# samples and skip any interval where the value went down (a restart or failover reset).
RATES_KQL = prelude + """
SqlServerPerformanceCountersCommon
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where CounterType == 272696576
| where CounterName in ('Number of Deadlocks/sec', 'Batch Requests/sec', 'Transactions/sec', 'Log Bytes Flushed/sec')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize (FirstT, FirstV) = arg_min(SampleTimeUTC, CounterValue), (LastT, LastV) = arg_max(SampleTimeUTC, CounterValue)
        by Target, ResourceID, SqlServerInstanceName, CounterName, InstanceName, Bucket = bin(SampleTimeUTC, _bin)
| extend T = pack_array(FirstT, LastT), V = pack_array(FirstV, LastV)
| mv-expand T to typeof(datetime), V to typeof(real)
| distinct Target, ResourceID, SqlServerInstanceName, CounterName, InstanceName, Bucket, T, V
| sort by ResourceID asc, SqlServerInstanceName asc, CounterName asc, InstanceName asc, T asc
| extend Keep = ResourceID == prev(ResourceID) and SqlServerInstanceName == prev(SqlServerInstanceName)
                and CounterName == prev(CounterName) and InstanceName == prev(InstanceName) and V >= prev(V)
| extend Delta = iff(Keep, V - prev(V), real(0)), Seconds = iff(Keep, datetime_diff('second', T, prev(T)), long(0))
| summarize Delta = sum(Delta), Seconds = sum(Seconds) by Target, CounterName, InstanceName, Bucket
| order by Bucket asc
| take 50000"""

WAITS_KQL = prelude + """
SqlServerWaitStats
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where WaitCategory !in ('Idle', 'Unknown')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize (FirstT, FirstW, FirstC) = arg_min(SampleTimeUTC, WaitTimeMs, WaitingTasksCount),
            (LastT, LastW, LastC) = arg_max(SampleTimeUTC, WaitTimeMs, WaitingTasksCount)
        by Target, ResourceID, SqlServerInstanceName, WaitCategory, WaitType, Bucket = bin(SampleTimeUTC, _bin)
| extend T = pack_array(FirstT, LastT), W = pack_array(FirstW, LastW), C = pack_array(FirstC, LastC)
| mv-expand T to typeof(datetime), W to typeof(long), C to typeof(long)
| distinct Target, ResourceID, SqlServerInstanceName, WaitCategory, WaitType, Bucket, T, W, C
| sort by ResourceID asc, SqlServerInstanceName asc, WaitType asc, T asc
| extend Keep = ResourceID == prev(ResourceID) and SqlServerInstanceName == prev(SqlServerInstanceName)
                and WaitType == prev(WaitType) and W >= prev(W) and C >= prev(C)
| extend WaitMs = iff(Keep, W - prev(W), long(0)), Tasks = iff(Keep, C - prev(C), long(0))
| summarize WaitMs = sum(WaitMs), Tasks = sum(Tasks) by Target, WaitCategory, PageIoLatch = WaitType startswith 'PAGEIOLATCH', Bucket
| where WaitMs > 0
| order by Bucket asc
| take 50000"""

SESSIONS_KQL = prelude + """
SqlServerActiveSessions
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where DatabaseName =~ _db
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize Sessions = dcount(SessionID), Running = dcountif(SessionID, SessionStatus == 'running')
        by Target, SqlServerInstanceName, SampleTimeUTC
| summarize MaxSessions = max(Sessions), AvgSessions = avg(Sessions), MaxRunning = max(Running)
        by Target, SqlServerInstanceName, Bucket = bin(SampleTimeUTC, _bin)
| order by Bucket asc"""


def prefer_total(rates):
    """Per target, counter, and bin: the _Total instance when the counter has one, otherwise the sum of instances."""
    if rates is None or rates.empty:
        return rates
    frame = rates.copy()
    frame["IsTotal"] = frame["InstanceName"].astype(str).str.strip().str.lower() == "_total"
    has_total = frame.groupby(["Target", "CounterName"])["IsTotal"].transform("any")
    chosen = frame[(~has_total) | frame["IsTotal"]]
    return chosen.groupby(["Target", "CounterName", "Bucket"], as_index=False)[["Delta", "Seconds"]].sum()


def blocking_incidents(gauges, bin_minutes):
    """Groups consecutive bins with at least one blocked process into incidents, per target (signal_episodes)."""
    blocked = to_series(gauges, "MaxValue", "max", lambda f: f["CounterName"] == "Processes blocked")
    return signal_episodes(blocked, 0, bin_minutes, 1, strict=True)


gauges = query_telemetry(GAUGES_KQL, "Blocking and transaction gauges")
rates = query_telemetry(RATES_KQL, "Cumulative counters as deltas")
waits = query_telemetry(WAITS_KQL, "Wait deltas by category")
sessions = query_telemetry(SESSIONS_KQL, "Active sessions in the demo database")
FRAMES.update(gauges=gauges, rates=prefer_total(rates), waits=waits, sessions=sessions)
WINDOWS = scenario_windows(WINDOW_START, WINDOW_END)

blocked_series = to_series(gauges, "MaxValue", "max", lambda f: f["CounterName"] == "Processes blocked")
lock_series = to_series(waits, "WaitMs", "sum", lambda f: f["WaitCategory"] == "Lock")
if lock_series is not None:
    lock_series["Value"] = lock_series["Value"] / 1000.0
transactions_series = to_series(gauges, "MaxValue", "max", lambda f: f["CounterName"] == "Active Transactions")
sessions_series = to_series(sessions, "MaxSessions", "sum")
deadlock_series = to_series(FRAMES["rates"], "Delta", "sum", lambda f: f["CounterName"] == "Number of Deadlocks/sec")
plot_panels("Locks and blocking", [
    ("Processes blocked (maximum per bin)", blocked_series, "processes"),
    ("Lock waits (seconds per bin)", lock_series, "seconds"),
    ("Active transactions (maximum across databases)", transactions_series, "transactions"),
    ("Active sessions in %s (maximum per bin)" % DEMO_DATABASE, sessions_series, "sessions"),
    ("Deadlocks (per bin)", deadlock_series, "deadlocks"),
], WINDOWS)

if gauges is not None:
    incidents = blocking_incidents(gauges, BIN)
    if not incidents:
        print("No blocking incidents in the window.")
    for incident in incidents:
        expected = any(incident["target"] in window["targets"] and window["start"] <= incident["end"]
                       and incident["start"] <= window["end"] + dt.timedelta(minutes=int(TELEMETRY_LAG_MINUTES)) for window in WINDOWS)
        lock_seconds = 0.0
        if waits is not None and not waits.empty:
            inside = waits[(waits["Target"] == incident["target"]) & (waits["WaitCategory"] == "Lock")
                           & (waits["Bucket"] >= incident["start"]) & (waits["Bucket"] < incident["end"])]
            lock_seconds = round(_safe_number(inside["WaitMs"].sum()) / 1000.0, 1)
        add_finding("blocking", incident["target"], "Pass" if expected else "Warning",
                    "Blocking on %s from %s to %s UTC: up to %d blocked process(es), %.1f s of lock waits%s." % (
                        incident["target"], incident["start"].strftime("%H:%M"), incident["end"].strftime("%H:%M"),
                        int(incident["peak"]), lock_seconds, " during a recorded scenario" if expected else ", outside any recorded scenario"),
                    incident["start"].strftime("%Y%m%dT%H%M"), evidence={"peakBlocked": incident["peak"], "lockWaitSeconds": lock_seconds},
                    start=incident["start"].to_pydatetime(), end=incident["end"].to_pydatetime())

if deadlock_series is not None:
    for target, group in deadlock_series.groupby("Target"):
        total = int(round(_safe_number(group["Value"].sum())))
        if total > 0:
            add_finding("blocking", target, "Warning", "%d deadlock(s) on %s in the window." % (total, target), "deadlocks",
                        evidence={"deadlocks": total}, start=WINDOW_START, end=WINDOW_END)

add_snapshot("gauges", gauges, WINDOW_START, WINDOW_END, metric_column="CounterName", value_column="MaxValue", units={})
add_snapshot("counter_deltas", FRAMES["rates"], WINDOW_START, WINDOW_END, metric_column="CounterName", value_column="Delta")
add_snapshot("sessions", sessions, WINDOW_START, WINDOW_END, metrics=["MaxSessions", "AvgSessions", "MaxRunning"],
             units={"MaxSessions": "sessions", "AvgSessions": "sessions", "MaxRunning": "sessions"})

## 7. Overall consumption

| Signal | Source | Notes |
|---|---|---|
| CPU average, 95th percentile, maximum, and time above 80% | `SqlServerCPUUtilization` (gauge) | T1 and T2 read `SQLProcessCPUPercent`, the SQL Server process's share of the machine, because `AvgCPUPercent` stayed at 0 for SQL Server when measured on 2026-10-23; both columns are in the table's documented schema. T3 reads `AvgCPUPercent`, a percentage of the service tier's limit |
| CPU timeline | `SqlServerCPUUtilization` | Average and maximum per bin |
| Wait profile | `SqlServerWaitStats` deltas | Share of wait time per category, `Idle` and `Unknown` excluded |
| `PAGEIOLATCH` waits | Buffer IO wait types | Data pages read from storage: scans, IO pressure, or a missing index |
| Storage IO | `SqlServerStorageIO` (cumulative) | Per-file deltas: MB read and written per bin, after adding the files that share a database and file ID within a sample (on Azure SQL Database, two files share FileID 1 and 2 in every sample); read and write latency for T1 and T2 only, because per-file latency isn't supported for Azure SQL Database |
| Memory clerks | `SqlServerMemoryUtilization` (gauge) | Largest clerk types at the latest sample |
| Database growth | `SqlServerDatabaseStorageUtilization` (gauge) | Used data and log space at the start and end of the window |
| **T3 `Log Rate Governor` waits** | `SqlServerWaitStats` | Azure SQL Database's log-rate governance; the T3 log-rate scenario's signature |
| Buffer Latch, Memory, Parallelism, Tran Log IO, and Network IO waits | `SqlServerWaitStats` deltas | The Phase 25b scenarios' signatures: tempdb contention (`PAGELATCH_*`), memory grants (`RESOURCE_SEMAPHORE`), parallel plans (`CXPACKET`, `CXCONSUMER`), commits (`WRITELOG`), and a slow client (`ASYNC_NETWORK_IO`); charted when present, and a finding when one passes `EPISODE_WAIT_SECONDS` and 10% of a target's wait time |

In [ ]:
# Overall consumption: CPU, waits, storage IO, memory, database growth, and T3 log-rate governance. Read-only.
CPU_SUMMARY_KQL = prelude + """
SqlServerCPUUtilization
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| extend CpuPercent = iff(ResourceType =~ 'servers/databases', AvgCPUPercent, coalesce(todouble(SQLProcessCPUPercent), AvgCPUPercent))
| summarize AvgCPU = round(avg(CpuPercent), 1), P95CPU = round(percentile(CpuPercent, 95), 1),
            MaxCPU = round(max(CpuPercent), 1), PctTimeAbove80 = round(100.0 * countif(CpuPercent > 80.0) / count(), 1)
        by Target, ResourceType, ResourceName, SqlServerInstanceName
| order by Target asc"""

CPU_TIMELINE_KQL = prelude + """
SqlServerCPUUtilization
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| extend CpuPercent = iff(ResourceType =~ 'servers/databases', AvgCPUPercent, coalesce(todouble(SQLProcessCPUPercent), AvgCPUPercent))
| summarize AvgCPU = avg(CpuPercent), MaxCPU = max(CpuPercent) by Target, ResourceName, SqlServerInstanceName, Bucket = bin(SampleTimeUTC, _bin)
| order by Bucket asc"""

# Storage IO (F36): on Azure SQL Database, every sample holds two files under each of DatabaseID and FileID 1 and 2,
# so the query adds the files of a key within each sample before its deltas, and skips a step whose number of files
# changed. Kql/Live.StorageIoDeltas.kql says why.
STORAGE_IO_KQL = prelude + """
SqlServerStorageIO
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where FileType in ('Data', 'Log')
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize Files = count(), R = sum(NumOfBytesRead), W = sum(NumOfBytesWritten), RS = sum(IOStallReadMs), WS = sum(IOStallWriteMs),
            NR = sum(NumOfReads), NW = sum(NumOfWrites)
        by Target, ResourceID, SqlServerInstanceName, DatabaseID, FileID, FileType, SampleTimeUTC
| summarize (FirstT, F1, R1, W1, RS1, WS1, NR1, NW1) = arg_min(SampleTimeUTC, Files, R, W, RS, WS, NR, NW),
            (LastT, F2, R2, W2, RS2, WS2, NR2, NW2) = arg_max(SampleTimeUTC, Files, R, W, RS, WS, NR, NW)
        by Target, ResourceID, SqlServerInstanceName, DatabaseID, FileID, FileType, Bucket = bin(SampleTimeUTC, _bin)
| extend T = pack_array(FirstT, LastT), F = pack_array(F1, F2), R = pack_array(R1, R2), W = pack_array(W1, W2), RS = pack_array(RS1, RS2),
         WS = pack_array(WS1, WS2), NR = pack_array(NR1, NR2), NW = pack_array(NW1, NW2)
| mv-expand T to typeof(datetime), F to typeof(long), R to typeof(long), W to typeof(long), RS to typeof(long), WS to typeof(long), NR to typeof(long), NW to typeof(long)
| distinct Target, ResourceID, SqlServerInstanceName, DatabaseID, FileID, FileType, Bucket, T, F, R, W, RS, WS, NR, NW
| sort by ResourceID asc, SqlServerInstanceName asc, DatabaseID asc, FileID asc, T asc
| extend Keep = ResourceID == prev(ResourceID) and SqlServerInstanceName == prev(SqlServerInstanceName)
                and DatabaseID == prev(DatabaseID) and FileID == prev(FileID) and F == prev(F)
                and R >= prev(R) and W >= prev(W) and RS >= prev(RS) and WS >= prev(WS) and NR >= prev(NR) and NW >= prev(NW)
| extend dR = iff(Keep, R - prev(R), long(0)), dW = iff(Keep, W - prev(W), long(0)), dRS = iff(Keep, RS - prev(RS), long(0)),
         dWS = iff(Keep, WS - prev(WS), long(0)), dNR = iff(Keep, NR - prev(NR), long(0)), dNW = iff(Keep, NW - prev(NW), long(0))
| summarize ReadMB = round(sum(dR) / 1048576.0, 2), WriteMB = round(sum(dW) / 1048576.0, 2), ReadStallMs = sum(dRS),
            WriteStallMs = sum(dWS), Reads = sum(dNR), Writes = sum(dNW) by Target, FileType, Bucket
| order by Bucket asc
| take 50000"""

MEMORY_KQL = prelude + """
SqlServerMemoryUtilization
| where SampleTimeUTC between ((_end - 1h) .. _end)
| where ResourceGroup =~ _rg
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize arg_max(SampleTimeUTC, MemorySizeMB) by Target, SqlServerInstanceName, MemoryClerkType, MemoryClerkName
| summarize MemoryMB = round(sum(MemorySizeMB), 1), LastSampleUTC = max(SampleTimeUTC) by Target, SqlServerInstanceName, MemoryClerkType
| where MemoryMB >= 1
| order by Target asc, MemoryMB desc
| take 1000"""

GROWTH_KQL = prelude + """
SqlServerDatabaseStorageUtilization
| where SampleTimeUTC between (_start .. _end)
| where ResourceGroup =~ _rg
| where DatabaseName =~ _db
| extend Target = _target(ResourceType, ResourceName)
| where isnotempty(Target)
| summarize (FirstUTC, StartUsedMB) = arg_min(SampleTimeUTC, DataSizeUsedMB),
            (LastUTC, EndUsedMB, AllocatedMB, LogUsedMB, VersionStoreMB) =
                arg_max(SampleTimeUTC, DataSizeUsedMB, DataSizeAllocatedMB, LogSizeUsedMB, PersistentVersionStoreSizeMB)
        by Target, ResourceName, SqlServerInstanceName, DatabaseName
| extend GrowthMB = round(EndUsedMB - StartUsedMB, 1)
| order by Target asc"""

cpu_summary = query_telemetry(CPU_SUMMARY_KQL, "CPU summary")
cpu = query_telemetry(CPU_TIMELINE_KQL, "CPU timeline")
storage_io = query_telemetry(STORAGE_IO_KQL, "Storage IO deltas")
memory = query_telemetry(MEMORY_KQL, "Memory clerks")
growth = query_telemetry(GROWTH_KQL, "Database growth")
FRAMES.update(cpu=cpu, storage_io=storage_io, memory=memory, growth=growth)

if cpu_summary is not None and not cpu_summary.empty:
    print("CPU over the window (T3: percentage of the service tier's limit):")
    display_frame(cpu_summary[["Target", "ResourceName", "SqlServerInstanceName", "AvgCPU", "P95CPU", "MaxCPU", "PctTimeAbove80"]])
    for _, row in cpu_summary.iterrows():
        hot = _safe_number(row["P95CPU"]) >= 80
        add_finding("consumption", row["Target"], "Warning" if hot else "Pass",
                    "%s CPU: average %.1f%%, 95th percentile %.1f%%, maximum %.1f%%, %.1f%% of the window above 80%%%s." % (
                        row["Target"], _safe_number(row["AvgCPU"]), _safe_number(row["P95CPU"]), _safe_number(row["MaxCPU"]),
                        _safe_number(row["PctTimeAbove80"]), ": sustained CPU pressure" if hot else ""),
                    "cpu", evidence={"avg": row["AvgCPU"], "p95": row["P95CPU"], "max": row["MaxCPU"]}, start=WINDOW_START, end=WINDOW_END)

waits = FRAMES.get("waits")
if waits is not None and not waits.empty:
    profile = waits.groupby(["Target", "WaitCategory"], as_index=False)["WaitMs"].sum()
    profile["WaitSeconds"] = (profile["WaitMs"] / 1000.0).round(1)
    profile["SharePct"] = (100.0 * profile["WaitMs"] / profile.groupby("Target")["WaitMs"].transform("sum")).round(1)
    print("Wait profile per target (share of non-idle wait time in the window):")
    display_frame(profile.sort_values(["Target", "SharePct"], ascending=[True, False])[["Target", "WaitCategory", "WaitSeconds", "SharePct"]])
    log_rate = profile[profile["WaitCategory"] == "Log Rate Governor"]
    for _, row in log_rate.iterrows():
        add_finding("consumption", row["Target"], "Warning",
                    "%s waited %.1f s on log-rate governance (%.1f%% of its wait time)." % (row["Target"], row["WaitSeconds"], row["SharePct"]),
                    "log_rate", evidence={"waitSeconds": row["WaitSeconds"], "sharePct": row["SharePct"]}, start=WINDOW_START, end=WINDOW_END)
    # The Phase 25b wait families: a finding when one passes EPISODE_WAIT_SECONDS and 10% of a target's wait time.
    for family in ("latch", "memory", "parallelism", "tran_log", "network"):
        part = profile[(profile["WaitCategory"] == WAIT_FAMILY_CATEGORIES[family]) & (profile["WaitSeconds"] >= EPISODE_WAIT_SECONDS)
                       & (profile["SharePct"] >= 10)]
        for _, row in part.iterrows():
            add_finding("consumption", row["Target"], "Warning",
                        "%s waited %.1f s on %s (%.1f%% of its wait time): %s." % (
                            row["Target"], row["WaitSeconds"], row["WaitCategory"], row["SharePct"], SIGNAL_LABELS[family].lower()),
                        family, evidence={"waitSeconds": row["WaitSeconds"], "sharePct": row["SharePct"]}, start=WINDOW_START, end=WINDOW_END)
    add_snapshot("wait_profile", profile.assign(Bucket=None), WINDOW_START, WINDOW_END, metric_column="WaitCategory", value_column="WaitSeconds")
    add_snapshot("wait_deltas", waits.assign(Metric=waits["WaitCategory"] + waits["PageIoLatch"].map({True: " (PAGEIOLATCH)", False: ""}).fillna("")),
                 WINDOW_START, WINDOW_END, metric_column="Metric", value_column="WaitMs")

if storage_io is not None and not storage_io.empty:
    io_view = storage_io.groupby("Target", as_index=False)[["ReadMB", "WriteMB", "ReadStallMs", "WriteStallMs", "Reads", "Writes"]].sum()
    io_view["ReadLatencyMs"] = (io_view["ReadStallMs"] / io_view["Reads"].where(io_view["Reads"] > 0)).round(2)
    io_view["WriteLatencyMs"] = (io_view["WriteStallMs"] / io_view["Writes"].where(io_view["Writes"] > 0)).round(2)
    # Per-file IO latency from this table isn't supported for Azure SQL Database (telemetry article, rule 5).
    io_view.loc[io_view["Target"] == "T3", ["ReadLatencyMs", "WriteLatencyMs"]] = None
    print("Storage IO in the window (data and log files):")
    display_frame(io_view[["Target", "ReadMB", "WriteMB", "Reads", "Writes", "ReadLatencyMs", "WriteLatencyMs"]])
    add_snapshot("storage_io", storage_io, WINDOW_START, WINDOW_END, metrics=["ReadMB", "WriteMB", "Reads", "Writes"],
                 units={"ReadMB": "MB", "WriteMB": "MB", "Reads": "reads", "Writes": "writes"})

if memory is not None and not memory.empty:
    top_memory = memory.sort_values("MemoryMB", ascending=False).groupby("Target").head(8)
    print("Largest memory clerk types at the latest sample:")
    display_frame(top_memory[["Target", "SqlServerInstanceName", "MemoryClerkType", "MemoryMB"]])
    add_snapshot("memory_clerks", top_memory.assign(Bucket=top_memory["LastSampleUTC"]), WINDOW_START, WINDOW_END,
                 metric_column="MemoryClerkType", value_column="MemoryMB")

if growth is not None and not growth.empty:
    print("Database %s storage over the window:" % DEMO_DATABASE)
    display_frame(growth[["Target", "ResourceName", "StartUsedMB", "EndUsedMB", "GrowthMB", "AllocatedMB", "LogUsedMB", "VersionStoreMB"]])
    add_snapshot("database_growth", growth.assign(Bucket=growth["LastUTC"]), WINDOW_START, WINDOW_END,
                 metrics=["StartUsedMB", "EndUsedMB", "GrowthMB", "LogUsedMB", "VersionStoreMB"],
                 units={"StartUsedMB": "MB", "EndUsedMB": "MB", "GrowthMB": "MB", "LogUsedMB": "MB", "VersionStoreMB": "MB"})

cpu_series = to_series(cpu, "AvgCPU", "max")
pageio_series = to_series(waits, "WaitMs", "sum", lambda f: f["PageIoLatch"].astype(bool))
if pageio_series is not None:
    pageio_series["Value"] = pageio_series["Value"] / 1000.0
log_rate_series = to_series(waits, "WaitMs", "sum", lambda f: f["WaitCategory"] == "Log Rate Governor")
if log_rate_series is not None:
    log_rate_series["Value"] = log_rate_series["Value"] / 1000.0
family_series = {}
for family in ("latch", "memory", "parallelism", "tran_log", "network"):
    family_series[family] = episode_series(family)
io_series = None
if storage_io is not None and not storage_io.empty:
    io_series = to_series(storage_io.assign(TotalMB=storage_io["ReadMB"] + storage_io["WriteMB"]), "TotalMB", "sum")
plot_panels("Overall consumption", [
    ("CPU, average per bin (T3: % of the tier limit)", cpu_series, "% CPU"),
    ("PAGEIOLATCH (Buffer IO) waits, seconds per bin", pageio_series, "seconds"),
    ("Log Rate Governor waits, seconds per bin", log_rate_series, "seconds"),
    ("Data and log IO, MB read and written per bin", io_series, "MB"),
    ("Buffer Latch waits (PAGELATCH, tempdb contention), seconds per bin", family_series["latch"], "seconds"),
    ("Memory waits (RESOURCE_SEMAPHORE), seconds per bin", family_series["memory"], "seconds"),
    ("Parallelism waits (CXPACKET, CXCONSUMER), seconds per bin", family_series["parallelism"], "seconds"),
    ("Tran Log IO waits (WRITELOG), seconds per bin", family_series["tran_log"], "seconds"),
    ("Network IO waits (ASYNC_NETWORK_IO), seconds per bin", family_series["network"], "seconds"),
], WINDOWS)
add_snapshot("cpu", cpu, WINDOW_START, WINDOW_END, metrics=["AvgCPU", "MaxCPU"], units={"AvgCPU": "percent", "MaxCPU": "percent"})

### T4: PostgreSQL and its read replica (Azure Monitor)

This notebook can't read T4 or its read replica: its tokens (`notebookutils`) cover storage, Power BI, Key Vault, and Kusto, not Azure Resource Manager, and Database Hub's PostgreSQL dashboard charts their Azure Monitor metrics, which aren't in the telemetry endpoint. So `./scripts/stop-demo.ps1 -SaveHistory` (before it deletes the replica, whose metrics go with it) and `./scripts/deploy.ps1`'s history snapshot read them with the Azure CLI ([Metrics - List](https://learn.microsoft.com/rest/api/monitor/metrics/list)) and pass them in the `POSTGRES_METRICS` parameter. The next cell checks that document and adds it to this run's results; it makes no network call.

| Metric | Server | Per interval |
|---|---|---|
| `physical_replication_delay_in_seconds`: Read replica lag, as Database Hub charts it | The read replica | Maximum |
| `cpu_percent`, `active_connections` | T4 and the read replica | Maximum |
| `memory_percent`, `physical_replication_delay_in_bytes` (Max Physical Replication Lag) | T4 | Maximum |
| `connections_failed` | T4 | Total |

- **`telemetry_snapshots`:** one row per point, with `analysis` `postgresql`, `target` `T4` (the primary, `pgflex01-<deploymentId>`) or `T4-replica` (`pgreplica01-<deploymentId>`), the server's name, the metric, the interval's start, the value and its unit, and the window the metrics cover.
- **`analysis_findings`:** `postgresql.T4.capture` (what was captured), `postgresql.T4-replica.lag` (the replica's peak lag: **Warning** at `REPLICA_LAG_WARNING_SECONDS`, 60, or more; **Not selected** without a replica, which exists only during a session), `postgresql.T4.cpu` (T4's peak CPU: **Warning** at `EPISODE_CPU_PERCENT` or more), `postgresql.T4.connections` (T4's peak active connections: **Warning** at `POSTGRES_USABLE_CONNECTIONS`, 35, or more, the slots clients can use: `max_connections` 50, less the 15 Azure reserves; `connection-surge`'s signal), and `postgresql.T4.connections_failed` (**Warning** above 0). Failed connections leave out a client refused at the connection limit (too many connections): Azure Monitor counted none of `connection-surge`'s refused attempts live (2026-11-21), so a surge shows in `postgresql.T4.connections`, never there. A document it can't use is one **Fail** finding and adds no rows.

With `POSTGRES_METRICS` empty, the cell says so in one line and adds nothing. For T4 live, use Database Hub's PostgreSQL dashboard (section 9). The thresholds are presenter heuristics, not product behavior; the connection slots are T4's own ([limits](https://learn.microsoft.com/azure/postgresql/configure-maintain/concepts-limits)).

In [ ]:
# T4 (PostgreSQL): the Azure Monitor metrics that the operator's scripts passed in POSTGRES_METRICS. No network calls.
if not POSTGRES_METRICS:
    print("T4's PostgreSQL metrics weren't passed in (POSTGRES_METRICS is empty), so this run adds none: "
          "./scripts/stop-demo.ps1 -SaveHistory and ./scripts/deploy.ps1 pass them; live, see Database Hub's PostgreSQL dashboard.")
else:
    POSTGRES_CAPTURE = record_postgres_metrics(POSTGRES_METRICS)
    if POSTGRES_CAPTURE is not None and POSTGRES_CAPTURE["series"]:
        peaks = []
        for series in POSTGRES_CAPTURE["series"]:
            at, peak = _pg_peak(series)
            peaks.append({"Target": series["target"], "Server": series["resourceName"], "Metric": series["metric"],
                          "Aggregation": series["aggregation"], "Unit": series["unit"], "Points": len(series["points"]),
                          "Peak": peak, "PeakUTC": _pg_text(at)})
        print("T4's metrics from %s to %s UTC, every %d minutes:" % (
            _pg_text(POSTGRES_CAPTURE["start"]), _pg_text(POSTGRES_CAPTURE["end"]), POSTGRES_CAPTURE["intervalMinutes"]))
        display_frame(pd.DataFrame(peaks))
        plot_panels("T4: PostgreSQL and its read replica (Azure Monitor, from POSTGRES_METRICS)", [
            ("Read replica lag, seconds (maximum per interval)", postgres_series_frame("physical_replication_delay_in_seconds"), "seconds"),
            ("CPU (maximum per interval)", postgres_series_frame("cpu_percent"), "% CPU"),
            ("Active connections (maximum per interval)", postgres_series_frame("active_connections"), "connections"),
            ("Failed connections per interval", postgres_series_frame("connections_failed"), "connections"),
        ])

### T5: Azure Cosmos DB (Azure Monitor)

This notebook can't read T5 either: Database Hub's Cosmos DB views chart its Azure Monitor platform metrics, which aren't in the telemetry endpoint ([Monitor a Cosmos DB account in the Database Hub](https://learn.microsoft.com/fabric/database/hub/monitor-cosmos-db)). So the control's runbook (at a session's end, into the control vault), `./scripts/stop-demo.ps1 -SaveHistory`, `./scripts/deploy.ps1`'s history snapshot, and `./scripts/configure-fabric.ps1 -RunLiveActivity` read them ([Metrics - List](https://learn.microsoft.com/rest/api/monitor/metrics/list)) and pass them in the `COSMOS_METRICS` parameter. T5's account stays between sessions, so its metrics can be read any time within Azure Monitor's retention. The next cell checks that document and adds it to this run's results; it makes no network call.

| Metric | Database Hub's signal | Per interval |
|---|---|---|
| `NormalizedRUConsumption` | Normalized RU Consumption (Peak Norm RU %) | Maximum |
| `TotalRequestUnits` | Total Request Units (Total RU consumed) | Total |
| `TotalRequests` | The requests that the 429 Rate % divides by | Count |
| `ThrottledRequests`: `TotalRequests` with `StatusCode` 429 | Throttled Requests (429s), and with `TotalRequests` the 429 Rate % | Count |
| `ServerSideLatencyGateway` | None: the Hub's Avg Latency is `ServerSideLatencyDirect`, which stays empty because the runner uses gateway mode | Average |
| `ProvisionedThroughput` | Provisioned throughput (Provisioned RU/s), hourly | Maximum |
| `ServiceAvailability` | Availability %, hourly | Average |

- **`telemetry_snapshots`:** one row per point, with `analysis` `cosmosdb`, `target` `T5`, `resource_type` `databaseAccounts`, the account's name (`cosmos01-<deploymentId>`) in `resource_name`, the metric (the names above), the interval's start, the value and its unit (`percent`, `RU`, `requests`, `ms`, or `RU/s`), and the window the metrics cover. `ThrottledRequests` is 0 in each interval with requests and no 429s, because Azure Monitor sends no point for a count of zero.
- **`analysis_findings`:** `cosmosdb.T5.capture` (what was captured); `cosmosdb.T5.throttling` (normalized RU and the throttled requests over the window: **Pass** without a 429, **Warning** with them); `cosmosdb.T5.throttle.<yyyymmddThhmm>` (each throttling episode, at most 24, its intervals with 429s less than `COSMOS_EPISODE_GAP_MINUTES`, 3, apart: when normalized RU reached 100% with how many 429s, and when the throttling stopped and normalized RU fell back); and, with 429s, `cosmosdb.T5.recovery` (**Pass** when T5 was back at its baseline, normalized RU under 100% with no 429s, before the capture's end; **Warning** otherwise). A document it can't use is one **Fail** finding and adds no rows.

With `COSMOS_METRICS` empty, the cell says so in one line and adds nothing. For T5 live, use Database Hub's Cosmos DB views (section 9). The episode gap is a presenter heuristic, not product behavior.

In [ ]:
# T5 (Azure Cosmos DB): the Azure Monitor metrics that the control or the operator's scripts passed in COSMOS_METRICS. No
# network calls.
if not COSMOS_METRICS:
    print("T5's Azure Cosmos DB metrics weren't passed in (COSMOS_METRICS is empty), so this run adds none: the control's "
          "Run all, ./scripts/stop-demo.ps1 -SaveHistory, and ./scripts/deploy.ps1 pass them; live, see Database Hub's Cosmos DB views.")
else:
    COSMOS_CAPTURE = record_cosmos_metrics(COSMOS_METRICS)
    if COSMOS_CAPTURE is not None and any(series["points"] for series in COSMOS_CAPTURE["series"]):
        peaks = []
        for series in COSMOS_CAPTURE["series"]:
            if series["points"]:
                at, peak = _pg_peak(series)
                peaks.append({"Metric": series["metric"], "Aggregation": series["aggregation"], "Unit": series["unit"],
                              "EveryMinutes": series["intervalMinutes"], "Points": len(series["points"]), "Peak": peak,
                              "PeakUTC": _pg_text(at)})
        print("T5's metrics (%s) from %s to %s UTC, every %d minutes:" % (
            COSMOS_CAPTURE["account"]["resourceName"], _pg_text(COSMOS_CAPTURE["start"]), _pg_text(COSMOS_CAPTURE["end"]),
            COSMOS_CAPTURE["intervalMinutes"]))
        display_frame(pd.DataFrame(peaks))
        requests_panel = [frame for frame in (cosmos_series_frame("TotalRequests", "requests"),
                                              cosmos_series_frame("ThrottledRequests", "throttled (429)")) if frame is not None]
        plot_panels("T5: Azure Cosmos DB (Azure Monitor, from COSMOS_METRICS)", [
            ("Normalized RU consumption (maximum per interval)", cosmos_series_frame("NormalizedRUConsumption"), "%"),
            ("Requests and throttled requests (429s) per interval", pd.concat(requests_panel) if requests_panel else None, "requests"),
            ("Request units consumed per interval", cosmos_series_frame("TotalRequestUnits"), "RU"),
        ])

## 8. Correlation: scenario windows and telemetry

Each recorded scenario window (from `scenario_runs`, and a demo session's beats, section 3) is compared with the signals its profile should produce on its targets, from the window's start to `TELEMETRY_LAG_MINUTES` after its end:

| Profile | Expected signal |
|---|---|
| `incident:locking-blocking` | `Processes blocked` above 0 or Lock waits |
| `incident:cpu-pressure` | Average CPU in the window at least 1.5 times, and 10 points above, the same target's CPU outside incident windows |
| `incident:io-pressure` | `PAGEIOLATCH` waits, or read volume above the same target's level outside incident windows |
| `incident:inefficient-query` | Higher CPU, or `Log Rate Governor` waits (T3) |
| `incident:missing-index` | Higher CPU or `PAGEIOLATCH` waits |
| `incident:resource-governance` | Higher CPU (T3: at the service tier's limit) |
| `incident:log-rate-pressure` | `Log Rate Governor` waits (T3) |
| `incident:blocking-chain` | `Processes blocked` above 0 or Lock waits |
| `incident:deadlock-storm` | `Number of Deadlocks/sec` above 0, or blocking |
| `incident:tempdb-contention` | Buffer Latch waits (`PAGELATCH_*`) averaging at least `EPISODE_WAIT_SECONDS` per bin |
| `incident:memory-grant-pressure` | Memory waits (`RESOURCE_SEMAPHORE`) averaging at least `EPISODE_WAIT_SECONDS` per bin |
| `incident:parallel-scans` | Parallelism waits averaging at least `EPISODE_WAIT_SECONDS` per bin, or higher CPU |
| `incident:log-flush-waits` | Tran Log IO waits (`WRITELOG`) averaging at least `EPISODE_WAIT_SECONDS` per bin |
| `incident:slow-consumer` | Network IO waits (`ASYNC_NETWORK_IO`) averaging at least `EPISODE_WAIT_SECONDS` per bin |
| `incident:capacity-max` | Higher CPU or `PAGEIOLATCH` waits |
| `incident:plan-regression` | Higher CPU |
| `baseline`, `business-day`, `session` | Activity: batch requests or sessions in `ContosoCare` |

**Pass** means the signal appeared; **Warning** means it didn't (yet): telemetry lags 3–15 minutes, the run may not have started, or it ran on another target. **Manual action required** means the telemetry couldn't be queried here. These thresholds are demo heuristics for the presenter, not product behavior.

**The Database Agent loop (2026-11-18).** A session's Database Agent beat, T3's plan regression in showcase cycle 2, also gets a `database_agent` finding: T3's average CPU from 8 to 1 minutes before the run's response and from 1 to 6 minutes after it, as `NB_DatabaseHub_Control`'s live view checks it. **Pass** means it fell by at least a quarter, or 10 points, once the run forced the last good plan; **Warning** means it didn't, or the telemetry doesn't cover the response. The agent's own issue, its analysis, and its closing show only in Database Hub, which has no API.

In [ ]:
# Correlation: did each recorded scenario window produce its expected signal? Read-only.
LAG = dt.timedelta(minutes=int(TELEMETRY_LAG_MINUTES))


def _slice(name, target, start, end):
    frame = FRAMES.get(name)
    if frame is None:
        return None
    if frame.empty:
        return frame
    part = frame[frame["Target"] == target]
    return part[(part["Bucket"] >= start) & (part["Bucket"] <= end)]


def _outside(name, target, incident_windows):
    frame = FRAMES.get(name)
    if frame is None or frame.empty:
        return frame
    part = frame[frame["Target"] == target]
    mask = pd.Series(True, index=part.index)
    for window in incident_windows:
        mask &= ~((part["Bucket"] >= window["start"]) & (part["Bucket"] <= window["end"] + LAG))
    return part[mask]


def evaluate_signal(signal, target, start, end, incident_windows):
    """Returns (observed, evidence): True or False when the data can tell, None when the telemetry wasn't available."""
    if signal == "blocking":
        gauges_part, waits_part = _slice("gauges", target, start, end), _slice("waits", target, start, end)
        if gauges_part is None and waits_part is None:
            return None, {}
        blocked = 0.0 if gauges_part is None or gauges_part.empty else _safe_number(
            gauges_part.loc[gauges_part["CounterName"] == "Processes blocked", "MaxValue"].max())
        lock_seconds = 0.0 if waits_part is None or waits_part.empty else _safe_number(
            waits_part.loc[waits_part["WaitCategory"] == "Lock", "WaitMs"].sum()) / 1000.0
        return blocked > 0 or lock_seconds > 0, {"maxProcessesBlocked": blocked, "lockWaitSeconds": round(lock_seconds, 1)}
    if signal == "cpu":
        inside = _slice("cpu", target, start, end)
        if inside is None:
            return None, {}
        if inside.empty:
            return False, {"cpuSamples": 0}
        inside_avg = _safe_number(inside["AvgCPU"].mean())
        outside = _outside("cpu", target, incident_windows)
        baseline = None if outside is None or outside.empty else _safe_number(outside["AvgCPU"].mean())
        observed = inside_avg >= 50 if baseline is None else inside_avg >= max(baseline * 1.5, baseline + 10)
        return observed, {"cpuAvgInWindow": round(inside_avg, 1), "cpuAvgOutside": None if baseline is None else round(baseline, 1)}
    if signal == "io":
        waits_part, io_part = _slice("waits", target, start, end), _slice("storage_io", target, start, end)
        if waits_part is None and io_part is None:
            return None, {}
        pageio_seconds = 0.0 if waits_part is None or waits_part.empty else _safe_number(
            waits_part.loc[waits_part["PageIoLatch"].astype(bool), "WaitMs"].sum()) / 1000.0
        read_rate, baseline_rate = 0.0, None
        if io_part is not None and not io_part.empty:
            read_rate = _safe_number(io_part.groupby("Bucket")["ReadMB"].sum().mean())
            outside = _outside("storage_io", target, incident_windows)
            if outside is not None and not outside.empty:
                baseline_rate = _safe_number(outside.groupby("Bucket")["ReadMB"].sum().mean())
        more_reads = read_rate > 0 and (baseline_rate is None or read_rate >= max(baseline_rate * 1.5, baseline_rate + 1))
        return pageio_seconds > 0 or more_reads, {"pageIoLatchSeconds": round(pageio_seconds, 1), "readMBPerBin": round(read_rate, 2),
                                                  "readMBPerBinOutside": None if baseline_rate is None else round(baseline_rate, 2)}
    if signal == "log_rate":
        waits_part = _slice("waits", target, start, end)
        if waits_part is None:
            return None, {}
        seconds = 0.0 if waits_part.empty else _safe_number(waits_part.loc[waits_part["WaitCategory"] == "Log Rate Governor", "WaitMs"].sum()) / 1000.0
        return seconds > 0, {"logRateGovernorSeconds": round(seconds, 1)}
    if signal == "deadlock":
        rates_part = _slice("rates", target, start, end)
        if rates_part is None:
            return None, {}
        deadlocks = 0.0 if rates_part.empty else _safe_number(
            rates_part.loc[rates_part["CounterName"] == "Number of Deadlocks/sec", "Delta"].sum())
        return deadlocks > 0, {"deadlocks": deadlocks}
    if signal in WAIT_FAMILY_CATEGORIES:
        # A wait family: its category's waits average at least EPISODE_WAIT_SECONDS per bin in the window.
        waits_part = _slice("waits", target, start, end)
        if waits_part is None:
            return None, {}
        category = WAIT_FAMILY_CATEGORIES[signal]
        part = waits_part[waits_part["WaitCategory"] == category] if not waits_part.empty else waits_part
        per_bin = 0.0 if part.empty else _safe_number(part.groupby("Bucket")["WaitMs"].sum().mean()) / 1000.0
        return per_bin >= EPISODE_WAIT_SECONDS, {"category": category, "waitSecondsPerBin": round(per_bin, 1)}
    rates_part, sessions_part = _slice("rates", target, start, end), _slice("sessions", target, start, end)
    if rates_part is None and sessions_part is None:
        return None, {}
    batches = 0.0 if rates_part is None or rates_part.empty else _safe_number(
        rates_part.loc[rates_part["CounterName"] == "Batch Requests/sec", "Delta"].sum())
    max_sessions = 0.0 if sessions_part is None or sessions_part.empty else _safe_number(sessions_part["MaxSessions"].max())
    return batches > 0 or max_sessions > 0, {"batchRequests": batches, "maxSessions": max_sessions}


WINDOWS = scenario_windows(WINDOW_START, WINDOW_END)
incident_windows = [window for window in WINDOWS if window["profile"].startswith("incident:") or window["profile"] == "business-day"]
rows = []
for window in WINDOWS:
    if window["profile"] == "off":
        continue
    for target in window["targets"]:
        if target not in SELECTED_TARGETS:
            continue
        check_end = min(window["end"] + LAG, WINDOW_END)
        others = [item for item in incident_windows if item["request_id"] != window["request_id"] and target in item["targets"]]
        outcomes = {signal: evaluate_signal(signal, target, window["start"], check_end, others + [window])
                    for signal in expected_signals(window["profile"])}
        observed = [value for value, _ in outcomes.values()]
        status = "Pass" if any(value is True for value in observed) else (
            "Manual action required" if all(value is None for value in observed) else "Warning")
        evidence = {signal: details for signal, (_, details) in outcomes.items()}
        add_finding("correlation", target, status,
                    "%s on %s, %s to %s UTC: %s." % (window["profile"], target, window["start"].strftime("%H:%M"), window["end"].strftime("%H:%M"),
                                                     "expected signal observed" if status == "Pass" else "telemetry not queried here" if status != "Warning"
                                                     else "expected signal not observed (yet)"),
                    window["request_id"], evidence=evidence, request_id=window["request_id"], start=window["start"], end=window["end"])
        rows.append({"Profile": window["profile"], "Target": target, "Start": window["start"].strftime("%H:%M"),
                     "End": window["end"].strftime("%H:%M"), "Status": status, "Evidence": json.dumps(evidence, default=str)})

if not WINDOWS:
    print("No scenario windows overlap the analysis window: run a session with NB_DatabaseHub_Control, or widen WINDOW_MINUTES.")
elif rows:
    display_frame(pd.DataFrame(rows))

# The Database Agent loop (2026-11-18): one database_agent finding per agent beat, with its CPU before and after the run's
# response.
CPU_SERIES = to_series(FRAMES.get("cpu"), "AvgCPU", "max")
for request_id, beat in (session_beats(SESSION) if SESSION else []):
    if not beat.get("agent") or beat["target"] not in SELECTED_TARGETS:
        continue
    points = [] if CPU_SERIES is None else [(item.Bucket, item.Value) for item in CPU_SERIES[CPU_SERIES["Target"] == beat["target"]].itertuples()]
    status, finding, evidence = agent_beat_finding(SESSION["id"], beat, points, WINDOW_END)
    add_finding("database_agent", beat["target"], status, finding, request_id, evidence=evidence, request_id=request_id,
                start=beat["start"], end=beat["end"])

plot_panels("Scenario windows over the telemetry", [
    ("CPU, average per bin", to_series(FRAMES.get("cpu"), "AvgCPU", "max"), "% CPU"),
    ("Processes blocked (maximum per bin)", to_series(FRAMES.get("gauges"), "MaxValue", "max", lambda f: f["CounterName"] == "Processes blocked"), "processes"),
    ("Batch requests per bin", to_series(FRAMES.get("rates"), "Delta", "sum", lambda f: f["CounterName"] == "Batch Requests/sec"), "requests"),
], WINDOWS)

## 9. See it in the Hub

Open [Database Hub](https://powerbi.com/workloads/fdh/databaseHub) (Fabric Free license; it shows only what your Azure permissions allow):

1. **Overview:** **What needs attention?** shows issues and suggestions; every card leads to Estate. The CPU, memory, storage, and IO summaries should match the trends above.
2. **Estate:** set **View** to **Microsoft SQL**, filter to the demo resource group, and select the target that had the incident. Issues appear as colored chips, suggestions as lightbulb chips, and agent findings with a summary and a recommended action.
3. **Performance:** the **Microsoft SQL** dashboard. Select the same targets and time range as this notebook; check the resource selection before reading a chart as the whole estate.
4. **Copilot:** ask the prompts that the next cell prints, filled in with this session's targets and scenario window. On T3, Database Agents also raises its own issue, in the session's Database Agent beat.

**T4, the PostgreSQL flexible server, and its read replica (ADR 0025).** This notebook can't read T4 live: its tokens (`notebookutils`) cover storage, Power BI, Key Vault, and Kusto, not Azure Resource Manager or Azure Database for PostgreSQL, T4 has no public endpoint, and its metrics aren't in the telemetry endpoint above. Its history comes from section 7's T4 cell, when the scripts pass `POSTGRES_METRICS`. Watch T4 live in Database Hub: in **Estate**, filter to **View: PostgreSQL** and select `pgflex01-<deploymentId>` (during a session, its read replica `pgreplica01-<deploymentId>` too), then open **Performance** and the **PostgreSQL** dashboard, which charts its Azure Monitor metrics (CPU, memory, storage, IOPS, connections, failed connections, and the replica's Read replica lag; [Monitor an Azure Database for PostgreSQL flexible server in the Database Hub](https://learn.microsoft.com/fabric/database/hub/monitor-postgresql)). Hub has no query-level diagnostics for PostgreSQL; T4's live sessions, waits, blockers, and connections against `max_connections` come from the runner: `./scripts/status.ps1 -Detail` (STATUS.T4 and STATUS.T4.LIVE) or `Invoke-DemoWorkloadCommand -Target T4 -ArgumentList status`.

**T5, the Azure Cosmos DB account (ADR 0027).** This notebook can't read T5 live either: it has no Azure Resource Manager token, T5 has no public endpoint, and its metrics aren't in the telemetry endpoint above. Its history comes from section 7's T5 cell, when the control or the scripts pass `COSMOS_METRICS`. Watch T5 live in Database Hub: in **Estate**, filter to **View: Azure Cosmos DB** and select `cosmos01-<deploymentId>`, then open **Performance**: Peak Norm RU % reaches 100% and Throttled Requests (429s) and the 429 Rate % rise during each ru-storm (two per 30-minute showcase cycle), then fall back to the baseline's level with no 429s; Availability % stays at 100%, because a 429 isn't a failure. Its charts use an hourly grain, and its Avg Latency (`ServerSideLatencyDirect`) stays empty, because the runner uses gateway mode ([Monitor a Cosmos DB account in the Database Hub](https://learn.microsoft.com/fabric/database/hub/monitor-cosmos-db)). The in-Hub agent covers SQL, so show T5 on its Cosmos DB views; the runner's view of it: `Invoke-DemoWorkloadCommand -Target T5 -ArgumentList status`.

**What to expect.** Describe the **signals**, never a specific answer: Copilot and Database Agents are previews, their output isn't deterministic, and which SQL targets Copilot covers isn't documented (Database Agents covers Azure SQL Database, T3 here). Any action they propose needs a person to approve it, and nothing in this repo applies a fix automatically. Where neither covers a target, use the charts above, Query Store, or the `microsoft-sql` skills as the fallback.

In [ ]:
# Copilot prompts to try, filled in with this session's names and latest scenario window. Prints text only.
names = {"T1": T1_VM_NAME, "T2": T2_MACHINE_NAME, "T3": T3_SERVER_PREFIX + "<id>/" + DEMO_DATABASE}
focus = next((window for window in reversed(WINDOWS) if window["profile"].startswith("incident:")), None)
focus_target = focus["targets"][0] if focus else "T1"
focus_name = names[focus_target]
if not focus:
    focus_time = "in the last 30 minutes"
elif (focus["end"] - focus["start"]).total_seconds() < 120:
    focus_time = "since %s UTC" % focus["start"].strftime("%H:%M")
else:
    focus_time = "between %s and %s UTC" % (focus["start"].strftime("%H:%M"), focus["end"].strftime("%H:%M"))
prompts = [
    ("Estate health", "Summarize the health of my SQL estate in resource group %s. What needs attention first?" % RESOURCE_GROUP,
     "every selected target named; issues and suggestions ranked"),
    ("Most pressure", "Which of my SQL resources is under the most pressure right now, and why?",
     "CPU, waits, or blocking tied to one target and a time window, matching sections 6 and 7"),
    ("Investigate", "Investigate what happened on %s %s." % (focus_name, focus_time),
     "the incident's signal in section 8's table: blocked processes, Lock waits, or deadlocks; CPU; or its wait family"),
    ("Correlate", "Correlate it with CPU, memory, IO, sessions, and recent changes.", "a timeline across several signals"),
    ("Root cause", "Explain the root cause in customer-friendly terms, with the evidence you used.", "plain language plus metrics and a time window"),
    ("Next action", "What's the safest next action? Don't apply it.", "a proposal that isn't executed and names what needs approval"),
    ("Baseline vs. incident", "Compare the baseline with the incident on %s." % focus_name, "before and after values for the same signals"),
    ("Approvals", "Which of these actions need approval?", "every change listed as needing a person to approve it"),
    ("Verify", "What should I verify in the Azure portal, SSMS, or VS Code?", "concrete places to confirm each finding"),
    ("Summary", "Write a concise incident summary.", "timeline, impact, cause, action, and verification"),
]
print("Database Hub: %s" % DATABASE_HUB_URL)
for title, prompt, signals in prompts:
    print("\n[%s]\n  Ask:        %s\n  Look for:   %s" % (title, prompt, signals))

## 10. Optional: Real-Time Dashboard

For a live view that refreshes on its own, or when the notebook can't query the endpoint directly, build a **Fabric Real-Time Dashboard** over the same telemetry. Both ways are UI steps; there's no documented API for Database Hub's template wizard, so this notebook only checks whether a dashboard exists.

- **From Database Hub:** **Performance** > **Create SQL monitors from template**: choose a template, select the demo databases, pick this workspace, and create a dashboard (or add a page to an existing one).
- **By hand:** create a Real-Time Dashboard in this workspace, add a data source with the URI `https://adx.centralus.arcdataservices.com/kusto/` and the database `ArcSqlTelemetry`, and paste the KQL from sections 5–7 into tiles. Each viewer sees only the resources their own Azure permissions allow; no stored Fabric connection is needed.

The next cell lists the workspace's Real-Time Dashboards with the Fabric REST API (**Items - List Items**, type `KQLDashboard`) and the notebook user's `pbi` token. It changes nothing.

In [ ]:
# Optional Real-Time Dashboard: lists KQLDashboard items in this workspace (read-only Fabric REST call).
dashboards, error = fabric_list_items(WORKSPACE_ID, "KQLDashboard")
if dashboards is None:
    add_finding("real_time_dashboard", "fabric", "Warning", "The workspace's Real-Time Dashboards couldn't be listed: %s" % error, "exists")
elif dashboards:
    add_finding("real_time_dashboard", "fabric", "Pass",
                "Real-Time Dashboards in this workspace: %s." % ", ".join(sorted(item.get("displayName", "?") for item in dashboards)), "exists",
                evidence={"count": len(dashboards)})
else:
    add_finding("real_time_dashboard", "fabric", "Manual action required",
                "No Real-Time Dashboard in this workspace yet. Optional: create one from Database Hub (Performance > Create SQL monitors "
                "from template) or by hand over %s, database %s." % (TELEMETRY_CONNECTION_URI, TELEMETRY_DATABASE), "exists")

## 11. Persist

The telemetry endpoint keeps up to 7 days. The next cell saves this run's analysis to `LH_DatabaseHub`, so the demo's history outlives that window:

| Table | Contents |
|---|---|
| `telemetry_snapshots` | Long-format results: analysis, target, metric, bin time, value, unit, and the analysis window; with `POSTGRES_METRICS`, T4's and its read replica's Azure Monitor metrics too (analysis `postgresql`, targets `T4` and `T4-replica`, section 7), and with `COSMOS_METRICS`, T5's (analysis `cosmosdb`, target `T5`, section 7) |
| `analysis_findings` | Every finding with its status, evidence, scenario request, and window; a session's Database Agent beat also gets a `database_agent` finding (section 8) |
| `scenario_runs` | A demo session's row and its beats (section 3), written last, so a session in it has its history saved |
| `rca_sessions`, `rca_waits`, `rca_events`, `rca_queries`, `rca_resource`, `rca_replica`, `rca_coverage` | The Incident RCA's detection tables (ADR 0031): what the observer captured, decoded from `RCA_EVIDENCE` or the staged file `RCA_EVIDENCE_FILE` names, with each document's `document_id`; their columns are `rca/history-tables.json`'s. Each document is written once: a re-run deletes its earlier rows first. The cell reports `RCA.EVIDENCE`: **Pass** when every document was written, **Warning** with why one was skipped (a missing part, a bad SHA-256, a newer version, another session), **Not selected** when none was passed |
| `rca_answer_key` | What the workload injected, from each document's answer key, kept apart: no detection table carries it, and only the tests and the scoreboard read it |
| `incident_windows` | One row per incident window: each recorded incident request with its expected-signal verdict, and each detected signal episode that no request explains (blocked processes above 0; CPU at or above `EPISODE_CPU_PERCENT` or `PAGEIOLATCH` waits at or above `EPISODE_PAGEIOLATCH_SECONDS` per bin, for `EPISODE_MIN_BINS` bins; Log Rate Governor waits above 0). Merged on its key, so re-running never duplicates one |

Every row also gets `report_date_utc`, the UTC date of its primary time. The RCA evidence is written first, so its `RCA.EVIDENCE` finding is saved with the others, and the staged file is deleted once it's handled; one that a failed run left is deleted a day later by the next run, or by the deployment notebook's purge. The cell then readies the tables that the Power BI history report reads (`fabric/powerbi`), the RCA tables among them: it creates any that are missing, adds and backfills `report_date_utc` once on tables written before it existed, merges the incident windows, and compacts any table with more than 200 files. Each write is a single Parquet file.

It writes lakehouse data only and changes no Azure or Fabric resources. Re-running it in the same session replaces this run's rows instead of duplicating them.

A demo session's runs save under one run ID, `session-<id>`. When a cancel cuts a run off before its `scenario_runs` row, as the Insights app's **Stop now** can, the next run that saves the session (Stop now's, or the next Run all's) completes the save table by table: a table that the earlier run saved keeps its rows, with the beat and `database_agent` findings that the session's plan gave it, which a run without the plan can't make, and nothing is saved twice.

In [ ]:
# Persist: the Incident RCA's evidence, telemetry_snapshots, and analysis_findings in the default lakehouse, then the report
# tables, and a session's rows last (lakehouse data only).
if FINDINGS:
    findings_view = pd.DataFrame(FINDINGS)[["status", "analysis", "target", "finding"]]
    display_frame(findings_view)
    counts = findings_view["status"].value_counts()
    print("Findings: " + ", ".join("%s %d" % (status, counts[status]) for status in STATUSES if status in counts))
if SAVE_RESULTS:
    # The RCA evidence first, so its RCA.EVIDENCE finding is saved with the others (ADR 0031).
    save_rca_evidence()
    write_delta("telemetry_snapshots", SNAPSHOT_ROWS)
    write_delta("analysis_findings", FINDINGS)
    REPORT_TABLES = prepare_report_tables()
    # A session's rows go last: its "session:<id>" row says its history is saved (section 3).
    SESSION_ROWS = [row for row in SESSION_REQUESTS if str(row.get("request_id", "")).startswith("session:")]
    if SESSION_ROWS:
        write_delta("scenario_runs", SESSION_ROWS)
    if REFERENCE_RUN and RCA_STATE["passed"] and _notebookutils() is not None:
        # The control notebook prints RCA.EVIDENCE from this run's exit value, which notebookutils.notebook.run returns.
        _notebookutils().notebook.exit(rca_exit_value())
else:
    print("SAVE_RESULTS is False, so nothing was saved.")

## 12. Next steps

- **Next demo:** select Run all in `NB_DatabaseHub_Control`, in the control workspace (ADR 0026). It runs the whole session: the start, the showcase on T1 to T5 with T4's read replica, the presenter's guide, and, at its end, the stop, then this notebook in its Spark session, which saves the session's history. For one activity from a PC instead, `./scripts/demo-control.ps1`; `./scripts/stop-demo.ps1` stops the demo.
- **History:** the Power BI report `RPT_DatabaseHub_History` (`fabric/powerbi/README.md`) shows `telemetry_snapshots`, `analysis_findings`, `scenario_runs`, `incident_windows`, and `validation_results` from `LH_DatabaseHub`, beyond the telemetry's 7-day retention. Its model, `SM_DatabaseHub_History`, also reads the Incident RCA's tables, which the Insights app's RCA view and chat investigate (ADR 0031). The Insights app (`fabric/app`) shows T4's health and its read replica's lag from the `postgresql` rows on T4's Estate page. T5's `cosmosdb` rows are in the history model's Cosmos DB measures (Peak Norm RU, throttled requests, the 429 rate, RU consumed; `fabric/powerbi/README.md`).
- **Validation:** run `NB_DatabaseHub_Deployment` for the deployment checks, or `./scripts/validate.ps1` from the repo for the full catalog, including the Azure-side checks this notebook can't do.

**References:** [Query performance monitoring telemetry](https://learn.microsoft.com/fabric/database/hub/microsoft-sql-query-performance-monitoring-telemetry) · [Monitor Microsoft SQL in Database Hub](https://learn.microsoft.com/fabric/database/hub/monitor-sql) · [NotebookUtils credentials](https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-credentials) · [NotebookUtils notebook run and exit](https://learn.microsoft.com/fabric/data-engineering/notebookutils/notebookutils-notebook-run) · [Items - List Items (Fabric REST API)](https://learn.microsoft.com/rest/api/fabric/core/items/list-items) · [Create a Real-Time Dashboard](https://learn.microsoft.com/fabric/real-time-intelligence/dashboard-real-time-create) · [The Database Hub CLI skill](https://github.com/microsoft/skills-for-fabric/tree/main/skills/databasehub-cli)